# 0. Page de garde

**Nom Prénom** : ALBURQUERQUE Julien ([github](https://github.com/jalb-code) / [linkedin](https://www.linkedin.com/in/jalb-84aa46259/))  
**Formation** : CISIA — Concevoir et implémenter une solution d'intelligence artificielle (référentiel C1→C9)  
**Cas d'usage** : Détection précoce du décrochage étudiant en L1  
**Dépôt GitHub** : [cisia-decrochage-etudiant](https://github.com/jalb-code/cisia-decrochage-etudiant.git)  
**Date de soutenance** : 2026-09-01 — **Version du notebook** : v0.3 (2026-08-16)

**Environnement d'exécution** : **Python 3.13**, gestionnaire de paquets **`uv`** - dépendances figées par `uv.lock`.  

**Comment lancer le notebook** :   
- Vérifier que les fichiers de données sont dans `/data/raw` ou les déposer dedans,  
- `uv sync --group analysis`, puis exécuter les cellules dans l'ordre.  

In [ ]:
import hashlib
import json
import logging
import platform
import shutil
import sys
import warnings
from datetime import UTC, datetime
from importlib.metadata import version as get_package_version

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import shap
import sklearn
import xlsxwriter
from codecarbon import OfflineEmissionsTracker
from IPython.display import Image, Markdown, display
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV

from decrochage_l1 import eda
from decrochage_l1.config import settings
from decrochage_l1.data import preparation
from decrochage_l1.data.utils import cleaning_utils, profiling_utils
from decrochage_l1.modeling import (
    ablation,
    evaluation,
    families,
    preprocessing,
    protocol,
    threshold,
    tuning,
)
from decrochage_l1.serving import (
    explain,
    model_card,
    normalization,
    schemas,
    scoring,
    store,
)
from decrochage_l1.serving.contract import ModelFacts, OperationalDefaults, ServiceContract
from decrochage_l1.serving.store import EntrepotModele

# Silences d'import : progress-bar tqdm (ipywidgets absent en exécution notebook) et
# journal verbeux de codecarbon - on garde les vraies erreurs, on coupe le bruit.
warnings.filterwarnings("ignore", message=".*IProgress.*")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 38)

# Enregistrer les paliers bronze/silver/gold en CSV sur disque. Par défaut False : le
# notebook calcule tout en mémoire, sans rien écrire (les paliers se rejouent à la demande).
ENREGISTRER_PALIERS = True

# §8+ : reprendre en mémoire le gold construit en §7 (False) ou le relire depuis
# data/gold/ (True), pour exécuter la modélisation indépendamment des sections amont.
CHARGER_GOLD_DEPUIS_CSV = False

# §9 : recharger les réglages Optuna depuis le cache (True) ou les relancer
# (False, write-through). Optuna étant seedé, le recalcul rend les mêmes paramètres.
REGLAGE_DEPUIS_CACHE = False
CACHE_REGLAGE = settings.report_dir / "tuning_cache.json"

## _Scaffolding du projet_

- **Création de la structure du projet** : Le code réutilisable dans `src/`, notebook unique `JALB-Decrochage-l1.ipynb` , données dans `data` non commité (voir journal de bord)
- **Hygiène de code** : format/lint (`ruff`), tests (`pytest`), `pre-commit` et intégration continue (GitHub Actions) → un livrable reproductible et vérifiable.

### _Journal de bord des décisions_

1. **[D01] Ne pas versionner les données.**  
⇒ Les fichiers data contiennent des données personnelles sensibles. Rien n'atteste formellement de leur caractère synthétique donc par précaution RGPD (minimisation), je les exclus du dépôt Git. Le dépôt cloné n'est donc pas exécutable en l'état - compromis assumé, la procédure de mise en place des données étant décrite dans le `README.md`.

 
2. **[D02] Utilisation de l'IA générative**  
⇒ J'ai choisi d'utiliser Claude Code pour m'assister dans la réalisation de ce projet. Au vu du temps limité que je peux y allouer et de la méta actuelle du développement logiciel, ce serait selon moi une erreur de ne pas l'utiliser. Il n'en reste pas moins qu'il s'agit d'un projet de certification où je dois démontrer mes compétences. C'est pourquoi j'ai décidé d'encadrer l'usage de l'IA de la manière suivante :
   - l'outil code selon mes directives et sous ma relecture ;
   - c'est moi qui prends les décisions, en m'appuyant sur des agents consultants pour m'éclairer avant de trancher (consultant ML, consultant RGPD, gardien du cas d'usage) ;
   - dans le notebook, les cellules de code sont écrites par l'IA sous mes directives ; les cellules markdown sont d'abord rédigées par moi, puis relues et corrigées par l'IA, avant une dernière relecture de validation de ma part.

# 1. Résumé exécutif

**Problème** : Dans une université pluridisciplinaire, ~28 % des étudiants de L1 de la cohorte abandonnent (1 479 sur 5 200). Aujourd'hui, l'abandon ne se constate qu'en fin de semestre — trop tard pour agir.

**Objectifs** : Concevoir une solution d'IA capable de prédire, **à mi-S1**, le risque de décrochage, afin de proposer un accompagnement au bon moment. 2 cibles de prédictions sont demandées: 
  - **Risque d'abandon** — score de risque (classification) : faut-il proposer un accompagnement ?
  - **Moyenne finale (/20)** — régression : calibrer et prioriser l'intensité de l'accompagnement.

## Approche

Une démarche de bout en bout, du besoin métier au service qui dure ; chaque étape est développée et **justifiée dans sa section** (journal de bord).

```mermaid
flowchart TD
    A["1 · Comprendre le besoin<br/>décrochage constaté trop tard → agir à mi-S1<br/>Section §2"]
    B["2 · Cadrer avant tout<br/>cibles · RGPD, éthique, responsabilités · aide à la décision<br/>Section §2—§4"]
    C["3 · Fiabiliser la donnée<br/>profilage · conformation · EDA<br/>exclusions de principe : anti-fuite, minimisation · features<br/>Section §5—§7"]
    D["4 · Modéliser et comparer<br/>baseline + ≥2 familles → départage sur AUC<br/>classification + régression<br/>Section §8—§9"]
    E["5 · Prouver et arbitrer<br/>métriques · seuil = coût FN&gt;FP · SHAP · audit d'équité<br/>Section §9, §12"]
    F["6 · Industrialiser<br/>pipeline sérialisé (score) · architecture cible<br/>Section §10—§11"]
    G["7 · Assumer les limites et durer<br/>limites connues · suivi de dérive · ré-entraînement · versioning<br/>Section §13—§14"]
    A --> B --> C --> D --> E --> F --> G
    G -.->|nouvelle promotion / dérive| C
```

## Résultats clés

- **Détection du décrochage (`abandon`)** : les modèles sont départagés sur l'**AUC = 0,945** (indépendante du seuil) ; au seuil retenu, le modèle retrouve **78,7 %** des décrocheurs sur le test scellé (Rappel, §12.2).
- **Priorisation (`moyenne_finale`)** : régression de la note finale (**MAE 2,24 · RMSE 3,08 pts/20**, R² = 0,685), pour calibrer l'intensité de l'accompagnement.
- **Décision métier** : seuil fixé par l'**asymétrie des coûts** (un décrochage manqué coûte plus qu'un signalement à tort) : sur le test scellé, **304** étudiants signalés, dont **233** décrocheurs réels (précision 76,6 %).
- **Explicabilité** : contributions signées par variable - **analytiques en production**, SHAP en **confirmation dans le notebook** (§12.4)
- **Livrable - un service API** : score toute une promotion en une campagne et rend, pour chaque étudiant, sa probabilité d'abandon, sa note prévue /20 et les facteurs qui pèsent le plus ; il **assiste la décision** sans trancher - l'équipe pédagogique garde la main sur l'accompagnement.
- **Industrialisation** : pipeline sérialisé + contrat `predict()`, architecture cible et suivi de dérive.

## Limites

- **Généralisation et dérive** : une seule cohorte (2024-2025, données synthétiques) et une validation intra-cohorte → le modèle est susceptible de devoir être **ré-entraîné** à la prochaine promotion.
- **Complétude des données** : manquants sur 10 colonnes → Mise en place de stratégie d'imputation comme parade à cette limite
- **Calibrage du seuil** : coûts métier (FN/FP) et capacité d'accompagnement non fournis → hypothèse de travail : un décrochage manqué (FN) pèse plus qu'un signalement à tort (FP), sans que ce dernier soit anodin.
- **Déployabilité conditionnée** : la mise en production dépend d'actes du responsable de traitement (AIPD, base légale confirmée par le DPO, information des étudiants) et d'effets de bord du marquage à documenter

# 2. Cadrage métier et cas d'usage — journal de bord [C1]

## Problématique métier

Une université pluridisciplinaire constate un **taux d'abandon élevé en L1**. Aujourd'hui,
l'abandon se **constate en fin de semestre** - trop tard pour agir.

La **direction de la réussite étudiante** veut identifier, **dès mi-parcours du S1**, les
étudiants à risque de décrochage, pour déclencher un accompagnement - tutorat, soutien
méthodologique, aide sociale.

## Enjeux métier

- **Humain** - un décrochage laisse une trace **psychologique** (sentiment d'échec, perte de
  confiance, déclassement ressenti) et **professionnelle** (employabilité réduite, entrée
  retardée dans la vie active). Il est souvent **réversible s'il est repéré tôt**.
- **Institutionnel** - les ressources d'accompagnement sont **limitées** (heures de tutorat,
  aides) ⇒ il faut **cibler**. D'où le rôle de la cible secondaire `moyenne_finale` - prioriser
  et calibrer l'intensité - et de l'explicabilité - désigner les **facteurs** sur lesquels agir.
- **Économique** - un abandon est un investissement gaspillé et se répercute en cascade :
  - **l'étudiant** perd l'année engagée et s'expose au chômage ou à un emploi moins rémunérateur ;
  - **l'institution** perd les moyens publics investis pour le former ;
  - **la société** compte un diplômé de moins - productivité et recettes fiscales moindres.

## Objectifs et contraintes

Deux prédictions complémentaires à mi-S1 :

| Cible | Variable | Tâche | Rôle |
|---|---|---|---|
| **Principale** | `abandon` (0/1) | Classification binaire | Faut-il proposer un accompagnement ? |
| **Secondaire** | `moyenne_finale` (/20) | Régression | Prioriser et calibrer l'intensité |

La cible principale déclenche l'action ; la secondaire la **module**.

Trois contraintes pèsent dès le cadrage et orientent toute la chaîne :

- **Horizon mi-S1 - risque de fuite temporelle.** Une variable n'est légitime que si elle est
  **connue à mi-parcours** ; une donnée consolidée en fin de S1 (`moyenne_partiels_s1`,
  `nb_ue_validees_s1`) gonflerait la validation tout en rendant le modèle inexploitable en
  production. Reste à établir en **§3** quelles variables sont disponibles à mi-S1.
- **Explicabilité exigée.** L'énoncé impose des indicateurs **explicables** ; la solution restitue
  des **contributions signées par variable** - analytiques en production, SHAP dans le notebook (§12.4).
- **Éthique et RGPD - variables sensibles.** L'énoncé exige un usage **éthique et conforme** des
  données étudiantes ; biais portés par `sexe`, `boursier`, `etablissement_origine`. Cet aspect sera traité en **§4**.

## Rythme d'usage : une campagne par an

Le scoring s'exécute **une fois par an**, à mi-S1, sur **toute la promotion** en cours - il n'y a **pas de besoin temps réel**. Ce que ce rythme implique pour le contrat de service et le choix du modèle est tranché au journal de bord.

## Le coût d'une erreur : FN (décrochage manqué) >> FP (signalement à tort)

Le modèle peut se tromper de **deux façons**, qui n'ont ni les mêmes conséquences ni le même coût :

- **Faux négatif (FN) - un décrochage manqué.** Le modèle prédit « pas d'abandon » pour un
  étudiant qui décroche : aucun accompagnement n'est proposé à qui en aurait eu besoin, l'occasion
  d'agir à temps est perdue. **Coût le plus lourd, et souvent irréversible** - une chance de
  réussite qui ne revient pas pour l'étudiant, une année d'investissement perdue pour l'institution.
- **Faux positif (FP) - un signalement à tort.** Le modèle prédit « à risque » pour un étudiant
  qui ne décroche pas : une ressource limitée est mobilisée pour rien. **Coût borné et
  réversible** - un créneau de tutorat consommé en trop ; s'y ajoute, pour l'étudiant, une
  étiquette « à risque » qui n'est pas neutre.

**FN >> FP** : un étudiant perdu pèse plus qu'un créneau de tutorat dépensé. La solution doit donc privilégier le rappel (*recall*) sur `abandon` - mieux vaut un signalement à tort qu'un décrochage manqué. Sans pour autant tout signaler : les FP consomment la capacité d'accompagnement, limitée. Où placer le curseur, c'est le seuil - arbitré en §9 (son effet mesuré sur le test en §12).

## Parties prenantes

| Acteur | Rôle | Ce qu'on doit lui fournir |
|---|---|---|
| **Direction réussite étudiante** | Commanditaire | Une solution répondant aux objectifs qu'elle a définis. |
| **Tuteurs / responsables pédagogiques** | Utilisateurs | Un outil qui, à mi-S1, signale le risque, estime la moyenne finale et **explique** la prédiction, pour adapter l'accompagnement. |
| **Étudiant** | Sujet des données **et** bénéficiaire | Minimisation des données ; vigilance sur biais et stigmatisation |
| **DPO / référent éthique** | Garant de la conformité | Les livrables de conformité |
| **SI scolarité / LMS** | Fournisseur des données | Une proposition d'intégration technique |

## _Journal de bord des décisions_

1. **[D03·D04] FN ≫ FP, sans coût métier facilement chiffrable**  
⇒ Je départage les modèles sur la **ROC/AUC** - indépendante du seuil et fiable à déséquilibre modéré (~28/72) - **complétée par la PR-AUC**, qui rend compte de la performance sur la classe minoritaire (les décrocheurs) que la ROC/AUC lisse.  
⇒ Je fixe le point de fonctionnement en §9, sur les probabilités out-of-fold du train : faute de coût métier chiffrable, aucun seuil ne se déduit par le calcul - il se **déclare** par une politique lisible et **configurable**, par défaut un **plancher de rappel** (détecter au moins R % des décrocheurs), à défaut une **capacité d'accompagnement** ; le seuil reste externe au modèle (D15). §12 n'en mesure que l'effet sur le test scellé - le choisir sur le test reviendrait à l'y ajuster puis à l'y juger.

2. **[D05] Explicabilité imposée par l'énoncé**  
⇒ Je fournis une explicabilité globale et locale (SHAP) et j'en fais un critère de choix du modèle : le besoin des équipes pédagogiques est double - **global** (quels facteurs pèsent sur le risque) et **local** (pourquoi *cet* étudiant est signalé, pour adapter l'accompagnement).

3. **[D06] Usage : une campagne par an, sur toute la promotion**  
⇒ Je prévois d'intégrer dans la solution finale un point d'entrée qui score une promotion entière en une passe (endpoint /predict-cohorte);

# 3. Données : disponibilité, gouvernance et alternatives — journal de bord [C1]

## 3.1 Description des sources

Deux fichiers CSV sont fournis :

- **`decrochage_etudiants_complet_V5.csv`** - le jeu principal : ~5 200 lignes, 33 colonnes, une ligne par étudiant de L1, **une seule cohorte 2024-2025**, observée à mi-parcours du S1. Un échantillon de 50 lignes accompagne le fichier pour une première lecture.
- **`catalogue_formations_V5.csv`** - table de référence des filières, jointe au jeu principal par `filiere`.

L'ensemble des colonnes peut être classées par thème métier :

| Thème | Colonnes |
|---|---|
| Identifiants | `student_id`, `id_dossier` |
| Contexte d'inscription | `annee_universitaire`, `filiere`, `date_inscription` |
| Profil social et démographique | `age`, `sexe`, `boursier`, `distance_domicile_km`, `heures_travail_remunere_sem` |
| Parcours antérieur | `bac_type`, `mention_bac`, `etablissement_origine` |
| Engagement LMS | `connexions_lms_30j`, `heures_lms_total`, `ressources_consultees`, `messages_forum` |
| Assiduité et travail rendu | `taux_presence_pct`, `retards_rendus`, `nb_devoirs_total`, `nb_devoirs_rendus` |
| Résultats académiques | `moyenne_partiels_s1`, `nb_ue_total`, `nb_ue_validees_s1` |
| Ressenti déclaré | `motivation`, `satisfaction`, `sentiment_appartenance` |
| Avis du tuteur | `commentaire_tuteur` |
| Leurres annoncés | `groupe_td`, `couleur_carte_etudiante`, `jour_inscription` |
| Cibles | `abandon`, `moyenne_finale` |
| Information filière | Dans le 2nd fichier : `faculte`, `niveau`, `ects_semestre`, `capacite_accueil`, `volume_horaire_s1`, `taux_reussite_historique_pct` |

 ## 3.2 Disponibilité, accès et gouvernance

**Disponibilité et accès**

L'énoncé ne documente pas comment ces données sont collectées. Au vu de la nature des colonnes (traces LMS, dossier de scolarité, état civil, enquêtes déclaratives), je pose l'hypothèse d'un jeu assemblé automatiquement par l'université depuis ses outils (LMS, inscription…).

**Gouvernance, à notre niveau**

Les données ne servent qu'à l'entraînement et à l'évaluation des modèles. Elles ne sont pas conservées au-delà de la période de VSR (vérification de service régulier) qui suit la mise en production - ni archivage, ni réutilisation à d'autres fins. Corollaire : tout ré-entraînement (§13) suppose que l'université remette à disposition un jeu de données à jour - nous n'en gardons aucune copie dormante. La conformité RGPD d'ensemble (base légale, finalité, information des personnes) est traitée en §4 ; le stockage des paliers en §7.

## 3.3 Données connues à mi-S1 et fuite temporelle

Une variable n'est exploitable que si sa valeur **existe à mi-parcours** : en utiliser une qui n'est
connue que plus tard ferait reposer le modèle sur une information absente au moment du scoring - la
validation paraîtrait flatteuse, mais le modèle serait inexploitable en production.

**Cas des colonnes `moyenne_partiels_s1` et `nb_ue_validees_s1`**

Résultats consolidés en fin de S1 : les intégrer comme features à mi-S1 serait une fuite
temporelle. Je les place hors du périmètre de scoring.

**Cas de la colonne `commentaire_tuteur`**

Les libellés relevés sont des bilans de semestre (« bon semestre », « résultats fragiles aux
partiels ») - or les partiels tombent en fin de semestre. Je présume donc un champ **rédigé en fin
de S1**, hors horizon, que je place hors périmètre au même titre que les deux colonnes ci-dessus.
La présomption s'appuie aussi sur un fort volume d'absence.

**Cas des cibles `abandon` et `moyenne_finale`**

Ce sont les cibles de prédiction, constatées en fin de S1 : inconnues à mi-S1 par nature.

**Cas des autres colonnes**

Aucune information ne laisse penser qu'elles ne soient pas connues à mi-S1 : je les présume disponibles à mi-parcours - présomption, non preuve (D18).

## 3.4 Stratégie d'imputation des valeurs manquantes

Le constat chiffré - colonnes concernées et taux - est établi en §5, le diagnostic mené
en §6 ; ici je fixe seulement la **méthode de décision**, pas son résultat - une imputation
se décide après la mesure qui la fonde.

Un mécanisme d'absence ne s'observe pas : je ne cherche pas à le *prouver*, mais à répondre à
trois questions dont chaque réponse commande la suite.

**Q1 - Les trous se concentrent-ils sur les mêmes dossiers ?** (structure, §6)
Si une poignée de dossiers portent l'essentiel des absences, leur suppression *peut* être
envisagée - à la double condition que la perte soit négligeable et l'absence plausiblement
aléatoire. Hors ce cas, je n'élimine aucune ligne : supprimer biaise dès que l'absence n'est
pas MCAR et sacrifie des décrocheurs (classe rare). C'est le défaut, pas la règle.

**Q2 - L'absence est-elle un signal ?** (lien avec `abandon`, §6)
Si manquer va de pair avec l'issue, j'ajoute un indicateur binaire `x_manquant` *avant*
d'imputer, pour conserver ce que le trou porte. Sinon, l'imputation seule suffit.

**Q3 - L'absence est-elle présumée liée à la valeur cachée ?** (hypothèse métier, MNAR)
Présomption, non testable. Si je la retiens, la valeur d'imputation relève alors du
**métier**, pas de la statistique : une modalité déclarée « aucune »/« Inconnu » (cat.), un
plancher métier tel que `0` pour une non-déclaration (num.).

À défaut, quand aucune des trois ne tranche : **médiane** (num.) / **mode** (cat.), sans indicateur.

**Anti-fuite** : imputation et indicateurs sont `fit` sur le **train seul**, dans le `Pipeline` (§8).

## 3.5 Enrichissement : catalogue, cohorte, ré-échantillonnage

**Catalogue des filières**

Toutes ses colonnes - faculté, niveau, ECTS, capacité, volume horaire, taux de réussite historique - sont des attributs de la filière : à filière donnée, leur valeur est fixe. Ce sont donc des fonctions déterministes de filiere, déjà présente dans le jeu étudiant - les joindre n'introduit aucune information nouvelle, seulement une réécriture de filiere.

**Une seule cohorte** 

Le jeu ne couvre que 2024-2025. Le modèle risque d'être spécifique à cette
promotion et de dériver aux suivantes - d'où la nécessité d'un ré-entraînement sur plusieurs
années et d'un suivi de dérive (§13). Limite à documenter et surveiller, pas à corriger ici.

**Déséquilibre de classes** 

Le taux d'abandon annoncé (~28 %) est un déséquilibre modéré : il ne justifie pas un ré-échantillonnage (SMOTE), qui fabriquerait des exemples synthétiques pour un gain douteux. Une pondération des classes (class_weight) suffit si besoin. Le taux exact sera confirmé en §6 - seule mesure susceptible de rouvrir la question.

## 3.6 Les leurres annoncés

L'énoncé désigne trois leurres : `groupe_td`, `couleur_carte_etudiante`, `jour_inscription`. Aucun
n'a de lien causal plausible avec le décrochage, et `jour_inscription` paraît n'être que le jour de
semaine de `date_inscription`. Je ne les exclus pas pour autant : je les **conserve dans le gold** et
je démontre leur nullité, plutôt que de la supposer, par trois contrôles convergents.

- **Redondance** - `jour_inscription` reproduit-il le jour de semaine de `date_inscription` ? (§6)
- **Lien bivarié** - chaque leurre est-il associé à `abandon`, seul à seul ? (§6)
- **Coût de retrait** - les enlever dégrade-t-il la détection ? Nul, voire négatif, attendu (ablation, §8.6)

## _Journal de bord des décisions_

1. **[D07] `commentaire_tuteur` hors périmètre de scoring**  
⇒ Présumé rédigé en fin de S1 (contenu de bilan + ~49 % de vides) : l'employer à mi-S1 serait une fuite temporelle.

1. **[D08] Pas d'enrichissement avec le catalogue de formation**  
⇒ Toutes ses colonnes sont des fonctions déterministes de filiere, déjà présente : les joindre n'ajoute aucune information, seulement une réécriture.

1. **[D09] Valeurs manquantes : trois questions avant d'imputer**  
⇒ (1) trous concentrés sur les mêmes dossiers → (2) absence corrélée à `abandon` → (3) absence présumée liée à la valeur cachée (MNAR) - chaque réponse commande le traitement (suppression au cas limite, indicateur `x_manquant`, valeur déclarée) ; à défaut médiane/mode. Diagnostic en §6, imputation ajustée sur le train seul (§8). *Écarté* : la suppression **systématique** des lignes incomplètes - elle biaise dès que l'absence n'est pas MCAR et sacrifie des décrocheurs (classe rare).

1. **[D10] Pas de ré-échantillonnage (SMOTE)**  
⇒ Le déséquilibre annoncé (~28 %) est modéré et ne le justifie pas ; une pondération des classes suffit si nécessaire. Le taux sera confirmé en §6 - seule mesure susceptible de rouvrir la question.

1. **[D11] Leurres : conservés et démontrés nuls, plutôt qu'exclus a priori**  
⇒ Je les garde dans le gold et démontre leur nullité plutôt que la supposer : redondance et lien bivarié (EDA, §6), coût de retrait mesuré par ablation (§8), confirmé par SHAP (§12). Écarté - les exclure a priori sur « aucun sens métier », jugement contestable qui prive de la preuve sans rien économiser.

1. **[D18] Les colonnes hors résultats fin-de-S1 sont présumées connues à mi-S1**  
⇒ Faute de documentation sur l'horizon de collecte, je présume disponibles à mi-parcours toutes les colonnes hors résultats consolidés (D07) et cibles - c'est une **présomption raisonnée, pas une preuve**. En production, le contrat d'entrée fige cet horizon. *Écarté* - inclure une variable au statut temporel douteux : gain de validation qui ne se retrouverait pas en production (fuite).


# 4. Enjeux éthiques, sociétaux et conformité — journal de bord [C2]

## 4.1 Base légale et finalité

Le traitement doit reposer sur une base légale avant toute autre considération. Je retiens la **mission d'intérêt public** (art. 6.1.e) : l'aide à la réussite étudiante relève des missions de service public de l'établissement. Cette base **dispense du consentement** - l'accompagnement ne saurait être conditionné à un accord préalable, ni laisser un étudiant renoncer à une aide. Elle suppose un rattachement à une base en droit national (art. 6.3) et doit être **confirmée par le DPO**. Elle a un mérite second : elle évite le point dur du consentement pour les étudiants mineurs, qu'une cohorte de L1 peut compter.

La **finalité est déterminée** (art. 5.1.b) : proposer un **accompagnement pédagogique**. Tout autre usage est proscrit et consigné dans la fiche du modèle - sélection à l'entrée, réorientation subie, évaluation des enseignants. La cible secondaire `moyenne_finale` relève du même cadre : une aide à la décision d'accompagnement, **jamais** une note anticipée ni un instrument d'évaluation.

## 4.2 Minimisation et données identifiantes

Le principe de **minimisation** (art. 5.1.c) commande de ne traiter que les données **nécessaires à la finalité** : le seul pouvoir prédictif d'une variable ne suffit pas à la justifier. Il gouverne les exclusions de ce chapitre (§4.4) et le sort des variables démontrées inutiles (D16).

Les identifiants `student_id` et `id_dossier` **n'entrent pas dans le modèle** : ils identifient un dossier, ils ne portent aucune information généralisable. Le contrat d'API pourra accepter un identifiant, mais **au seul titre de la corrélation** entre une requête et sa réponse, à l'usage de l'appelant ; la solution ne le **journalise ni ne le conserve**. Les modalités de conservation côté établissement relèvent de sa responsabilité (§4.7).

## 4.3 Décision automatisée et droit à l'explication

Le système ne prend **aucune décision** au sens de l'art. 22 : il produit une **probabilité de décrochage**, et l'équipe pédagogique décide de l'accompagnement (garde-fous en §4.6). L'humain n'est pas une validation de façade - c'est lui qui arbitre, la probabilité n'étant qu'un signal parmi d'autres. Cette garantie tient à la décision humaine et à sa **consignation dans la documentation du modèle** (model card, §12), non à un avertissement répété à chaque réponse du service : un message systématique en sortie, noyé dans chaque prédiction, perdrait sa force - la conformité est organisationnelle et documentée, pas un champ de payload.

Le **droit à l'explication** est servi par l'explicabilité du modèle : pour chaque étudiant signalé, la **contribution signée de ses variables** est restituée - analytique en production, SHAP en confirmation dans le notebook (§12.4). Ce droit se rattache aux **art. 13-14** (information sur la logique sous-jacente), non à l'art. 15 qui vise l'accès de la personne à ses propres données. L'explicabilité est déjà actée comme critère de choix du modèle (D05, §2) et démontrée en §12.

## 4.4 Variables protégées et proxies socio-économiques

Le jeu porte deux **caractéristiques protégées au sens du droit de la non-discrimination** - le `sexe` et le statut `boursier` - non des données sensibles de l'art. 9 RGPD, qui ne couvre ni l'un ni l'autre. L'**art. 225-1 du Code pénal** interdit de distinguer les personnes sur le fondement du sexe et de la « particulière vulnérabilité résultant de la situation économique », à laquelle le statut boursier se rattache.

Je les **exclus du modèle par principe** - de non-discrimination et de minimisation - **quel que soit leur pouvoir prédictif** (D13). Elles ne disparaissent pas du jeu pour autant : conservées **hors modèle**, elles servent l'**audit d'équité** par sous-groupes (§12), qui vérifie que le modèle ne rate pas davantage de décrocheurs selon le sexe ou le statut boursier.

Trois variables ne sont pas protégées mais appellent un contrôle : `etablissement_origine`, `heures_travail_remunere_sem`, `distance_domicile_km`. On peut soupçonner qu'elles laissent **déduire la situation économique** de l'étudiant - la même information que porte `boursier`, exclu ci-dessus. Les intégrer sans vérification risquerait de **ré-encoder** cette variable protégée écartée (discrimination indirecte). Je conditionne donc leur entrée dans le modèle à l'**absence de corrélation avec `boursier`** - contrôle reporté à l'EDA (§6, D14).

## 4.5 Biais de représentation

Un modèle apprend mal ce que les données représentent peu. Outre le biais de *décision* (§4.6), il faut surveiller le biais de *représentation* : une modalité rare - le `sexe` renseigné « Autre », un type d'établissement marginal - risque d'être mal apprise, et le modèle peu fiable sur le groupe correspondant. Le repérage de ces modalités est reporté à l'EDA (§6) ; selon le résultat, une modalité insuffisamment représentée est soit **regroupée** (modalité « Autre » déclarée), soit **signalée comme zone de moindre fiabilité** ; dans les deux cas, l'**audit d'équité** couvre ces modalités rares (§12, D17), pour vérifier que la performance ne s'effondre pas sur ces groupes peu représentés.

## 4.6 Marquage, stigmatisation et garde-fous

Signaler un étudiant n'est pas neutre. **Ne pas signaler** le laisse sans proposition d'aide - le risque même que le projet entend réduire. **Signaler** ouvre deux risques opposés : la **prophétie auto-réalisatrice** - un étudiant désigné « à risque » peut intérioriser ce jugement - et la **stigmatisation** si l'information circule au-delà de ceux qui accompagnent. Aucun n'est nul ; les garde-fous visent à les contenir.

**Côté établissement**

- la probabilité n'est **jamais** le seul critère de déclenchement ; la décision revient à l'**équipe pluridisciplinaire** ;
- l'étudiant peut **refuser** un accompagnement, et peut en **demander** un même à probabilité faible - c'est l'équipe qui statue ;
- il incombe à l'établissement de **s'assurer que ces garde-fous sont effectifs**, et pas seulement affichés.

**Côté solution**

- exposer une **probabilité**, non un **verdict binaire** : celui-ci reste **masqué par défaut** (minimisation, art. 22 - la décision reste à l'équipe), activable par le responsable de traitement pour un tri à capacité (§11) ; l'anti-stigmatisation tient aux garde-fous d'usage ci-dessus (D15).

## 4.7 Ce qui relève du responsable de traitement

L'établissement est **responsable de traitement** ; la solution agit comme **sous-traitant** (art. 28). Les points suivants sont cadrés ici mais **produits par l'établissement**, hors périmètre du notebook :

- **information des personnes** (art. 13-14) - informer les étudiants de l'existence du traitement et de sa logique ;
- **conservation** (art. 5.1.e) - durée limitée à ce que la finalité exige ; cohérent avec §3.2, aucune copie dormante côté solution ;
- **sécurité** (art. 32) - mesures techniques et organisationnelles de protection des données ;
- **analyse d'impact** (AIPD, art. 35) - le profilage d'un public étudiant en fait un traitement candidat.

## _Journal de bord des décisions_

1. **[D12] Base légale = mission d'intérêt public (art. 6.1.e)**  
⇒ Je fonde le traitement sur la mission de service public d'aide à la réussite, ce qui dispense du consentement ; à confirmer par le DPO et à rattacher à une base en droit national. *Écarté* - le consentement (art. 6.1.a) : il conditionnerait l'accompagnement à un accord préalable et laisserait un étudiant renoncer à une aide, à rebours de la finalité.

2. **[D13] Variables protégées exclues du modèle par principe**  
⇒ `sexe` et `boursier` (motifs de l'art. 225-1) sont retirés du vecteur de features quel que soit leur pouvoir prédictif ; conservés hors modèle pour l'audit d'équité (§12). *Écarté* - les conserver pour leur pouvoir prédictif : gain possible mais discrimination directe et effet stigmatisant, que la minimisation proscrit dès lors que la finalité n'en dépend pas.

3. **[D14] Proxies socio-économiques : intégration conditionnée à l'absence de corrélation avec `boursier`**  
⇒ `etablissement_origine`, `heures_travail_remunere_sem`, `distance_domicile_km` peuvent laisser déduire la situation économique, comme `boursier` (exclu) ; leur entrée dans le modèle est conditionnée à un contrôle de non-corrélation avec `boursier` (§6), pour ne pas ré-encoder la variable protégée écartée ; ce contrôle levé, leur maintien relève encore de la minimisation, mesurée par ablation (§8, D16). *Écarté* - les inclure sur leur seul pouvoir prédictif, sans contrôle : conserverait un proxy potentiellement corrélé au protégé.

4. **[D15] Exposer une probabilité, masquer le verdict binaire par défaut**  
⇒ La solution restitue une **probabilité** ; le **verdict binaire** (`abandon` Oui/Non) reste **masqué par défaut** - non par anti-marquage (une probabilité affichée marque déjà), mais par **minimisation** (donnée dérivée superflue) et pour **ne pas trancher à la place de l'équipe** (art. 22) ni figer un seuil que le coût métier ne permet pas de calculer (D03/D04). Le responsable de traitement peut l'**activer** pour un tri à capacité donnée (§11). L'anti-stigmatisation réelle est **organisationnelle** (garde-fous d'usage ci-dessus), pas la forme de la sortie. De même, l'avertissement de l'art. 22 est **porté par la documentation** (model card). *Écarté* - exposer le verdict **par défaut** : plus lisible, mais livre une catégorie « à risque » transportable et une décision de fait automatisée.

1. **[D16] Retrait des blocs sans apport mesuré de l'artefact déployé (minimisation)**  
⇒ Les blocs candidats - leurres (D11), proxies non prédictifs (D14), variables LMS redondantes sauf une (D20) - sont retirés par défaut du pipeline déployé (§10) au titre de la minimisation (art. 5.1.c) ; l'ablation (§8) mesure le coût de leur retrait sur les décrocheurs, on ne conserve qu'en cas de chute matérielle, et SHAP (§12) confirme. Généralise D11. *Écarté* - embarquer une donnée que la finalité n'utilise pas au motif qu'elle serait inoffensive : contraire à la minimisation.

1. **[D17] Audit d'équité étendu aux modalités rares**  
⇒ Au-delà des attributs protégés (D13), l'audit d'équité (§12) couvre les modalités rares repérées en §6 : je vérifie que la performance ne s'effondre pas sur ces groupes peu représentés.

# 5. Chargement et compréhension des données [C3]

**Objectif** : établir ce que les fichiers contiennent, comment c'est écrit et ce qui cloche - doublons exacts, nombres restés en texte, dates multi-formats, casse et accents hétérogènes - puis normaliser ces écritures pour produire un DataFrame exploitable, support de l'EDA. Constat et mise en forme seuls : aucune ligne ni colonne n'est arbitrée ici.

## 5.1 Profilage des fichiers

> 🔧 **Comment je m'y prends** - un script Python profile chaque fichier dans un rapport HTML. Il applique d'abord un pré-nettoyage *de comparaison* (espaces, casse, accents), sans modifier les données, pour regrouper deux écritures d'une même valeur.

Par fichier, le rapport donne l'en-tête technique - délimiteur, encodage, taille, lignes, doublons - puis, par colonne, trois familles de mesures :

- nature - type pandas réel et type sémantique déduit (identifiant, constante, catégorielle, date, nombre…) ;
- écriture - motif dominant, nombre de motifs, distincts bruts vs normalisés : c'est là que se voient les formats hétérogènes ;
- distribution - bornes min/max, taux de nulls, et la liste des modalités pour les catégorielles.

> 🔎 Les cellules cliquables du rapport (`n_motifs`, `exemples`) déplient la liste complète.

In [ ]:
# `report_dir` déclenche l'écriture du rapport HTML complet - hors dépôt, car il cite
# des valeurs brutes. Sans lui, mesurer ne touche pas au disque.
profils = {
    chemin.name: profiling_utils.profile_csv(chemin, report_dir=settings.report_dir)
    for chemin in sorted(settings.raw_dir.glob("*.csv"))
}
profil_catalogue, profil_etudiants = sorted(profils.values(), key=lambda profil: profil.file.n_rows)

# Liens vers les rapports HTML complets
liens = "\n".join(
    f"- [{profil.report_path.name}]"
    f"(../{profil.report_path.relative_to(settings.root_dir).as_posix()})"
    for profil in profils.values()
    if profil.report_path is not None
)
display(Markdown(f"**> Rapports de profilage générés**\n\n{liens}"))

### Interprétation

Les rapports de profilage font ressortir trois défauts : des formats à normaliser, des synonymes à recoder, et - contrôle des bornes fait - aucune valeur hors domaine.

#### Fichier `catalogue_formations_V5`

- `filiere` compte **8 valeurs pour 8 lignes** : clé primaire de la table, retenue comme clé de jointure.
- Les autres colonnes ne sont que des attributs de la filière (faculté, ECTS, capacité, volume horaire, taux de réussite historique) - non jointes au jeu étudiant (D08, §3.5).

#### Fichier `decrochage_etudiants_complet_V5`

- **40 lignes strictement identiques** sur les 33 colonnes : des doublons exacts, à supprimer.
  
- Deux constats sans suite immédiate : 
  - `annee_universitaire` est constante (`2024-2025`, variance nulle, donc hors périmètre) 
  - `commentaire_tuteur` ne porte que 19 modalités - une catégorielle, pas du texte libre.

- **Formats hétérogènes à normaliser** - la valeur est juste, l'écriture non. Deux familles :
  - des nombres restés en texte : `date_inscription` (3 formats : `%Y-%m-%d`, `%d/%m/%Y`, `%d %b %Y`), `taux_presence_pct` et `distance_domicile_km` (unité collée + virgule décimale), `moyenne_partiels_s1` (virgule) ;
  - des catégorielles éclatées par la casse, les accents et les espaces : `filiere` (31→8), `bac_type` (12→7), `sexe` (11→8), `mention_bac` (11→8), `boursier` (8→6).

- **Synonymes à recoder** - une même modalité s'écrit de plusieurs façons. Chaque groupe est ramené à une forme canonique - minuscule, sans accent, libellé le plus explicite :
  - `sexe` : `f`/`femme` → femme · `h`/`m`/`homme` → homme · `nb`/`autre` → autre · `nr` → non renseigné ;
  - `bac_type` : `gen`/`general`/`generale` → general · `techno`/`technologique` → technologique · `pro`/`professionnel` → professionnel ;
  - `mention_bac` : `p`/`passable` → passable · `ab`/`assez bien` → assez bien · `b`/`bien` → bien · `tb`/`tres bien` → tres bien ;
  - `boursier` : `non`/`n`/`0` → non · `oui`/`o`/`1` → oui.

- **Aucune valeur hors domaine.** Le contrôle des bornes confirme que chaque colonne reste dans son intervalle métier :
  - `age` : 17-27 ans ;
  - `heures_travail_remunere_sem` : 0-28 h ;
  - `taux_presence_pct` : 34,8-99,9 % ;
  - `distance_domicile_km` : 0-138 km ;
  - `motivation`, `satisfaction`, `sentiment_appartenance` : 1-5 (échelle d'accord) ;
  - `moyenne_partiels_s1` : 0-20, `moyenne_finale` : 0,22-19,9 (notes /20).

### Nettoyage et recodage des écritures

Le profilage a révélé des écritures non conformes - nombres restés en texte, même modalité sous plusieurs orthographes, lignes en double - sur lesquelles aucune statistique n'est fiable : on ne mesure pas un nombre stocké en texte, `f` et `femme` comptent pour deux modalités, une ligne répétée pèse double. J'applique donc les décisions de mise en forme en trois temps, colonne par colonne, pour rendre le jeu **analysable** en §6 :

- **(1) nettoyage** - chaque colonne convertie selon ce qu'elle représente : les nombres en nombres (virgule décimale, unité collée), les dates en dates (les trois formats relevés), le reste ramené à une forme de comparaison (minuscules, sans accent, espaces réduits) ;
- **(2) recodage** - les modalités de même sens métier ramenées à un libellé de référence, pour qu'une modalité ne soit plus éclatée en plusieurs écritures ;
- **(3) dédoublonnage** - les lignes strictement identiques retirées, pour qu'aucune n'entre deux fois dans les comptes.

In [ ]:
# Jeu de travail : `preparation.transform` conforme les écritures, recode le vocabulaire métier et
# retire les doublons exacts - sans rien écrire sur disque (le palier, c'est §7).
df_etudiants_clean, transfo = preparation.transform(profil_etudiants.data, profil_etudiants.columns)

In [ ]:
# Contrôle : effet de la normalisation puis du recodage sur le nombre de modalités.
# `blank_to_na` exclut les cellules vides du décompte brut - comme le fait le rapport
# de profilage pour `n_distinct` : un blanc est une absence, pas une modalité.
SUIVIES = ("filiere", "bac_type", "mention_bac", "sexe", "boursier", "etablissement_origine")
apercu = pd.DataFrame(
    {
        "brut": {
            c: cleaning_utils.blank_to_na(profil_etudiants.data[c]).nunique() for c in SUIVIES
        },
        "normalisé": {
            c: cleaning_utils.normalize_text(profil_etudiants.data[c]).nunique() for c in SUIVIES
        },
        "recodé": {c: df_etudiants_clean[c].nunique() for c in SUIVIES},
    }
).rename_axis("colonne")
display(apercu)

brut = len(profil_etudiants.data)
display(
    Markdown(
        f"Lignes : **{brut}** → **{len(df_etudiants_clean)}** "
        f"(**{brut - len(df_etudiants_clean)}** doublons exacts retirés)"
    )
)

✅ Jeu de travail construit - écritures normalisées, modalités recodées, doublons exacts retirés

## 5.2 Contrôles qualités

Le profilage raisonne **colonne par colonne** : il ne voit pas les incohérences *entre* colonnes, *entre lignes*, ni *entre fichiers*. Je vérifie donc sur le jeu de travail trois relations que lui ne peut pas contrôler :
- l'unicité des identifiants (une ligne = un étudiant),
- la correspondance de `filiere` avec le catalogue (jointure possible),
- et trois inégalités métier ligne à ligne.

In [ ]:
# Chaque contrôle rend un NOMBRE D'ANOMALIES - 0 signifie cohérent. Aucun n'est visible
# au profil, qui raisonne colonne par colonne : ni les doublons d'un identifiant, ni la
# jointure au catalogue, ni les inégalités entre deux colonnes d'une même ligne.
reference = set(cleaning_utils.normalize_text(profil_catalogue.data["filiere"]))
filieres = cleaning_utils.normalize_text(df_etudiants_clean["filiere"])

# Inégalités métier à respecter ligne à ligne (colonne inférieure, colonne supérieure).
# Une ligne dont une valeur manque n'est pas comparable : écartée, jamais tenue conforme.
CONTRAINTES = (
    ("nb_devoirs_rendus", "nb_devoirs_total"),
    ("nb_ue_validees_s1", "nb_ue_total"),
    ("retards_rendus", "nb_devoirs_rendus"),
)

anomalies = {
    "Doublons de student_id": int(df_etudiants_clean["student_id"].duplicated().sum()),
    "Doublons de id_dossier": int(df_etudiants_clean["id_dossier"].duplicated().sum()),
    "Filières hors catalogue": len(set(filieres.dropna()) - reference),
}
for petite, grande in CONTRAINTES:
    paire = df_etudiants_clean[[petite, grande]].dropna()
    anomalies[f"Lignes où {petite} > {grande}"] = int((paire[petite] > paire[grande]).sum())

controles = pd.Series(anomalies, name="anomalies").rename_axis("Contrôle").to_frame()
display(controles)

### Interprétation

- **Six contrôles, zéro anomalie.** :
  - unicité (2) - `student_id` et `id_dossier` sans doublon : une ligne = un étudiant ;
  - jointure (1) - toutes les filières du jeu figurent au catalogue : l'appariement est possible ;
  - cohérence métier (3) - `nb_devoirs_rendus ≤ nb_devoirs_total`, `nb_ue_validees_s1 ≤ nb_ue_total`, `retards_rendus ≤ nb_devoirs_rendus`.

✅ Jeu de travail cohérent - prêt pour l'EDA §6

## 5.3 Bilan

**> Étudiants** (5 200 lignes, 33 colonnes)

| Colonne | Type | Traitement | % nuls | Bornes / modalités |
|---|---|---|---|---|
| `student_id` | identifiant (texte) | nettoyage | - | 5 200 modalités |
| `id_dossier` | identifiant (texte) | nettoyage | - | 5 200 modalités |
| `annee_universitaire` | constante (texte) | nettoyage | - | 1 modalité |
| `filiere` | catégoriel | nettoyage | - | 8 modalités |
| `sexe` | catégoriel | nettoyage + recodage | - | 4 modalités |
| `bac_type` | catégoriel | nettoyage + recodage | - | 3 modalités |
| `mention_bac` | catégoriel | nettoyage + recodage | 4 % | 4 modalités |
| `etablissement_origine` | catégoriel | nettoyage | 5 % | 4 modalités |
| `boursier` | binaire (oui/non) | nettoyage + recodage | - | 2 modalités |
| `groupe_td` | catégoriel | nettoyage | - | 8 modalités |
| `couleur_carte_etudiante` | catégoriel | nettoyage | - | 5 modalités |
| `jour_inscription` | catégoriel | nettoyage | - | 5 modalités |
| `commentaire_tuteur` | catégoriel | nettoyage | 49,29 % | 19 modalités |
| `date_inscription` | date | normalisation du format (3 formats) | - | 2024-09-02 → 2024-09-27 |
| `age` | entier | aucun | - | 17 → 27 |
| `distance_domicile_km` | décimal | normalisation du format (unité « km », virgule) | 6 % | 0 → 138 |
| `heures_travail_remunere_sem` | décimal | aucun | 12 % | 0 → 28 |
| `taux_presence_pct` | décimal | normalisation du format (unité « % », virgule) | - | 34,8 → 99,9 |
| `connexions_lms_30j` | entier | aucun | 4 % | 0 → 81 |
| `heures_lms_total` | décimal | aucun | - | 0 → 170,5 |
| `ressources_consultees` | entier | aucun | - | 0 → 222 |
| `retards_rendus` | entier | aucun | - | 0 → 10 |
| `nb_devoirs_total` | entier | aucun | - | 8 → 13 |
| `nb_devoirs_rendus` | entier | aucun | - | 0 → 13 |
| `messages_forum` | entier | aucun | - | 0 → 18 |
| `moyenne_partiels_s1` | décimal | normalisation du format (virgule) | 3 % | 0 → 20 |
| `nb_ue_total` | entier | aucun | - | 5 → 7 |
| `nb_ue_validees_s1` | entier | aucun | - | 0 → 7 |
| `motivation` | entier | aucun | 5 % | 1 → 5 |
| `satisfaction` | entier | aucun | 8 % | 1 → 5 |
| `sentiment_appartenance` | entier | aucun | 7 % | 1 → 5 |
| `moyenne_finale` | décimal | aucun | - | 0,22 → 19,9 |
| `abandon` | binaire (0/1) | aucun | - | 0 → 1 |

# 6. Analyse exploratoire (EDA) : visualisations et interprétation — journal de bord [C3]

**Objectif** : mesurer et interpréter le **jeu de travail conformé de §5** pour décider - cibles, manquants, valeurs extrêmes, variables à écarter - sans modifier aucune donnée

In [ ]:
eda.set_style()

# §6 mesure sur le JEU DE TRAVAIL CONFORMÉ de §5 (nombres typés, dates parsées, vocabulaire
# recodé) - le bronze littéral est inexplorable. §7 rejouera ces mêmes règles depuis la source.
jeu: pd.DataFrame = df_etudiants_clean
cible_abandon, cible_moyenne = jeu["abandon"], jeu["moyenne_finale"]

# Périmètre de scoring : ce qui existe à mi-S1. La fuite temporelle est cadrée en §3 - ici on
# s'appuie dessus, on ne la re-tranche pas. Identifiants et colonnes de fin de S1 sortent ;
# la constante `annee_universitaire` reste pour démontrer son absence de pouvoir prédictif,
# écartée en §7.
HORS_PERIMETRE = [
    "student_id",
    "id_dossier",
    "abandon",
    "moyenne_finale",  # cibles
    "moyenne_partiels_s1",
    "nb_ue_validees_s1",  # consolidées fin S1 - fuite (§3.3)
    "commentaire_tuteur",  # bilan rédigé fin S1 (§3, D07)
]
# `str(c)` : les noms de colonnes sont typés Hashable par pandas ; les forcer en `str` donne à
# explicatives / num_cols / cat_cols un type `list[str]`, sans quoi `jeu[num_cols]` serait vu
# comme `Series | DataFrame` et refusé par les fonctions attendant un DataFrame.
explicatives: list[str] = [str(c) for c in jeu.columns if str(c) not in HORS_PERIMETRE]
num_cols: list[str] = [c for c in explicatives if pd.api.types.is_numeric_dtype(jeu[c])]
dt_cols: list[str] = [c for c in explicatives if pd.api.types.is_datetime64_any_dtype(jeu[c])]
cat_cols: list[str] = [
    c
    for c in explicatives
    if not pd.api.types.is_numeric_dtype(jeu[c])
    and not pd.api.types.is_datetime64_any_dtype(jeu[c])
]

display(
    Markdown(
        f"Périmètre exploré : **{len(explicatives)}** explicatives "
        f"(**{len(num_cols)}** numériques, **{len(cat_cols)}** catégorielles, "
        f"**{len(dt_cols)}** temporelle) · "
        f"**{len(HORS_PERIMETRE)}** colonnes hors périmètre (cf §3)"
    )
)

## 6.1 Déséquilibre de la cible `abandon`

In [ ]:
# Barres horizontales : une couleur par classe, l'étiquette porte l'effectif ET la part.
balance = (
    eda.class_balance(cible_abandon)
    .assign(classe=lambda d: d["modalite"].map({"0": "réussite", "1": "abandon"}))
    .dropna(subset=["classe"])
    .sort_values("n", ascending=False)  # réussite en bas, abandon en haut
)
COULEURS = {"réussite": eda.PALETTE["series_1"], "abandon": eda.PALETTE["series_2"]}

figure, axis = plt.subplots(figsize=(6.4, 2.2))
axis.barh(
    balance["classe"], balance["n"], color=[COULEURS[c] for c in balance["classe"]], height=0.6
)
for y, (n, part) in enumerate(zip(balance["n"], balance["part_%"], strict=True)):
    axis.text(
        n,
        y,
        f"  {n} ({part:.1f} %)".replace(".", ","),
        va="center",
        ha="left",
        fontsize=8,
        color=eda.PALETTE["ink_secondary"],
    )
axis.set_xlabel("nombre d'étudiants")
axis.set_xlim(0, int(balance["n"].max() * 1.18))
axis.grid(axis="y", visible=False)
eda.show(figure)

display(
    Markdown(
        f"Prévalence de l'abandon **{100 * cible_abandon.mean():.1f} %** · "
        f"rapport de déséquilibre **{eda.imbalance_ratio(cible_abandon):.1f}:1**"
    )
)

### Interprétation

- La cible est déséquilibrée, mais pas au point de fausser l'apprentissage : **28,4 % d'abandon** contre 71,6 % de réussite, soit un rapport de 2,5:1. Deux conséquences en découlent :
  - l'écart est assez marqué pour qu'un modèle prédisant toujours « pas d'abandon » atteigne 71,6 % d'accuracy sans aucune valeur prédictive - l'évaluation se fera donc au rappel, au F1 et à l'AUC, jamais sur la seule accuracy ;
  - il reste trop modéré pour justifier un ré-échantillonnage (confirme D10) ; une pondération des classes (`class_weight`) suffira si besoin, à trancher au choix du modèle (§8/§9).

## 6.2 Lien entre les deux cibles

Les deux cibles sont-elles indépendantes, ou `abandon` n'est-il que le seuillage de `moyenne_finale` ?

In [ ]:
SEUIL_VALIDATION = 10  # une moyenne finale >= 10/20 vaut validation de l'année
recouvrement = jeu.groupby(cible_abandon)["moyenne_finale"].agg(["min", "max"])
coincide = (cible_abandon == (cible_moyenne < SEUIL_VALIDATION)).mean()

figure, axis = plt.subplots(figsize=(6.6, 3.2))
bornes = range(0, 21)
axis.hist(
    eda.as_float(cible_moyenne[cible_abandon == 0]),
    bins=bornes,
    color=eda.PALETTE["series_1"],
    alpha=0.85,
    label="réussite",
)
axis.hist(
    eda.as_float(cible_moyenne[cible_abandon == 1]),
    bins=bornes,
    color=eda.PALETTE["series_2"],
    alpha=0.85,
    label="abandon",
)
axis.axvline(
    SEUIL_VALIDATION,
    color=eda.PALETTE["ink_secondary"],
    linewidth=1.0,
    linestyle=(0, (4, 3)),
    label=f"seuil {SEUIL_VALIDATION}/20",
)
axis.set_xlabel("moyenne finale sur 20")
axis.set_ylabel("nombre d'étudiants")
axis.legend(loc="upper left")
eda.show(figure)

display(
    Markdown(
        f"**Meilleur note d'un décrocheur** : {recouvrement.loc[1, 'max']:g}<br>"
        f"**Pire note d'un reçu** : {recouvrement.loc[0, 'min']:g}<br>"
        f"`abandon == (moyenne_finale < {SEUIL_VALIDATION})` "
        f"sur **{100 * coincide:.0f} %** des lignes"
    )
)

### Interprétation

- **Les deux nuages ne se recouvrent pas** : le pire des reçus reste au-dessus du meilleur des décrocheurs. `abandon` n'est donc rien d'autre que `moyenne_finale` seuillée à 10/20 - la validation de l'année. La conséquence est double. `moyenne_finale` ne peut servir de variable explicative d'`abandon`, sous peine d'une fuite triviale - prédire le seuil à partir de la valeur seuillée. Elle reste pour autant une cible à part entière, traitée en régression pour prioriser et calibrer l'accompagnement (§2).

## 6.3 Distribution des variables explicatives

Je profile chaque explicative pour en connaître la **forme** - centre, dispersion et asymétrie des numériques, équilibre des modalités des catégorielles - et repérer ce qui appelle une décision de principe : variance nulle, échelles à harmoniser, modalité trop rare pour le découpage train / test.

In [ ]:
display(eda.numeric_overview(jeu[num_cols]))
eda.show(eda.plot_histograms(jeu[num_cols]))

In [ ]:
display(eda.modality_overview(jeu[cat_cols]))
eda.show(eda.plot_bars({c: eda.modality_counts(jeu[c]) for c in cat_cols}, labels="{:.0f}"))

display(Markdown("**> Modalités rares (< 5 %)**"))
display(eda.rare_modalities(jeu[cat_cols], max_share=0.05))

### Interprétation

Le profilage des explicatives ne révèle aucune anomalie de forme, seulement des points à traiter plus loin dans la chaîne.

- **Forme des numériques** 
  - Asymétrie : une seule variable est nettement asymétrique, `distance_domicile_km` (asymétrie 1,94, moyenne 15,2 > médiane 10,4), à imputer par la médiane (§6.4) ; `messages_forum` frôle le seuil (0,95) mais est complète, sans imputation à prévoir. 
  - Echelle : Les échelles sont très hétérogènes - de 1-5 (`motivation`) à 0-222 (`ressources_consultees`) - d'où une standardisation à placer dans le `Pipeline` (§8) dans le cas où le modèle choisi est une régression logistique.

- **Une constante à écarter** 
  - `annee_universitaire` ne porte qu'**une seule modalité** (`2024-2025`, 100 %) : variance nulle, aucun pouvoir prédictif - écartée de principe.
  - aucune autre variance nulle parmi les explicatives - écarts-types tous > 0, dominante la plus forte 69 % pour `etablissement_origine`.

- **Modalités des catégorielles** 
  - `filiere` (8 modalités, 8-17 %), `boursier` (38/62), `bac_type` et `mention_bac` sont équilibrées, rien de dégénéré.
  - Les trois leurres `couleur_carte_etudiante`, `groupe_td`, `jour_inscription` sont quasi uniformes (~20 % par modalité), cohérent avec des variables décoratives (§6.8). 
  - **Trois modalités rares (< 5 %)** portent sur `sexe` (« autre », « non renseigné ») et `etablissement_origine` (« autre ») - des modalités **déclarées**, pas des manquants, transmises telles quelles à l'encodeur (`handle_unknown="ignore"`, §8) pour ne pas disparaître au découpage train / test. Le repérage annoncé en §4.5 se règle ici : `sexe` est **hors modèle** (protégée, D13), sa rareté ne biaise aucune prédiction - laissée à l'audit d'équité (§12, D17) ; `etablissement_origine` (proxy, D14) **entre dans la comparaison** (maintien tranché en §8), mais « autre » étant déjà le fourre-tout, elle n'est pas regroupable - **signalée zone de moindre fiabilité** et couverte par le même audit.

## 6.4 Valeurs manquantes

§5 a compté les valeurs absentes ; reste à décider de leur **traitement**. Le mécanisme d'une absence ne s'observe pas (§3.4, D09) - je ne cherche pas à le prouver, mais à répondre aux **deux questions dont dépend le traitement** :

- **structure** - les trous se donnent-ils rendez-vous sur les mêmes dossiers ? (matrice et carte de nullité) - une concentration ouvrirait la voie à une suppression de lignes ;
- **signal** - manquer va-t-il de pair avec l'abandon ? Un manque informatif appellerait un **indicateur `x_manquant`** pour ne pas effacer ce qu'il porte ; un manque sans lien avec l'issue non.

In [ ]:
troues = eda.missing_overview(jeu[explicatives])
display(troues)

# Structure des trous : où sont-ils, et se donnent-ils rendez-vous sur les mêmes lignes ?
cols_troues = troues["colonne"].tolist()
eda.show(eda.plot_missing_matrix(jeu[cols_troues]))

In [ ]:
# Par paires : deux colonnes manquent-elles ensemble ? (corrélation de nullité φ)
eda.show(eda.plot_missing_heatmap(jeu[cols_troues]))

In [ ]:
# Signal : l'absence va-t-elle de pair avec l'abandon ? (oriente l'ajout d'un indicateur x_manquant)
ecarts = eda.missing_vs_target(jeu[explicatives], cible_abandon)
display(ecarts)
eda.show(
    eda.plot_intervals(
        ecarts.set_index("colonne"),
        value="ecart_pt",
        span="bruit_95_pt",
        labels="{:+.1f}",
        legend=("plafond de bruit (±)", "écart de taux (pts)"),
    )
)

### Interprétation

Le mécanisme d'une absence ne s'observe pas directement ; §3.4 (D09) l'a contourné par deux questions dont la réponse commande le traitement. L'EDA les tranche ici dans l'ordre : d'abord la structure des trous - se concentrent-ils ? -, puis leur signal - vont-ils de pair avec l'abandon ?

1. Les trous se concentrent-ils sur les mêmes dossiers ?

    La réponse décide s'il est envisageable de supprimer des lignes plutôt que d'imputer : la suppression n'aurait de sens que si une poignée de dossiers portaient l'essentiel des absences, et à la double condition d'une perte négligeable et d'une absence plausiblement aléatoire.

    - **Aucune corrélation entre colonnes** - la carte de nullité ne fait ressortir aucun coefficient qui se distingue de 0 : l'absence d'une colonne n'entraîne pas celle d'une autre.
    - **Aucune concentration par dossier** - la matrice de nullité montre des traits clairs dispersés, sans bande alignée : aucune ligne ne cumule les absences.

    Les trous sont diffus, répartis sur tout le jeu : la condition n'est pas réunie, aucune ligne n'est retirée. La supprimer biaiserait le jeu dès que l'absence n'est pas purement aléatoire, et sacrifierait des décrocheurs (classe rare) sans contrepartie.

2. L'absence est-elle un signal ? (lien avec `abandon`)

    La réponse décide s'il faut, avant d'imputer, ajouter un indicateur binaire `x_manquant` : il n'est utile que si le fait de manquer va de pair avec l'issue, auquel cas effacer le trou effacerait une information.

    - **Écarts dans le bruit** - entre présents et absents, chaque écart de taux d'abandon reste dans sa bande de bruit, au plus +2,8 pt pour un plafond de 3,7.
    - **Aucun écart significatif** - tous les tests donnent p ≥ 0,15, très au-dessus du seuil usuel de 0,05 en deçà duquel un lien se distinguerait du hasard.

    Manquer ne prédit donc pas l'abandon : aucun indicateur `x_manquant` n'est justifié, l'absence ne portant rien qu'il faille conserver.

**Conclusion** 

Aucune des deux conditions qui appelleraient un autre traitement n'est réunie - ni concentration (pas de suppression de lignes), ni signal (pas d'indicateur `x_manquant`) : l'imputation relève du `Pipeline`, sur le train seul (§8) ; les options ouvertes par D09 restent en jeu - **médiane**, **mode**, ou **valeur fixe** si l'absence est porteuse de sens. Ce choix concret, variable par variable, ne relève pas de l'EDA - il est appliqué en §8.

## 6.5 Valeurs extrêmes

Je cherche les valeurs **rares** par la règle de Tukey (écart interquartile, k = 1,5), sur les seules variables que le métier **ne borne pas**. Les échelles à domaine fixe - accord 1-5, pourcentage 0-100 - sont déjà contrôlées par leurs bornes en §5.3 ; les y soumettre confondrait le *rare* et le *légitime* (une échelle 1-5 ferait passer la réponse « 1 » pour une anomalie). L'IQR dit ici ce qui est **rare**, jamais ce qui est **faux**.

In [ ]:
# Règle de Tukey (k = 1,5) : borne_basse = Q1 - 1,5·IQR, borne_haute = Q3 + 1,5·IQR.
# Variables à domaine métier borné (échelles 1-5, pourcentage 0-100) : déjà contrôlées
# par leurs bornes en §5.3 - exclues ici, où Tukey confondrait le rare et le légitime.
BORNEES = ["taux_presence_pct", "motivation", "satisfaction", "sentiment_appartenance"]
non_bornees = [c for c in num_cols if c not in BORNEES]

extremes = eda.iqr_outliers(jeu[non_bornees])
display(extremes)
ordre = extremes["colonne"].tolist()  # graphiques alignés sur le tableau trié
masque = eda.iqr_mask(jeu[non_bornees])
display(
    Markdown(
        f"**{int(masque.any(axis=1).sum())}** lignes portent au moins une valeur hors IQR "
        f"(sur {len(jeu)})"
    )
)
eda.show(
    eda.plot_boxplots(
        jeu[ordre],
        annotations={r.colonne: f"{r.n_hors_iqr} hors IQR" for r in extremes.itertuples()},  # type: ignore
    )
)

### Interprétation

La règle de Tukey signale des valeurs rares, pas des valeurs fausses : reste à décider, au sens métier, si chaque extrême est une erreur à corriger (écrêtage, mise à l'écart) ou un profil réel à conserver.

- **Des queues à droite, toutes plausibles** - la règle ne relève que des valeurs hautes, jamais négatives (`min ≥ 0` partout), et aucune ne sort du domaine possible :
  - `distance_domicile_km` (jusqu'à 138 km), `heures_lms_total`, `ressources_consultees`, `messages_forum`, `connexions_lms_30j` : de l'engagement numérique et de l'éloignement, avec quelques profils extrêmes mais bien réels ;
  - `age` : borne haute vers 25,5 ans, quelques 26-27 - des reprises d'études, plausibles en L1.

Aucune valeur n'est anormale au sens métier : le rare est réel, rien n'est faux. Aucun écrêtage ni retrait n'est justifié - corriger ces extrêmes effacerait des étudiants qui existent.

## 6.6 Associations entre explicatives

Je mesure les liens **entre variables explicatives** - le seul terrain où une redondance justifie d'écarter une colonne. Une carte d'association **0→1 tous types confondus** (V de Cramér, η, |ρ| selon la paire) situe toutes les liaisons, catégorielle/numérique comprises ; une seconde carte, la corrélation **signée** de Spearman, rend le sens dans le bloc numérique.

Le lien **explicative↔cible n'est pas mesuré ici** : en présumer un pouvoir prédictif orienterait les choix de features sur la cible - ce que je m'interdis - et n'induirait qu'un a priori que l'explicabilité du modèle (§12, SHAP) peut démentir. Les seules mesures faisant intervenir la cible dans §6 démontrent une **absence** (leurres, §6.8) ou protègent un attribut (non-ré-encodage de `boursier`, §6.7).

In [ ]:
# Entre explicatives - le seul terrain où une redondance justifie d'écarter une colonne.
# Carte unifiée 0→1 (V de Cramér, η ou |ρ| selon la nature de la paire) : elle situe TOUTES les
# associations, y compris les couples catégorielle/numérique qu'aucune matrice par type ne voit.
# La carte signée de Spearman rend ensuite le sens que la première aplatit ; le tableau ne garde
# que les redondances fortes.
eda.show(
    eda.plot_heatmap(
        eda.association_matrix(jeu, num_cols, cat_cols),
        diverging=False,
        annotate=False,
        label="association 0→1 (V, η, |ρ|)",
    )
)
eda.show(
    eda.plot_heatmap(
        eda.correlation_matrix(jeu[num_cols]),
        diverging=True,
        annotate=True,
        label="ρ de Spearman",
    )
)
display(Markdown("**> Corrélations numériques fortes (|ρ| >= 0,8)**"))
display(eda.correlation_pairs(jeu[num_cols], threshold=0.8))

### Interprétation

La carte unifiée sépare deux régimes : un bloc numérique structuré, tout le reste au niveau du fond.

- **Catégorielles mutuellement indépendantes**, et indépendantes des numériques - leur bloc reste partout au niveau du fond, aucune liaison ne s'en détache. Rien à écarter pour redondance de ce côté.
- **Un bloc d'engagement** porte presque toute la structure numérique : présence, activité LMS, forum, devoirs rendus et ressenti déclaré co-varient, `retards_rendus` en négatif - l'attitude déclarée s'aligne sur le comportement observé.
- **Trio LMS quasi-redondant** - `connexions_lms_30j`, `ressources_consultees`, `heures_lms_total` co-varient de ρ 0,80 à 0,92 (les deux paires du tableau) : trois mesures d'une même activité.
- **Quatre variables isolées** - `age`, `distance_domicile_km`, `heures_travail_remunere_sem`, `nb_ue_total` ne rejoignent aucun bloc (`distance` frôle seule `sentiment_appartenance`, ρ −0,22).

Cette structure est de la **redondance, pas du pouvoir prédictif** - ces variables recouvrent la même information. Deux conséquences, aucune décision ici :

- **Modèle (§8)** - la colinéarité déstabilise un linéaire (à régulariser), laisse un ensemble d'arbres indifférent : argument pour comparer les deux familles.
- **Minimisation (§8.6)** - n'en garder qu'une se décidera par ablation : le coût de retrait des deux autres, nul attendu, tranchera - pas la corrélation seule.

**Aucun retrait en §6** - même le trio traverse le gold : on mesure avant de retirer (D16), le retrait des redondantes - en garder une - se décidera par ablation (§8.6), pas sur une corrélation entre explicatives.

## 6.7 Non-ré-encodage d'un attribut protégé - proxies vs `boursier`

`boursier` est exclu du modèle par principe (§4.4) ; reste à vérifier qu'aucun proxy socio-économique ne le **ré-encode** par une porte dérobée - ce serait une discrimination indirecte. 

Je mesure le lien de chaque proxy candidat avec `boursier` sur une échelle **0 (indépendance) → 1 (déterminisme)**, comparable d'un type à l'autre - **V de Cramér** pour la catégorielle, **rapport de corrélation η** pour les numériques - lu contre son **plafond de bruit** par permutation. Sur 5 200 lignes, la p-valeur ne tranche pas à elle seule ; c'est la **magnitude** qui dit s'il y a ré-encodage.

In [ ]:
# D14 (§4.4) : un proxy socio-économique ré-encode-t-il `boursier` (protégé, exclu) ? Mesure 0→1
# comparable entre types - V de Cramér (catégorielle) ou η (numérique) - lue contre son bruit.
PROXIES = ["etablissement_origine", "heures_travail_remunere_sem", "distance_domicile_km"]
proximite = eda.association_with(jeu, PROXIES, jeu["boursier"])
display(proximite)
eda.show(
    eda.plot_intervals(
        proximite.set_index("colonne"),
        value="association",
        span="bruit_95",
        labels="{:.3f}",
        legend=("plafond de bruit (permutation)", "association observée"),
    )
)

### Interprétation

- **Les trois proxies sous leur plafond de bruit** - `etablissement_origine` (V 0,027 < 0,044) · `distance_domicile_km` (η 0,007 < 0,027) · `heures_travail_remunere_sem` (η 0,003 < 0,031), p ≥ 0,47 : aucun ne ré-encode `boursier`.
- Le lien avec `boursier` est **indiscernable du hasard** - aucune discrimination indirecte à redouter.

La décision qui en résulte - l'entrée des trois proxies au modèle - est portée au **journal [D14]**.

## 6.8 Leurres annoncés

Trois variables annoncées comme décoratives (§3.6) - `groupe_td`, `couleur_carte_etudiante`, `jour_inscription`. Plutôt que de les croire nulles, je le **démontre** par deux contrôles :

- une **redondance déterministe** - `jour_inscription` se recalcule-t-il depuis `date_inscription` ?
- un **lien bivarié avec `abandon`**, jugé sur sa magnitude.

Le verdict multivarié - le seul qui tranche vraiment - reviendra à l'ablation en §8.6 (coût de retrait). Elles restent dans le gold jusque-là.

In [ ]:
# `jour_inscription` se recalcule-t-il depuis `date_inscription` ? (dérivation déterministe)
JOURS = {
    0: "lundi",
    1: "mardi",
    2: "mercredi",
    3: "jeudi",
    4: "vendredi",
    5: "samedi",
    6: "dimanche",
}
jour_recalcule = jeu["date_inscription"].dt.weekday.map(JOURS)
comparables = jeu["jour_inscription"].notna() & jour_recalcule.notna()
coincide_jour = (jeu.loc[comparables, "jour_inscription"] == jour_recalcule[comparables]).mean()
display(
    Markdown(
        f"`jour_inscription` = jour de semaine de `date_inscription` sur "
        f"**{100 * coincide_jour:.1f} %** des lignes comparables"
    )
)

In [ ]:
# Point 2 - le lien de chaque leurre avec la cible sort-il du bruit ? Même mesure et même
# lecture qu'en §6.7, `abandon` remplaçant l'attribut protégé.
LEURRES = ["groupe_td", "couleur_carte_etudiante", "jour_inscription"]
liens_leurres = eda.association_with(jeu, LEURRES, cible_abandon)
display(liens_leurres)
eda.show(
    eda.plot_intervals(
        liens_leurres.set_index("colonne"),
        value="association",
        span="bruit_95",
        labels="{:.3f}",
        legend=("plafond de bruit (permutation)", "association observée"),
    )
)

### Interprétation

Deux contrôles convergents ; l'un livre un résultat contraire à l'attendu, à ne pas surinterpréter.

- `jour_inscription` **reproduit intégralement** le jour de semaine de `date_inscription` : une réécriture sans information propre, sans intérêt prédictif au-delà de `date_inscription`.
- `groupe_td` et `jour_inscription` restent **dans leur bande de bruit** : leur lien bivarié avec `abandon` est indiscernable du hasard - l'attendu pour un leurre.
- **`couleur_carte_etudiante` sort de sa bande** - contraire à l'attendu pour un leurre. L'écart tient à la *taille de l'effet*, non à sa réalité : le V reste infime, collé à zéro, et connaître la couleur de carte ne déplace quasiment pas la probabilité d'abandon. Sur 5 200 lignes, un effet négligeable devient « significatif » (p < 0,05) sans cesser d'être négligeable - **la magnitude mesure le pouvoir prédictif ; la p-valeur, seulement l'existence du lien** (§6.7).
- Le bivarié ne tranche pas seul - un leurre pourrait peser par interaction, ou ne pas peser malgré un lien brut. Le **verdict du retrait revient à l'ablation multivariée** : §8.6 mesurera le coût de retrait des leurres - nul attendu - et actera leur suppression par minimisation (D11, D16).

## _Journal de bord des décisions_

1. **[D22] Seuil de rareté des modalités - 5 %**  
⇒ Le repérage des modalités sous-représentées annoncé en §4.5 se tranche ici. Seuil à **5 %** : au-dessous, l'effectif d'un groupe (quelques dizaines de lignes, une poignée de décrocheurs) est trop mince pour qu'un audit d'équité en conclue quoi que ce soit. Le seuil fait remonter `etablissement_origine = autre` (proxy au modèle, D14), invisible à 1 % - déjà le fourre-tout, non regroupable, donc signalée zone de moindre fiabilité et couverte par l'audit §12 (D17) ; les modalités rares de `sexe` (protégée, D13) restent hors modèle et relèvent du même audit.

1. **[D09] Manquants - imputation par défaut : médiane/mode, sans indicateur, sans suppression**  
⇒ Diagnostic mené en §6. Ni **concentration** (trous diffus, aucun dossier ne les cumule - pas de suppression de lignes), ni **signal** (l'absence ne va pas de pair avec l'abandon - pas d'indicateur `x_manquant`) : l'imputation relève du `Pipeline`, train seul (anti-fuite, §8) : les options ouvertes par D09 restent en jeu - médiane, mode, ou valeur fixe si l'absence a une signification propre - le choix concret par colonne est appliqué en §8. *Écarté* - **supprimer les lignes trouées** : les trous étant diffus, on sacrifierait des décrocheurs (classe rare) sans contrepartie. *Écarté* - **un indicateur `x_manquant`** : justifié seulement si l'absence prédit l'issue, ce que la mesure infirme.

1. **[D19] Extrêmes (Tukey) - ni écrêtage ni retrait**  
⇒ Les queues hautes signalées sont rares mais **réelles** : toutes dans le domaine possible, aucune aberrante au sens métier. Conservées telles quelles dans le gold. *Écarté* - **winsoriser / écrêter** : corrigerait un problème qui n'existe pas et effacerait des profils réels (éloignés, reprises d'études) susceptibles de porter le signal.

1. **[D20] Colinéarité (bloc d'engagement) - aucun retrait en §6, différé à l'ablation §8**  
⇒ Le trio LMS (`connexions_lms_30j`, `ressources_consultees`, `heures_lms_total`) mesure une même activité ; les trois traversent le gold, le retrait éventuel - en garder une - se décide par ablation (§8) - la redondance n'est pas un pouvoir prédictif, un ensemble d'arbres l'encaisse, l'ablation tranche en multivarié et SHAP (§12) confirme. *Écarté* - **retirer une des trois dès §6** sur la seule corrélation : présume ce que l'ablation §8 doit mesurer. Généralise l'esprit de D16 (mesurer avant de trancher le retrait).

1. **[D14] Proxies socio-économiques - condition RGPD levée, minimisation en §8**  
⇒ `etablissement_origine`, `heures_travail_remunere_sem`, `distance_domicile_km` peuvent laisser deviner la situation économique, comme `boursier` (exclu, D13) ; leur lien avec `boursier` reste indiscernable du hasard (§6.7) - aucun ne le ré-encode : la condition RGPD posée en §4 est levée, les trois entrent dans la comparaison ; leur maintien dans l'artefact relève ensuite de la minimisation, mesurée par ablation (§8, D16). *Écarté* - **les exclure par précaution** : priverait le modèle de variables prédictives légitimes alors que le contrôle a écarté le risque de discrimination indirecte. *Écarté* - **les inclure sans contrôle** : laisserait passer un éventuel ré-encodage du protégé.

# 7. Préparation des données (nettoyage, manquants, transformations, features) — journal de bord [C3]

**Objectif** : rejouer depuis les fichiers reçus les règles de mise en forme de §5, appliquer les décisions de §6, et produire le jeu de référence - **bronze → silver → gold**.

## 7.1 Bronze - copie fidèle des fichiers reçus

Je repars des fichiers déposés dans `raw/` - la source, toujours présente et jamais modifiée - et non du jeu de travail de §5. Le bronze en est la **copie exacte**, immuable : l'assurance de pouvoir rejouer toute la chaîne depuis l'origine si une règle de conformation se révélait fausse.

In [ ]:
# Rechargement depuis la source : `profile_csv` lit et profile chaque fichier reçu (le profil
# pilotera la conformation en 7.2). `.data` est le texte brut, tel qu'écrit - le bronze en mémoire.
fichiers_recus = sorted(settings.raw_dir.glob("*.csv"))
profils_bronze = {chemin.name: profiling_utils.profile_csv(chemin) for chemin in fichiers_recus}
profil_catalogue, profil_etudiants = sorted(
    profils_bronze.values(), key=lambda profil: profil.file.n_rows
)
df_etudiants_bronze = profil_etudiants.data

# Copie-octet des fichiers reçus (et non un `to_csv` qui reformaterait), seulement si demandé.
if ENREGISTRER_PALIERS:
    settings.bronze_dir.mkdir(parents=True, exist_ok=True)
    for chemin in fichiers_recus:
        shutil.copy(chemin, settings.bronze_dir / chemin.name)

display(
    Markdown(
        f"Bronze : **{len(profils_bronze)}** fichiers reçus, "
        f"**{len(df_etudiants_bronze)}** lignes, **{df_etudiants_bronze.shape[1]}** colonnes"
        f" · écriture disque : **{'oui' if ENREGISTRER_PALIERS else 'non'}**"
    )
)

## 7.2 Silver - écritures conformées, vocabulaire recodé, doublons retirés

Les trois règles de §5, rejouées à l'identique par le même code (`preparation.transform`) : nombres et dates typés, synonymes ramenés à leur forme canonique, doublons exacts retirés. Aucune colonne n'est perdue - c'est exactement le jeu que §6 a exploré.

In [ ]:
df_etudiants_silver, transfo_silver = preparation.transform(
    profil_etudiants.data, profil_etudiants.columns
)

if ENREGISTRER_PALIERS:
    settings.silver_dir.mkdir(parents=True, exist_ok=True)
    df_etudiants_silver.to_csv(settings.silver_dir / "etudiants_silver.csv", index=False)

display(
    Markdown(
        f"Silver : **{transfo_silver.n_rows_source}** → **{transfo_silver.n_rows}** lignes "
        f"(**{transfo_silver.n_duplicates_removed}** doublons retirés), "
        f"**{transfo_silver.n_columns}** colonnes conservées · "
        f"écriture disque : **{'oui' if ENREGISTRER_PALIERS else 'non'}**"
    )
)

## 7.3 Gold - exclusions de principe et variables dérivées

Le gold applique au silver les décisions de §6 et le cadrage de §3-§4 : 
- Suppression : 
  - identifiants (§4.2) : `student_id` et `id_dossier`,
  - résultats consolidés en fin de S1 (§3.3) : `moyenne_partiels_s1`, `nb_ue_validees_s1` et `commentaire_tuteur`
  - constante (§6.3) : `annee_universitaire`,
- Ajout de 2 features d'engagement :
  - `taux_rendu` = `nb_devoirs_rendus` / `nb_devoirs_total` ;
  - `ratio_retards` = `retards_rendus` / `nb_devoirs_total`.  

Ce qui reste traverse le gold sans arbitrage : 
   - leurres (D11), 
   - proxies (D14)
   - et trio LMS (D20) 
qui attendent le verdict d'ablation de §8 (minimisation), confirmé en §12. 

Aucune imputation ici - D09 la reporte au `Pipeline`, ajustée sur le train seul (§8). Les cibles sont conservées : le split X/y est en §8.

In [ ]:
# Colonnes retirées PAR PRINCIPE - chaque retrait cite la décision qui le fonde. Les cibles
# (abandon, moyenne_finale) n'y sont pas : le gold les conserve pour l'entraînement (§8).
EXCLUES_GOLD = [
    "student_id",
    "id_dossier",  # identifiants - minimisation (§4.2)
    "moyenne_partiels_s1",
    "nb_ue_validees_s1",  # consolidés fin S1 - fuite temporelle (§3.3)
    "commentaire_tuteur",  # bilan rédigé fin S1 (§3.3, D07)
    "annee_universitaire",  # constante, 1 modalité (mesurée §6.3) - variance nulle
]
df_etudiants_gold, faits_gold = preparation.build_gold(df_etudiants_silver, EXCLUES_GOLD)

if ENREGISTRER_PALIERS:
    settings.gold_dir.mkdir(parents=True, exist_ok=True)
    df_etudiants_gold.to_csv(settings.gold_dir / "etudiants_gold.csv", index=False)

display(
    Markdown(
        f"Gold : **{len(faits_gold.dropped)}** colonnes retirées par principe, "
        f"**{len(faits_gold.derived)}** features dérivées "
        f"({', '.join(f'`{f}`' for f in faits_gold.derived)}), "
        f"forme finale **{faits_gold.n_rows}** lignes, **{faits_gold.n_columns}** colonnes · "
        f"écriture disque : **{'oui' if ENREGISTRER_PALIERS else 'non'}**"
    )
)
display(df_etudiants_gold[list(faits_gold.derived)].describe().round(3))

### Variables dérivées

Deux taux d'engagement, calculés ligne à ligne depuis des comptes déjà présents :

- `taux_rendu` = `nb_devoirs_rendus` / `nb_devoirs_total` - part des devoirs attendus effectivement rendus.
- `ratio_retards` = `retards_rendus` / `nb_devoirs_total` - part des devoirs attendus rendus en retard.

Les rapporter au nombre de devoirs **attendus** les rend comparables d'un étudiant à l'autre : un compte brut (« 3 retards ») ne dit rien sans le volume qui le porte. Le choix du dénominateur de `ratio_retards` - le total attendu plutôt que les seuls devoirs rendus - est défendu au journal (D21).

✅ Palier gold construit - jeu de référence prêt, deux features dérivées incluses

## _Journal de bord des décisions_

1. **[D21] `ratio_retards` - dénominateur = `nb_devoirs_total`, non `nb_devoirs_rendus`**  
⇒ La part de devoirs rendus en retard se rapporte au nombre de devoirs **attendus**, pour rester comparable d'un étudiant à l'autre. *Écarté* - **`retards_rendus / nb_devoirs_rendus`** : un étudiant ne rendant qu'un seul devoir, en retard, afficherait 100 % - un artefact de faible base, incomparable à qui a rendu vingt devoirs. Rapporter au total attendu évite ce biais d'échelle.

# 8. Choix du modèle et démarche scientifique (baseline, modèles, comparaison) — journal de bord [C4]

**Objectif** : arrêter une **famille de modèle** et un **jeu de variables** pour prédire l'abandon, sur une évaluation qui met de côté un jeu de test jusqu'à la mesure finale (§12).

**Les choix structurants** - chacun est argumenté à sa sous-section, puis acté au journal de bord en fin de §8 :

- **Évaluer sans tricher** - un jeu de **test** (20 %) mis de côté jusqu'à §12, le reste comparé par validation croisée (§8.1).
- **Une méthode de comparaison explicite** - baseline, familles candidates et mesures, posées *avant* de mesurer (§8.3).
- **Minimiser les variables par principe** (RGPD) - et mesurer ce que le retrait coûte (§8.6, §8.10).
- **Prédire aussi la note finale** en cible secondaire (§8.7).

In [ ]:
# Le gold produit en §7 est repris en mémoire ; l'option CHARGER_GOLD_DEPUIS_CSV (en tête
# de notebook) le relit depuis le disque pour exécuter §8 indépendamment de l'amont.
if CHARGER_GOLD_DEPUIS_CSV:
    df_gold = pd.read_csv(settings.gold_dir / "etudiants_gold.csv")
else:
    df_gold = df_etudiants_gold

# Le gold en mémoire porte des dtypes pandas nullables (Int64, string) issus de la
# conformation ; scikit-learn travaille en numpy - on aligne l'entrée sur la forme CSV.
df_gold = preprocessing.to_numpy_dtypes(df_gold)

SEED = 42
CIBLE = "abandon"
CIBLE_REGRESSION = "moyenne_finale"
df_gold[CIBLE] = df_gold[CIBLE].astype(int)

display(
    Markdown(
        f"Gold repris : **{df_gold.shape[0]}** lignes × **{df_gold.shape[1]}** colonnes · "
        f"taux d'`abandon` **{df_gold[CIBLE].mean():.1%}**"
    )
)

## 8.1 Partition train / test et validation croisée

On met de côté **20 % des étudiants** comme jeu de **test**, qu'aucune décision ne regardera avant la mesure finale (§12) : c'est la seule façon d'estimer honnêtement ce que vaudra le modèle sur des étudiants jamais vus. Les 80 % restants servent à entraîner et comparer, par **validation croisée** - le train est découpé en **5 folds**, chaque fold étant prédit tour à tour par un modèle entraîné sur les 4 autres. Chaque étudiant reçoit ainsi une probabilité issue d'un modèle qui **ne l'a pas vu** (prédiction dite *out-of-fold*) - une comparaison sans complaisance. Deux réglages, chacun pour une raison :

- **découpe stratifiée** - elle conserve la même proportion d'abandons dans le test et dans chaque fold ; sinon un tirage défavorable sur une classe minoritaire (~28 %) ferait varier la mesure au petit bonheur ;
- **5 folds** - compromis habituel biais/variance : assez de décrocheurs par fold pour une mesure stable, sans alourdir le calcul (utile pour l'optimisation de §9).

In [ ]:
train_df, test_df = protocol.make_split(df_gold, CIBLE, test_size=0.2, seed=SEED)
cv = protocol.make_cv(n_splits=5, seed=SEED)

display(Markdown("**> Répartition train / test**"))
display(
    pd.DataFrame(
        {
            "Lignes": [len(train_df), len(test_df)],
            "Taux abandon": [f"{train_df[CIBLE].mean():.1%}", f"{test_df[CIBLE].mean():.1%}"],
        },
        index=pd.Index(["train", "test"], name="Partition"),
    )
)

### Interprétation

- Le taux d'abandon est **quasi préservé** de part et d'autre (28,4 % au train, 28,5 % au test) : le test reflète fidèlement la population, aucune mesure ne s'en trouvera biaisée.

## 8.2 Encodage et imputation des variables

Un modèle ne lit que des nombres : il faut **compléter les valeurs manquantes** et **transformer les catégories** en colonnes. Ces opérations s'apprennent sur les données (une médiane, un vocabulaire) : elles vivent donc **dans** le pipeline et se ré-apprennent à chaque fold - jamais calées d'avance sur tout le jeu, ce qui laisserait fuiter de l'information. Le vocabulaire des catégories est en revanche **déclaré d'avance** (dans `preparation`), non déduit du train, pour qu'une modalité rare ne disparaisse pas d'un fold à l'autre.

Le plan de traitement, variable par variable :

| Traitement (dans le pipeline) | Variables | Pourquoi |
|---|---|---|
| **Imputation médiane** (+ mise à l'échelle pour le linéaire) | les **17 numériques** - âge, présence, activité LMS, devoirs, forum, distances, heures, et les échelles de Likert `motivation`/`satisfaction`/`sentiment_appartenance` | - valeur centrale robuste aux extrêmes <br> - l'absence ne vaut pas 0 (ne pas avoir renseigné ses heures de travail n'est pas « 0 heure ») <br> - l'échelle sert au linéaire, inutile aux arbres |
| **Encodage ordinal** + imputation médiane | `mention_bac` | - l'ordre (passable < assez bien < bien < très bien) porte l'information <br> - Imputation des NaN par la **médiane** car une modalité « inconnu » ne pourrait pas s'ordonner dans l'échelle |
| **One-hot** (vocabulaire déclaré) | `filiere`, `bac_type`, `etablissement_origine`, `groupe_td`, `couleur_carte_etudiante`, `jour_inscription` | - catégories → colonnes 0/1 <br> - un `etablissement_origine` absent forme sa propre modalité `inconnu` - l'absence est elle-même un renseignement (D09) |
| **Retirées de la matrice** | `abandon`, `moyenne_finale` (cibles) · `sexe`, `boursier` (protégées, D13) · `date_inscription` | - ce sont les cibles <br> - les protégées sont hors modèle par principe <br> - une date brute est inexploitable telle quelle |

In [ ]:
# Typage des colonnes - un JUGEMENT (Likert en numérique, protégées et date exclues), posé
# ici et passé au préprocesseur. L'inventaire des modalités, lui, est un FAIT déclaré dans
# preparation (garde-fou : déclaré, jamais déduit du seul train).
CIBLES = [CIBLE, CIBLE_REGRESSION]
PROTEGEES = ["sexe", "boursier"]  # hors modèle par principe (D13), gardées pour l'audit §12
EXCLUES_MATRICE = [
    "date_inscription"
]  # date brute inexploitable ; jour de semaine = leurre jour_inscription

ORDINALES = ["mention_bac"]
NOMINALES = [
    "filiere",
    "bac_type",
    "etablissement_origine",
    "groupe_td",
    "couleur_carte_etudiante",
    "jour_inscription",
]
NUMERIQUES = [
    c
    for c in df_gold.columns
    if c not in CIBLES + PROTEGEES + EXCLUES_MATRICE + ORDINALES + NOMINALES
]
FEATURES = NUMERIQUES + ORDINALES + NOMINALES

X_train, y_train = train_df[FEATURES], train_df[CIBLE]

# Modalités : socle déclaré + toute modalité inédite du train (extend) ; inédites signalées.
_, inedites = preprocessing.resolve_categories(preparation.NOMINAL_MODALITIES, train_df, NOMINALES)


def construire_preprocesseur(features, *, scale):
    """Restreint le préprocesseur aux colonnes de `features` - tel quel et pour l'ablation."""
    num = [c for c in NUMERIQUES if c in features]
    ordc = [c for c in ORDINALES if c in features]
    nomc = [c for c in NOMINALES if c in features]
    oc, _ = preprocessing.resolve_categories(
        preparation.ORDINAL_MODALITIES, train_df, ordc, extend=False
    )
    nc, _ = preprocessing.resolve_categories(
        preparation.NOMINAL_MODALITIES, train_df, nomc, extend=True
    )
    return preprocessing.make_preprocessor(
        numeric=num,
        ordinal=ordc,
        onehot=nomc,
        ordinal_categories=oc,
        onehot_categories=nc,
        scale=scale,
    )


n_colonnes = construire_preprocesseur(FEATURES, scale=False).fit_transform(X_train).shape[1]
interdites = [c for c in CIBLES + PROTEGEES + EXCLUES_MATRICE if c in FEATURES]

display(
    Markdown(
        f"**{len(FEATURES)}** variables en entrée → **{n_colonnes}** colonnes après encodage · "
        f"colonnes interdites présentes : **{interdites or 'aucune'}** · "
        f"modalités inédites au train : **{inedites or 'aucune'}**"
    )
)

## 8.3 Cible principale - Méthode de comparaison

Avant de mesurer, on fixe **avec quoi** on compare - sinon la comparaison n'a pas de sens.

**La baseline** - une **régression logistique nue** sur le jeu complet. Simple, rapide, directement interprétable : c'est le **point de repère** minimal. Un modèle plus complexe ne se justifie que s'il fait mieux qu'elle.

**Les familles comparées** - à la baseline linéaire s'ajoutent deux modèles à base d'**arbres de décision**, la **forêt aléatoire** et **XGBoost**, capables de capter des effets non linéaires et des interactions. Opposer un linéaire à des arbres est la question laissée ouverte par l'exploration (§6.6), où le bloc d'engagement s'est révélé fortement corrélé - une colinéarité qui gêne en théorie un linéaire, mais laisse les arbres indifférents.

**Les mesures** - le modèle rend une **probabilité d'abandon** (entre 0 et 1), pas une étiquette « oui / non » : le seuil de déclenchement est une décision séparée, prise en §9 (D15). On juge donc la probabilité sur des critères **indépendants du seuil** :

- **ROC-AUC** - probabilité de classer un vrai décrocheur au-dessus d'un non-décrocheur ; 0,5 = hasard, 1 = parfait. **Critère de départage** (D03) ;
- **PR-AUC** (précision-rappel) - même logique, centrée sur les **décrocheurs**, la classe minoritaire ;
- **Brier** + **courbe de calibration** - la probabilité annoncée est-elle **fiable** ? (quand le modèle annonce 30 %, observe-t-on ~30 % d'abandons ?), Brier plus bas = mieux.

## 8.4 Cible principale - Baseline

On entraîne la baseline définie en §8.3 et on lit son niveau - la barre à dépasser.

In [ ]:
preproc_lineaire = construire_preprocesseur(FEATURES, scale=True)
preproc_arbre = construire_preprocesseur(FEATURES, scale=False)
modeles = families.build_classifiers(
    preproc_linear=preproc_lineaire, preproc_tree=preproc_arbre, seed=SEED
)

proba_baseline = evaluation.oof_proba(modeles["logreg"], X_train, y_train, cv)
m_baseline = evaluation.classification_metrics(y_train, proba_baseline)
display(
    pd.DataFrame(
        {
            "Baseline (LogReg, OOF)": [
                f"{m_baseline[k]:.3f}" for k in ("roc_auc", "pr_auc", "brier")
            ]
        },
        index=pd.Index(["ROC-AUC", "PR-AUC", "Brier"], name="Mesure"),
    )
)
evaluation.plot_score_distribution(y_train, proba_baseline)
plt.show()

### Interprétation

- **La barre est déjà haute** - l'histogramme sépare nettement les deux populations : les décrocheurs se massent vers les probabilités élevées, le reste vers les basses. 
- La **zone de recouvrement** au centre est précisément là où un seuil devra trancher (§9).

## 8.5 Cible principale - Comparaison des familles

On oppose la baseline aux deux modèles à arbres, à **armes égales** (mêmes folds, même préparation, aucune pondération de classe - réglage renvoyé à §9, D10).

> 🔎 **Deux façons de résumer les 5 folds.**
> - Le **tableau** ci-dessous donne la ROC-AUC **par fold** : une AUC calculée séparément sur chacun des 5 folds de validation (~832 étudiants), puis leur **moyenne ± écart-type σ** - le σ dit si l'écart entre familles dépasse le bruit d'un tirage. 
> - Les **courbes** plus bas sont tracées autrement : sur les probabilités *out-of-fold regroupées* (chaque étudiant prédit par le modèle qui l'a mis de côté, soit un seul jeu de 4160 probabilités classées ensemble) - un tracé unique, sans dispersion.

In [ ]:
# Comparaison en moyenne par fold de validation (± écart-type pour le critère de départage) :
# un écart de moyenne ne compte que s'il dépasse la dispersion inter-folds.
familles_scores = {}
for nom, pipe in modeles.items():
    auc = evaluation.cv_scores(pipe, X_train, y_train, cv, scoring="roc_auc")
    ap = evaluation.cv_scores(pipe, X_train, y_train, cv, scoring="average_precision")
    brier = -evaluation.cv_scores(pipe, X_train, y_train, cv, scoring="neg_brier_score")
    familles_scores[nom] = {
        "ROC-AUC (moy. folds)": auc.mean(),
        "σ ROC-AUC": auc.std(),
        "PR-AUC (moy. folds)": ap.mean(),
        "Brier (moy. folds)": brier.mean(),
    }
display(pd.DataFrame(familles_scores).T.rename_axis("Famille").round(4))

- **La régression logistique mène** sur les trois mesures - meilleur classement (ROC-AUC, PR-AUC) et probabilités les mieux calibrées (Brier le plus bas).
- **Ecart de rang modeste** - elle ne devance la forêt que d'environ un écart-type inter-folds (σ) : le classement seul ne tranche pas. Ce qui départage se lit plus bas, dans la calibration et le sur-apprentissage.

In [ ]:
# Courbes sur les prédictions OOF regroupées (une courbe tracée sur tous les points).
probas = {nom: evaluation.oof_proba(pipe, X_train, y_train, cv) for nom, pipe in modeles.items()}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
evaluation.plot_roc(y_train, probas, ax=axes[0])
evaluation.plot_pr(y_train, probas, ax=axes[1])
evaluation.plot_calibration(y_train, probas, ax=axes[2])
plt.tight_layout()
plt.show()

# Sur-apprentissage : ROC-AUC en resubstitution (entraînement) vs validation croisée.
diagnostic = (
    pd.DataFrame(
        [
            {"Famille": nom, **evaluation.train_vs_oof(pipe, X_train, y_train, cv)}
            for nom, pipe in modeles.items()
        ]
    )
    .set_index("Famille")
    .rename(columns={"train_roc_auc": "ROC-AUC (entraînement)", "oof_roc_auc": "ROC-AUC (OOF)"})
)
diagnostic["écart"] = diagnostic["ROC-AUC (entraînement)"] - diagnostic["ROC-AUC (OOF)"]
display(diagnostic.round(3))

### Interprétation

- **Courbes** (sur prédictions OOF regroupées) - ROC et précision-rappel confirment le classement ; la régression logistique colle le mieux à la diagonale de calibration.
- **Sur-apprentissage** - la forêt et XGBoost atteignent un score **parfait (1,0) à l'entraînement** mais bien plus bas en validation : elles **mémorisent** les exemples. La régression logistique obtient des scores quasi identiques à l'entraînement et en validation - elle **généralise**, malgré la colinéarité du bloc d'engagement (§6.6) qu'on redoutait pour un linéaire.

## 8.6 Cible principale - Minimisation

Par principe de **minimisation** (ne garder que le nécessaire - art. 5.1.c du RGPD), on cherche à **retirer** des variables plutôt qu'à en ajouter : les trois **leurres** (D11), les trois **proxies socio-économiques** (D14) et les variables **LMS redondantes** (D20). Ces dernières mesurent la même activité (corrélées de 0,80 à 0,92, §6.6) : on n'en garde **qu'une**, `heures_lms_total` - un choix en partie arbitraire, mais `heures_lms_total` n'a **aucun manquant** (argument favorable) et mesure directement un volume d'engagement.

Pour chaque bloc, on mesure ce que son retrait **coûte** - c'est l'**ablation** : on ré-entraîne sans le bloc et on compare au jeu complet. On ne conserve un bloc que si son retrait dégrade **nettement** la détection des décrocheurs.

In [ ]:
FAMILLE_RETENUE = "logreg"  # verdict de §8.5, porté au journal
blocs = {
    "leurres": ["groupe_td", "couleur_carte_etudiante", "jour_inscription"],
    "proxies": ["etablissement_origine", "heures_travail_remunere_sem", "distance_domicile_km"],
    "LMS redondantes": [
        "connexions_lms_30j",
        "ressources_consultees",
    ],  # heures_lms_total conservée
}


def construire_pipeline_retenu(features):
    """Le pipeline de la famille retenue, préprocesseur restreint à `features` (pour l'ablation)."""
    pl = construire_preprocesseur(features, scale=(FAMILLE_RETENUE == "logreg"))
    pt = construire_preprocesseur(features, scale=False)
    return families.build_classifiers(preproc_linear=pl, preproc_tree=pt, seed=SEED)[
        FAMILLE_RETENUE
    ]


table_ablation = ablation.ablate(construire_pipeline_retenu, X_train, y_train, blocs, cv)
display(
    table_ablation[["n_features", "roc_auc", "pr_auc", "d_roc_auc", "d_pr_auc"]]
    .round(4)
    .rename(
        columns={
            "roc_auc": "ROC-AUC (OOF)",
            "pr_auc": "PR-AUC (OOF)",
            "d_roc_auc": "Δ ROC-AUC (OOF)",
            "d_pr_auc": "Δ PR-AUC (OOF)",
        }
    )
)
evaluation.plot_ablation_cost(table_ablation)
plt.show()

### Interprétation

- **Retirer ne coûte presque rien** - chaque bloc ôté laisse les scores à quelques **millièmes** du jeu complet ; retirer les leurres les **améliore** même légèrement.
- Le jeu **minimisé** (16 variables, tous les blocs retirés) reste à quelques millièmes du complet : la minimisation exigée par le RGPD est **gratuite** en performance.

## 8.7 Cible secondaire — méthode de comparaison

En complément de l'abandon, on prédit la **note finale sur 20** - une régression, menée avec la même méthode qu'en §8.3.

**La baseline** - une régression **Ridge** (linéaire régularisée), simple et interprétable : le point de repère à battre.

**Les familles comparées** - à la baseline s'ajoutent la **forêt aléatoire** et le **gradient boosting**, deux modèles à base d'**arbres de décision** capables de capter des effets non linéaires et des interactions.

**Les prédictions bornées** - une note est comprise entre **0 et 20** ; on borne donc les prédictions à cet intervalle (`clip`). Sans cela, un modèle linéaire comme Ridge peut extrapoler au-dessus de 20 ou sous 0 - ce qui n'a aucun sens pour une note ; les modèles à arbres, eux, y restent naturellement.

**Les mesures** - en points sur 20, indépendantes de tout seuil :

- **MAE** (erreur absolue moyenne) - l'écart typique entre note prédite et note réelle ;
- **RMSE** - même logique, mais pénalise davantage les grosses erreurs ;
- **R²** - part de variance expliquée : 0 = aussi bien que prédire la moyenne, 1 = parfait.

Comparaison sur le train en validation croisée à 5 folds ; la mesure définitive hors échantillon se fera sur le **test scellé** en §12.

## 8.8 Cible secondaire - Baseline

On entraîne la baseline Ridge définie en §8.7 - le niveau à dépasser.

In [ ]:
y_moyenne = train_df[CIBLE_REGRESSION]
cv_reg = protocol.make_cv(
    n_splits=5, seed=SEED, stratified=False
)  # cible continue : pas de stratification
regresseurs = families.build_regressors(
    preproc_linear=construire_preprocesseur(FEATURES, scale=True),
    preproc_tree=construire_preprocesseur(FEATURES, scale=False),
    seed=SEED,
)

# Une note est bornée à [0, 20] : on borne les prédictions (un modèle linéaire peut
# extrapoler hors de cet intervalle, ce qui n'a pas de sens pour une note).
pred_ridge = evaluation.oof_pred(regresseurs["ridge"], X_train, y_moyenne, cv_reg).clip(0, 20)
m_ridge = evaluation.regression_metrics(y_moyenne, pred_ridge)
display(
    pd.DataFrame(
        {"Baseline (Ridge, OOF)": [f"{m_ridge[k]:.3f}" for k in ("mae", "rmse", "r2")]},
        index=pd.Index(["MAE (pts/20)", "RMSE (pts/20)", "R²"], name="Mesure"),
    )
)
evaluation.plot_regression_fit(y_moyenne, pred_ridge)
plt.show()

### Interprétation

- La baseline explique déjà une bonne part de la variance (R² lisible ci-dessus) - le nuage réel/prédit suit la diagonale, avec une dispersion résiduelle de quelques points. C'est le niveau que les modèles à arbres doivent battre.

## 8.9  Cible secondaire - Comparaison des familles

On oppose la baseline Ridge aux deux modèles à arbres - mêmes folds, mêmes mesures.

In [ ]:
predictions, lignes_reg = {}, []
for nom, pipe in regresseurs.items():
    pred = evaluation.oof_pred(pipe, X_train, y_moyenne, cv_reg).clip(0, 20)  # note bornée [0, 20]
    predictions[nom] = pred
    m = evaluation.regression_metrics(y_moyenne, pred)
    lignes_reg.append(
        {
            "Famille": nom,
            "MAE (pts/20, OOF)": m["mae"],
            "RMSE (pts/20, OOF)": m["rmse"],
            "R² (OOF)": m["r2"],
        }
    )
display(pd.DataFrame(lignes_reg).set_index("Famille").round(3))

meilleure_reg = min(
    predictions, key=lambda n: evaluation.regression_metrics(y_moyenne, predictions[n])["mae"]
)
display(Markdown(f"**> Ajustement du meilleur modèle - {meilleure_reg}**"))
evaluation.plot_regression_fit(y_moyenne, predictions[meilleure_reg])
plt.show()

### Interprétation

- **Le gradient boosting mène** - erreur moyenne d'environ **2,2 points sur 20**, devant Ridge et la forêt. Son nuage réel/prédit se resserre autour de la diagonale.

## 8.10 Cible secondaire - Minimisation

On rejoue donc ici la **même ablation** qu'en §8.6, mais sur la régression - on ne conserve un bloc que si son retrait dégrade nettement la prédiction de la note.

In [ ]:
def construire_regresseur_retenu(features):
    """Le régresseur retenu (gradient boosting), préprocesseur restreint à `features`."""
    pt = construire_preprocesseur(features, scale=False)
    return families.build_regressors(preproc_linear=pt, preproc_tree=pt, seed=SEED)[meilleure_reg]


table_ablation_reg = ablation.ablate_regression(
    construire_regresseur_retenu,
    X_train,
    y_moyenne,
    blocs,
    cv_reg,
    postprocess=lambda pred: pred.clip(0, 20),
)  # note bornée [0, 20]
display(
    table_ablation_reg[["n_features", "mae", "rmse", "r2", "d_mae", "d_rmse"]]
    .round(4)
    .rename(
        columns={
            "mae": "MAE (pts/20, OOF)",
            "rmse": "RMSE (pts/20, OOF)",
            "r2": "R² (OOF)",
            "d_mae": "Δ MAE (pts/20, OOF)",
            "d_rmse": "Δ RMSE (pts/20, OOF)",
        }
    )
)
evaluation.plot_ablation_cost(
    table_ablation_reg,
    cols=("d_mae", "d_rmse"),
    labels=("Δ MAE", "Δ RMSE"),
    xlabel="Écart au jeu complet (positif = coût du retrait)",
)
plt.show()

### Interprétation

- **Retirer coûte ici aussi très peu** - le jeu minimisé (16 variables) n'alourdit la MAE que de quelques centièmes de point sur 20 ; les proxies socio-économiques n'apportent quasi rien à la prédiction de la note.
- **Jeu retenu : le minimisé (16 variables)**, le même que pour l'abandon - la minimisation devient cohérente à l'échelle du système, sans coût matériel.

## 8.11 Contraintes opérationnelles et éco-conception

Le choix de la régression logistique n'est pas que statistique, il est aussi **opérationnel et frugal** :

- **léger** - un modèle linéaire s'entraîne et prédit en une fraction de seconde, sans matériel spécialisé ; le ré-entraînement annuel (D06) coûte quasi rien et l'artefact sérialisé (§10) est minuscule face à un ensemble d'arbres ;
- **interprétable** - l'équipe pédagogique peut lire *pourquoi* un étudiant est signalé (SHAP §12), condition d'un accompagnement actionnable plutôt que subi (D05) ;
- **sobre en données** - le jeu minimisé (16 variables) réduit d'autant la collecte, le stockage et la surface RGPD - une éco-conception qui rejoint la minimisation (§8.6).

## 8.12 Choix du modèle

Au terme de la comparaison, les modèles retenus que §9 optimisera (hyperparamètres, seuil) :

| Cible | Modèle retenu | Jeu de variables | Ce qui l'a départagé |
|---|---|---|---|
| **Abandon** (classification) | **Régression logistique** | minimisé - 16 variables | meilleure en validation croisée, la mieux calibrée, la seule à ne pas sur-apprendre, interprétable |
| **Note finale** (régression) | **Gradient boosting** | minimisé - 16 variables | plus faible erreur (~2,2 pts/20) |

## _Journal de bord des décisions_

Chaque choix est argumenté à sa sous-section ; le journal en garde la trace et, s'il y a lieu, l'**alternative écartée**.

1. **[D24] Partition stratifiée 80/20 + validation croisée en 5 folds, test scellé jusqu'à §12.**

2. **[D26 · D27 · D28] Mise en forme des variables** (cf. tableau §8.2) - imputation médiane, `mention_bac` en ordinal, catégories en one-hot à vocabulaire déclaré ; exclusion des cibles, des protégées (D13) et de `date_inscription`.
*Écarté* - Likert en **ordinal** (l'échelle est déjà numérique et régulière) ; **ancienneté** dérivée de `date_inscription` (aucun signal en §6, et contraire à la minimisation).

3. **[D25] Famille retenue pour l'abandon : régression logistique.** Elle devance les arbres en validation croisée (à ~1 σ près), est la mieux calibrée et la seule à ne pas sur-apprendre.
*Écarté* - **HistGB** (même famille que XGBoost, n'ajoute pas une famille distincte) ; le challenger à base d'arbres sera néanmoins re-testé après réglage des hyperparamètres en §9.

4. **[D11 · D14 · D16 · D20 · D23] Jeu de variables retenu : le minimisé (16 variables).** Le coût du retrait est négligeable (quelques millièmes d'AUC, sous la dispersion inter-folds).
*Écarté* - **garder les blocs** pour ce gain minime, sans commune mesure avec le coût RGPD. Nullité déjà établie - distribution uniforme (§6.3), lien bivarié nul (§6.8), coût de retrait négatif à l'ablation (§8.6).

5. **[D29] Note finale : gradient boosting, sur le jeu minimisé (16 variables), toute la promotion.** Même minimisation que l'abandon - coût mesuré négligeable (§8.10), cohérence RGPD à l'échelle du système.
*Écarté* - **restreindre aux non-décrocheurs** : biaiserait la note vers le haut.

# 9. Entraînement, validation et ajustement (sélection du modèle final) — journal de bord [C5]

**Objectif** : optimiser les deux modèles retenus en §8 : régler leurs hyperparamètres, fixer le seuil de décision de l'abandon, vérifier la fiabilité des probabilités - **sur le train seul**. Le test reste scellé jusqu'à §12 : ni réglage, ni seuil, ni calibration ne le regardent.

Le pouvoir discriminant est acquis en §8 (ROC-AUC 0,945 pour l'abandon) ; §9 ne cherche pas à gratter une AUC déjà excellente, mais à s'assurer qu'elle est **fiable** (la calibration, §9.3), puis à en faire une **décision** (le seuil, §9.4). Le réglage des hyperparamètres est conduit pour **démontrer la démarche et prouver** - non supposer - que le modèle est déjà à son plafond.

In [ ]:
logging.getLogger("codecarbon").disabled = True

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Jeu minimisé (16 variables) retenu en §8.6 : FEATURES moins les blocs retirés (D11 · D14 · D20).
blocs_flat = [c for bloc in blocs.values() for c in bloc]
FEATURES_MIN = [f for f in FEATURES if f not in blocs_flat]
X_train_min = X_train[FEATURES_MIN]
y_note = y_moyenne  # note /20 (cible secondaire, définie en §8)

display(
    Markdown(
        f"Jeu minimisé : **{len(FEATURES_MIN)}** variables · "
        f"train **{len(X_train_min)}** étudiants · "
        f"réglage sur le train seul, test scellé jusqu'à §12"
    )
)

## 9.1 Cible principale - Déséquilibre de classes

La promotion compte environ **28 % de décrocheurs** - un déséquilibre léger. Deux leviers déplacent le compromis rappel / précision, chacun à un moment différent : **pondérer les classes** (`class_weight="balanced"`) agit *pendant* l'entraînement en pénalisant davantage les erreurs sur la minorité ; **régler le seuil** (§9.4) agit *après*, en abaissant le point de décision sur des probabilités déjà apprises. Comme le seuil sera réglé de toute façon en §9, la seule chose que la pondération pourrait apporter en plus, c'est une **meilleure séparation des deux populations** - et non un simple déplacement du curseur, que le seuil fait déjà.

Or la pondération a un **coût connu d'avance** : en sur-pénalisant la minorité, elle pousse le modèle à sur-annoncer le décrochage, si bien que ses probabilités se retrouvent **gonflées par construction** - une décalibration attendue, pas un effet de bord. On ne peut donc pas la juger sur la seule discrimination ; il faut regarder ce qu'elle fait aux probabilités.

On la met à l'épreuve **contre un témoin non pondéré** (D10), sur les probabilités *out-of-fold* (chaque étudiant prédit par le pli qui l'a mis de côté, §8.5), selon deux critères : la **discrimination, indépendante du seuil** (ROC-AUC, PR-AUC) - dit si pondérer sépare mieux ; et la **calibration** (Brier, diagramme de fiabilité) - dit à quel coût, les probabilités restant fiables ou non.

In [ ]:
# Deux LogReg identiques au modèle retenu (§8) ; seule diffère la pondération des classes.
logreg_temoin = construire_pipeline_retenu(FEATURES_MIN)
logreg_pondere = construire_pipeline_retenu(FEATURES_MIN).set_params(model__class_weight="balanced")

proba_temoin = evaluation.oof_proba(logreg_temoin, X_train_min, y_train, cv)
proba_pondere = evaluation.oof_proba(logreg_pondere, X_train_min, y_train, cv)

comparaison_poids = pd.DataFrame(
    {
        "Témoin (sans pondération)": evaluation.classification_metrics(y_train, proba_temoin),
        "Pondéré (balanced)": evaluation.classification_metrics(y_train, proba_pondere),
    }
).T.rename(columns={"roc_auc": "ROC-AUC (OOF)", "pr_auc": "PR-AUC (OOF)", "brier": "Brier (OOF)"})
display(comparaison_poids.round(4))

evaluation.plot_calibration(y_train, {"Témoin": proba_temoin, "Pondéré": proba_pondere})
plt.show()

### Interprétation

- **Discrimination inchangée** - ROC-AUC (0,9451 vs 0,9452) et PR-AUC (0,8712 vs 0,8720) sont identiques à quelques dix-millièmes : pondérer ne fait pas mieux *voir* le modèle.
- **Calibration dégradée** - le Brier passe de **0,0845 à 0,0948** et la courbe pondérée s'écarte de la diagonale : `balanced` sur-annonce le décrochage, gonflant les probabilités de la classe minoritaire.
- Pondérer ne fait donc que **déplacer le point de fonctionnement** - exactement ce que le seuil (§9.4) règle de façon contrôlée, sans abîmer les probabilités dont il a besoin : le déséquilibre se traite au seuil, non par pondération (D10).

**Conclusion** : La pondération via `class_weight="balanced"` n'est pas pertinente

## 9.2 Optimisation des hyperparamètres

On règle chaque finaliste **sur le train seul**, aux **mêmes plis et métrique qu'en §8** (aucune fuite : le préprocesseur est refité dans le pipeline à chaque pli). L'outil est choisi selon la taille de l'espace de recherche - c'est là qu'est l'éco-conception :

- **Abandon - régression logistique.** Espace minuscule (`C`, `penalty`) : une **recherche exhaustive** (`GridSearchCV`) suffit, déterministe et quasi gratuite - Optuna serait de la sur-ingénierie. Le **challenger à base d'arbres est re-testé après réglage** (D25) pour contrôler si la forêt aléatoire optimisée repasse devant la LogReg.

- **Note finale - gradient boosting.** Espace large : une recherche **Optuna** (échantillonnage TPE) bornée par un budget (`n_trials` + `timeout`), avec *pruning* (essai mal parti abandonné tôt) et *early stopping* (arbres stoppés quand la validation plafonne). On compare trois régimes - **base** (non réglé), **frugal**, **lourd** - au gain de MAE **et** au coût carbone.

#### Cible principale - régression logistique

In [ ]:
# class_weight figé par le verdict de §9.1 (None) ; la pénalité reste L2 (défaut, adaptée à un
# modèle dense à 16 variables) ; on règle la seule force de régularisation C.
CLASS_WEIGHT = None
grille_logreg = {"model__C": [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]}
base_logreg = construire_pipeline_retenu(FEATURES_MIN).set_params(model__class_weight=CLASS_WEIGHT)
recherche_logreg = GridSearchCV(base_logreg, grille_logreg, scoring="roc_auc", cv=cv, n_jobs=-1)
recherche_logreg.fit(X_train_min, y_train)

logreg_final = recherche_logreg.best_estimator_
auc_base = evaluation.cv_scores(base_logreg, X_train_min, y_train, cv, scoring="roc_auc").mean()
auc_regle = recherche_logreg.best_score_
display(
    Markdown(
        f"**LogReg** - grille de **{len(grille_logreg['model__C'])}** valeurs de C · "
        f"ROC-AUC base **{auc_base:.4f}** → réglée **{auc_regle:.4f}** · "
        f"HP retenu : `C={recherche_logreg.best_params_['model__C']}` (pénalité L2, défaut)"
    )
)

### Interprétation

- **Plafond confirmé** - le réglage ne gagne que **4 dix-millièmes** de ROC-AUC (0,9453 → 0,9457) : la régression logistique par défaut était déjà à son maximum. Le `C=0,1` retenu régularise un peu plus fort que le défaut (`C=1`), cohérent avec un modèle dense à 16 variables.

#### Challenger - forêt aléatoire re-testée

En §8, la LogReg devançait les arbres à hyperparamètres par défaut, à environ un écart-type près (D25). On règle donc la **forêt aléatoire** sur le jeu minimisé - même budget Optuna - pour vérifier qu'elle ne repasse pas devant après réglage.

In [ ]:
# Forêt aléatoire réglée (Optuna frugal) : la LogReg réglée garde-t-elle l'avantage ? (D25)
def build_rf(params):
    """Construit la forêt aléatoire (jeu minimisé) selon les hyperparamètres proposés."""
    pt = construire_preprocesseur(FEATURES_MIN, scale=False)
    pipe = families.build_classifiers(preproc_linear=pt, preproc_tree=pt, seed=SEED)[
        "random_forest"
    ]
    return pipe.set_params(**{f"model__{k}": v for k, v in params.items()})


def suggest_rf(trial):
    """Espace de recherche de la forêt aléatoire (Optuna)."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500, step=100),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
    }


def regler_rf():
    """Règle la forêt (Optuna frugal) ; ne renvoie que des scalaires (cache §9)."""
    best, etude = tuning.optuna_search(
        build_rf,
        suggest_rf,
        X_train_min,
        y_train,
        cv,
        scoring="roc_auc",
        n_trials=20,
        timeout=120,
        seed=SEED,
    )
    return {
        "best_params": best,
        "essais": len(etude.trials),
        "ratio_elagues": tuning.pruned_ratio(etude),
    }


rf = tuning.cached_search(CACHE_REGLAGE, "foret", regler_rf, use_cache=REGLAGE_DEPUIS_CACHE)
auc_rf = evaluation.cv_scores(
    build_rf(rf["best_params"]), X_train_min, y_train, cv, scoring="roc_auc"
).mean()
verdict_rf = (
    "la LogReg conserve l'avantage" if auc_regle >= auc_rf else "la forêt dépasse la LogReg"
)
display(
    Markdown(
        f"**Forêt réglée** ({rf['essais']} essais, {rf['ratio_elagues']:.0%} élagués) - "
        f"ROC-AUC **{auc_rf:.4f}** · LogReg réglée **{auc_regle:.4f}** → **{verdict_rf}**"
    )
)

##### Interprétation

- **D25 confirmée** - même réglée (20 essais), la forêt aléatoire plafonne à **0,9397**, sous la régression logistique réglée (**0,9457**) : le challenger à base d'arbres ne repasse pas devant après optimisation. La famille retenue en §8 tient.

#### Cible secondaire - gradient boosting

On règle le **gradient boosting** (D29) de la note, en comparant trois régimes de recherche - **base** (défauts §8), **frugal** (20 essais), **lourd** (150 essais) - chacun sous mesure d'empreinte carbone. L'objet n'est pas seulement de trouver les meilleurs hyperparamètres, mais de **mettre en regard le gain de performance et le coût carbone** de la recherche.

In [ ]:
preproc_tree_reg = construire_preprocesseur(FEATURES_MIN, scale=False)
# base = défauts §8 (D29), sans early stopping - c'est le point de référence à battre.
base_gb = families.build_regressors(
    preproc_linear=preproc_tree_reg, preproc_tree=preproc_tree_reg, seed=SEED
)["gradient_boosting"]


def build_gb(params):
    """GB à régler : défauts §8 + early stopping (stoppé quand la validation plafonne)."""
    pipe = families.build_regressors(
        preproc_linear=preproc_tree_reg, preproc_tree=preproc_tree_reg, seed=SEED
    )["gradient_boosting"]
    return pipe.set_params(
        model__n_iter_no_change=10,
        model__validation_fraction=0.1,
        **{f"model__{k}": v for k, v in params.items()},
    )


def suggest_gb(trial):
    """Espace de recherche du gradient boosting (Optuna)."""
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 50),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
    }


def mae_oof(pipe):
    """MAE out-of-fold, prédictions bornées [0, 20] (contrainte de domaine, §8.7)."""
    pred = evaluation.oof_pred(pipe, X_train_min, y_note, cv_reg).clip(0, 20)
    return evaluation.regression_metrics(y_note, pred)["mae"]


def regler_gb(n_trials, timeout):
    """Règle le GB sous budget borné + mesure l'empreinte ; renvoie des scalaires (cache §9)."""
    suivi = OfflineEmissionsTracker(
        country_iso_code="FRA", save_to_file=False, log_level="error", measure_power_secs=3600
    )
    logging.getLogger("codecarbon").disabled = True  # coupe le journal périodique du tracker
    suivi.start()
    best, etude = tuning.optuna_search(
        build_gb,
        suggest_gb,
        X_train_min,
        y_note,
        cv_reg,
        scoring="neg_mean_absolute_error",
        n_trials=n_trials,
        timeout=timeout,
        seed=SEED,
    )
    suivi.stop()
    donnees = suivi.final_emissions_data
    return {
        "params": best,
        "essais": len(etude.trials),
        "duree_s": donnees.duration,
        "co2_g": donnees.emissions * 1000,
    }


# Réglage caché (D30) : un run complet mesure et persiste, un run rapide recharge le cache.
gb_frugal = tuning.cached_search(
    CACHE_REGLAGE, "gb_frugal", lambda: regler_gb(20, 120), use_cache=REGLAGE_DEPUIS_CACHE
)
gb_lourd = tuning.cached_search(
    CACHE_REGLAGE, "gb_lourd", lambda: regler_gb(150, 420), use_cache=REGLAGE_DEPUIS_CACHE
)
best_frugal, best_lourd = gb_frugal["params"], gb_lourd["params"]

mae_base, mae_frugal, mae_lourd = (
    mae_oof(base_gb),
    mae_oof(build_gb(best_frugal)),
    mae_oof(build_gb(best_lourd)),
)
recap_gb = pd.DataFrame(
    [
        {
            "Régime": "base (défauts §8)",
            "essais": 0,
            "MAE (pts/20, OOF)": mae_base,
            "durée (s)": 0.0,
            "CO₂ (g)": 0.0,
        },
        {
            "Régime": "Optuna frugal",
            "essais": gb_frugal["essais"],
            "MAE (pts/20, OOF)": mae_frugal,
            "durée (s)": gb_frugal["duree_s"],
            "CO₂ (g)": gb_frugal["co2_g"],
        },
        {
            "Régime": "Optuna lourd",
            "essais": gb_lourd["essais"],
            "MAE (pts/20, OOF)": mae_lourd,
            "durée (s)": gb_lourd["duree_s"],
            "CO₂ (g)": gb_lourd["co2_g"],
        },
    ]
).set_index("Régime")
display(recap_gb.round({"MAE (pts/20, OOF)": 3, "durée (s)": 1, "CO₂ (g)": 3}))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
recap_gb["MAE (pts/20, OOF)"].plot.bar(ax=a1, color="steelblue")
a1.set_title("MAE (pts/20, OOF) - plus bas = mieux")
a1.set_ylim(recap_gb["MAE (pts/20, OOF)"].min() * 0.98, recap_gb["MAE (pts/20, OOF)"].max() * 1.02)
recap_gb["CO₂ (g)"].plot.bar(ax=a2, color="crimson")
a2.set_title("Empreinte de la recherche (gCO₂eq)")
for a in (a1, a2):
    a.tick_params(axis="x", rotation=15)
    a.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

# Éco-conception : à MAE équivalente (< 0,03 pt/20, ~1 %), on retient le régime le plus sobre.
EPS = 0.03
meilleure = min(mae_base, mae_frugal, mae_lourd)
if mae_base <= meilleure + EPS:
    regime_note, gb_regle, best_note = "base (défauts §8)", base_gb, {}
elif mae_frugal <= meilleure + EPS:
    regime_note, gb_regle, best_note = "Optuna frugal", build_gb(best_frugal), best_frugal
else:
    regime_note, gb_regle, best_note = "Optuna lourd", build_gb(best_lourd), best_lourd
display(
    Markdown(
        f"Régime retenu pour la note : **{regime_note}** - "
        f"meilleure MAE à moins de {EPS} pt/20, le plus sobre"
    )
)

### Interprétation

- **Gain marginal** - la recherche lourde (150 essais) n'améliore la MAE que de **0,025 pt/20** sur la base non réglée (2,233 → 2,208), soit ~1 %, quand la frugale (20 essais) en capte déjà l'essentiel (2,221).
- **Coût carbone disproportionné** - au régime lourd, la recherche émet un ordre de grandeur de CO₂ de plus qu'au frugal (valeurs exactes au tableau ci-dessus), pour un écart de MAE de seulement 0,012 pt/20 entre les deux.
- Sous un seuil de matérialité de **0,03 pt/20**, l'écart est négligeable : on retient la **base non réglée**, la plus sobre. Le réglage aura servi à *prouver* le plafond, non à le déplacer.

## 9.3 Cible principale - Calibration

Avant de transformer la probabilité en décision (le seuil, §9.4), on s'assure qu'elle est **fiable** : une proba de 0,30 doit correspondre à environ 30 % de décrochage observé. La §8.5 montrait déjà la régression logistique comme la mieux calibrée (Brier le plus bas) ; on **vérifie** ici que le modèle réglé le reste, et on teste un **recalibrage** isotonic - qu'on ne conservera que s'il **baisse le Brier**. Calibrer d'abord garantit que le seuil de §9.4 sera posé sur des probabilités dignes de confiance.

In [ ]:
# Probabilités OOF du modèle réglé - base de la calibration ici, et du seuil en §9.4.
proba_logreg = evaluation.oof_proba(logreg_final, X_train_min, y_train, cv)

# Recalibrage isotonic appris en validation croisée interne (sans fuite) ; comparé au modèle réglé.
logreg_calibre = CalibratedClassifierCV(logreg_final, method="isotonic", cv=cv)
proba_calibre = evaluation.oof_proba(logreg_calibre, X_train_min, y_train, cv)

brier_brut = evaluation.classification_metrics(y_train, proba_logreg)["brier"]
brier_cal = evaluation.classification_metrics(y_train, proba_calibre)["brier"]
verdict_cal = "recalibrage sans gain, écarté" if brier_cal >= brier_brut else "recalibrage retenu"
display(
    Markdown(
        f"Brier - modèle réglé **{brier_brut:.4f}** · "
        f"recalibré isotonic **{brier_cal:.4f}** → **{verdict_cal}**"
    )
)

evaluation.plot_calibration(y_train, {"Réglé": proba_logreg, "Recalibré (isotonic)": proba_calibre})
plt.show()

### Interprétation

- **Déjà calibrée** - le recalibrage isotonic laisse le Brier quasi inchangé (**0,0841 → 0,0842**) et les deux courbes se superposent sur la diagonale : la régression logistique produit des probabilités fiables sans retouche (confirmé depuis §8.5). Recalibrage écarté.

## 9.4 Cible principale - Seuil de décision

Le modèle rend une **probabilité** (jugée fiable en §9.3) mais décider « à risque / pas à risque » exige un **seuil**. 

Le seuil par défaut de 0,5 minimise le nombre d'erreurs *en supposant qu'un faux négatif coûte autant qu'un faux positif* - or ici **rater un décrocheur (FN) coûte bien plus qu'une alerte à tort (FP)** (D03). Il faut donc un seuil plus bas.

Faute d'un coût métier chiffré (l'énoncé ne le fournit pas et il est complexe de l'estimer), le seuil ne se calcule pas : il se **déclare** par une politique lisible, appliquée aux probabilités *out-of-fold*. La table ci-dessous donne, pour chaque seuil, le **nombre d'étudiants à accompagner** (`n_alertes`), sa part de la promotion, et l'arbitrage rappel / précision. L'outil final (§10) exposera deux politiques : un **plancher de rappel** (« attraper au moins R % des décrocheurs ») et une **capacité** (« accompagner N étudiants »).

> **Comment lire la table** (noms techniques, sens dans notre cas d'usage) :  
    - `seuil` - probabilité au-delà de laquelle l'étudiant est **signalé à risque**.  
    - `rappel` - part des décrocheurs **effectivement signalés** : le levier qu'on veut haut, car rater un décrocheur (FN) coûte plus qu'alerter à tort (D03).  
    - `precision` - part des **signalés qui décrochent vraiment** : mesure les alertes à tort (précision basse = beaucoup d'étudiants dérangés pour rien et risque du marquage).  
    - `f2` - synthèse rappel / précision **pondérant le rappel 2× la précision**, cohérente avec l'asymétrie FN >> FP (un F1 les pèserait à parts égales, à contre-emploi ici).  
    - `n_alertes` - nombre d'étudiants signalés = **accompagnements à mener** (vrais et faux positifs confondus, c'est la charge réelle).  
    - `tp` - parmi eux, les **décrocheurs réellement attrapés**.  
    - `n_FN` - décrocheurs **manqués** : le coût qu'on cherche à minimiser.  
    - `pct_promo` - **part de la promotion** signalée (`n_alertes` / effectif).  

In [ ]:
# proba_logreg (probabilités OOF du modèle réglé) est calculé en §9.3.
table_seuils = threshold.threshold_table(y_train, proba_logreg)

# Noms techniques conservés (leur sens : légende ci-dessus) ; taux arrondis à 3 décimales.
vue = table_seuils.assign(
    rappel=table_seuils["rappel"].round(3),
    precision=table_seuils["precision"].round(3),
    f2=table_seuils["f2"].round(3),
    pct_promo=table_seuils["pct_promo"].round(3),
)[["rappel", "precision", "f2", "n_alertes", "tp", "n_FN", "pct_promo"]]
display(vue)

### Interprétation

- **0,5 est trop élevé ici** - au seuil par défaut, le rappel n'est que de **0,747** : près de 300 décrocheurs (n_FN = 299) passeraient inaperçus, inacceptable quand un faux négatif coûte bien plus qu'une alerte à tort (D03).
- **Le rappel se paie en volume** - abaisser le seuil remonte le rappel mais élargit les alertes signalées.

**Conclusion** : Faute d'un coût métier chiffrable, le seuil ne se *calcule* pas : on le **déclare** par un objectif de rappel - **détecter 80 % des décrocheurs** (défaut de travail, D04). Cet objectif place le seuil autour de **0,42** et signale **un peu moins de 30 % de la cohorte** (cf. arbitrage ci-dessous).

**Comparaison des trois planchers de rappel.** Détecter **80 %, 90 % ou 95 %** des décrocheurs fixe chacun un seuil. La courbe situe ces points sur l'arbitrage rappel / précision / volume ; les trois matrices de confusion qui suivent en donnent le détail au grain des effectifs (OOF train).

In [ ]:
# Points de fonctionnement : planchers de rappel 80/90/95 % et rappel 100 % (borne haute).
SEUIL_DEFAUT = threshold.pick_threshold(y_train, proba_logreg, recall_target=0.80)
SEUIL_90 = threshold.pick_threshold(y_train, proba_logreg, recall_target=0.90)
SEUIL_95 = threshold.pick_threshold(y_train, proba_logreg, recall_target=0.95)
SEUIL_100 = threshold.pick_threshold(y_train, proba_logreg, recall_target=1.0)
pct_defaut = (proba_logreg >= SEUIL_DEFAUT).mean() * 100
pct_90 = (proba_logreg >= SEUIL_90).mean() * 100
pct_95 = (proba_logreg >= SEUIL_95).mean() * 100
pct_100 = (proba_logreg >= SEUIL_100).mean() * 100
display(
    Markdown(
        f"Rappel ≥ 80 % → seuil **{SEUIL_DEFAUT:.2f}**, **{pct_defaut:.1f} %** signalée · "
        f"≥ 90 % → **{SEUIL_90:.2f}**, **{pct_90:.1f} %** · ≥ 95 % → **{SEUIL_95:.2f}**, "
        f"**{pct_95:.1f} %** · rappel 100 % → **{SEUIL_100:.2f}**, **{pct_100:.1f} %** "
        f"signalée pour détecter l'ensemble des décrocheurs"
    )
)

# Arbitrage rappel / précision / volume : courbe pleine largeur, légende hors cadre pour lisibilité.
fig, ax_courbe = plt.subplots(figsize=(11, 4.5))
ax_courbe.plot(table_seuils.index, table_seuils["rappel"], marker="o", label="rappel")
ax_courbe.plot(table_seuils.index, table_seuils["precision"], marker="s", label="précision")
ax_courbe.plot(
    table_seuils.index, table_seuils["pct_promo"], marker="^", label="part de cohorte signalée"
)
ax_courbe.axvline(
    SEUIL_DEFAUT, color="crimson", ls="--", label=f"rappel ≥ 80 % (seuil {SEUIL_DEFAUT:.2f})"
)
ax_courbe.axvline(SEUIL_90, color="orange", ls="--", label=f"≥ 90 % (seuil {SEUIL_90:.2f})")
ax_courbe.axvline(SEUIL_95, color="goldenrod", ls="--", label=f"≥ 95 % (seuil {SEUIL_95:.2f})")
ax_courbe.axvline(
    SEUIL_100, color="darkgreen", ls=":", label=f"rappel 100 % (seuil {SEUIL_100:.2f})"
)
ax_courbe.set(
    xlabel="seuil", ylabel="score / proportion", title="Arbitrage rappel / précision / volume"
)
ax_courbe.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False)
ax_courbe.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Matrices de confusion aux trois planchers de rappel - OOF train, test scellé jusqu'en §12.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, seuil, cible in zip(
    axes, (SEUIL_DEFAUT, SEUIL_90, SEUIL_95), ("80 %", "90 %", "95 %"), strict=True
):
    evaluation.plot_confusion(
        y_train,
        proba_logreg,
        seuil,
        ax=ax,
        title=f"Seuil {seuil:.2f} - rappel ≥ {cible} (OOF train)",
    )
plt.tight_layout()
plt.show()

### Interprétation

- **Le rappel se paie en volume** - relever le plancher de 80 % à 90 % puis 95 % élargit nettement la part de cohorte signalée : chaque décrocheur de plus attrapé coûte de plus en plus de faux positifs.
- **Le point retenu (seuil 0,42, rappel ≥ 80 %)** signale un peu moins de 30 % de la promotion - compromis entre l'exigence de rappel (D03, FN ≫ FP) et la capacité d'accompagnement, limitée.
- **Le rappel 100 % est hors de portée opérationnelle** - détecter l'ensemble des décrocheurs imposerait de signaler la quasi-totalité de la cohorte : le seuil tombe à 0, la précision s'effondre.


## 9.5 Modèles finaux

On fige les deux modèles éprouvés : **refit sur tout le train** avec les hyperparamètres retenus, `random_state` fixés, versions loguées - prêts pour la sérialisation en §10. La note (gradient boosting) n'a ni seuil ni calibration : elle rend directement une valeur bornée à [0, 20].

In [ ]:
# Refit sur tout le train (le préprocesseur se ré-ajuste sur l'ensemble).
modele_abandon = logreg_final  # déjà refité sur le train entier par GridSearchCV
modele_note = clone(gb_regle).fit(X_train_min, y_note)

hp_note = (
    "défauts §8"
    if not best_note
    else ", ".join(
        f"{k}={round(v, 4) if isinstance(v, float) else v}" for k, v in best_note.items()
    )
)
recap_final = pd.DataFrame(
    {
        "Cible": ["abandon (probabilité)", "moyenne_finale (/20)"],
        "Modèle": ["régression logistique", f"gradient boosting - {regime_note}"],
        "Hyperparamètres retenus": [
            f"C={recherche_logreg.best_params_['model__C']}, "
            f"penalty=L2 (défaut), class_weight={CLASS_WEIGHT}",
            hp_note,
        ],
        "Point de fonctionnement": [
            f"seuil {SEUIL_DEFAUT:.2f} (rappel ≥ 80 %)",
            "sortie bornée [0, 20]",
        ],
    }
).set_index("Cible")
display(recap_final)

versions = pd.DataFrame(
    {"Version": [sklearn.__version__, optuna.__version__, np.__version__, pd.__version__]},
    index=pd.Index(["scikit-learn", "optuna", "numpy", "pandas"], name="Bibliothèque"),
)
display(versions)

**Conclusion** : 

- **Modèles reproductibles** - hyperparamètres et versions des bibliothèques tracés (tableaux ci-dessus), `random_state` fixés partout.

✅ Modèles finaux entraînés - régression logistique (abandon) · gradient boosting (note) - prêts pour §10

## _Journal de bord des décisions_

Chaque choix est argumenté à sa sous-section ; le journal en garde la trace et, s'il y a lieu, l'**alternative écartée**.

1. **[D10] Pas de pondération de classe.** À armes égales, `class_weight="balanced"` laisse la discrimination inchangée (ROC-AUC 0,945, PR-AUC 0,87) mais dégrade la calibration (Brier 0,0845 → 0,0948) : le déséquilibre léger (~28 %) se traite au seuil (§9.4), non par pondération.
*Écarté* - **`balanced`** (déplace le point de fonctionnement au prix des probabilités) et **SMOTE** (sur-échantillonnage superflu à ce niveau de déséquilibre, et déformant).

2. **[D30] Réglage des hyperparamètres dosé selon l'espace, sous budget borné.** Régression logistique par grille exhaustive (`C=0,1`, L2) ; gradient boosting par Optuna borné (20/150 essais, *pruning* + *early stopping*), empreinte carbone mesurée. Pour la note, la **base non réglée est conservée** : le réglage lourd ne gagne que 0,025 pt/20 (~1 %) pour ~7× le carbone du frugal, sous le seuil de matérialité de 0,03 pt/20.
*Écarté* - **Optuna pour la LogReg** (sur-ingénierie sur 7 valeurs de `C`) ; **grille pour le GB** (explosion combinatoire) ; **régime de réglage lourd** (gain non matériel, coût carbone disproportionné).

3. **[D25] Famille de l'abandon confirmée après réglage : régression logistique.** Réglée, la forêt aléatoire plafonne à 0,9397, sous la régression logistique réglée (0,9457) : le challenger à base d'arbres ne repasse pas devant.
*Écarté* - **forêt aléatoire réglée** (reste en deçà, et sur-ajuste - §8.5).

4. **[D31] Pas de recalibrage des probabilités.** Le recalibrage isotonic laisse le Brier quasi inchangé (0,0841 → 0,0842) : la régression logistique est déjà fiable (§8.5).
*Écarté* - **recalibrage isotonic / Platt** (sans gain mesuré).

5. **[D04] Seuil de l'abandon : plancher de rappel de 80 % (défaut de travail).** Faute de coût métier chiffré, le seuil se déclare par une politique : à rappel ≥ 80 % il tombe à 0,42 et signale 29 % de la promotion. Le modèle expose la probabilité ; le seuil reste externe (D15) et configurable - plancher de rappel ou capacité d'accompagnement. Le choix de la politique (rappel visé vs nombre de places) fixe la matrice de confusion évaluée en §12.

# 10. Implémentation et mise en exploitation (déploiement, exemple d’usage) — journal de bord [C6]

**Objectif** : figer l'artefact déployable issu du §9 - les deux pipelines et la fiche qui les décrit -, publier son contrat d'entrée/sortie, et prouver son usage par un `predict()` exécuté.

> 🔧 **Comment je m'y prends** - je sérialise le modèle final tel quel (préprocesseur inclus) avec une *fiche* de service (typage, thèmes, seuil, référence de dérive) ; le **contrat d'entrée**, lui, est un schéma typé publié par l'API (§10.3). Le service décrit en §11 relit ces artefacts, sans jamais réapprendre.


## 10.1 La fiche du modèle

La fiche accompagne les pipelines : elle porte le typage (pour la dérive), les **thèmes** d'explicabilité, le **seuil** et les seuils de dérive (surchargeables, D38), et la **distribution de référence**. Le contrat d'entrée, lui, est le schéma typé de l'API (§10.3). Seuls les thèmes sont déclarés ici ; le reste est dérivé de ce que le §9 a produit.


In [ ]:
# --- Faits du modèle entraîné (dérivés de ce que le §9 a produit) ---
model_features = list(modele_abandon.feature_names_in_)  # ordre d'entraînement
derivees = [c for c in faits_gold.derived if c in model_features]  # calculées par le service (§7)
numeriques_fiche = [c for c in NUMERIQUES if c in model_features]
categorielles_fiche = [c for c in ORDINALES + NOMINALES if c in model_features]

# Thèmes d'explicabilité : DÉCLARÉS ici (variable -> thème), défendus à l'oral.
THEMES = {
    "taux_presence_pct": "assiduité",
    "nb_devoirs_total": "assiduité",
    "nb_devoirs_rendus": "assiduité",
    "retards_rendus": "assiduité",
    "taux_rendu": "assiduité",
    "ratio_retards": "assiduité",
    "heures_lms_total": "engagement",
    "messages_forum": "engagement",
    "motivation": "ressenti",
    "satisfaction": "ressenti",
    "sentiment_appartenance": "ressenti",
    "age": "parcours",
    "nb_ue_total": "parcours",
    "mention_bac": "parcours",
    "filiere": "parcours",
    "bac_type": "parcours",
}
themes = {c: THEMES[c] for c in model_features if c in THEMES}

# La fiche ne porte que ce qu'un schéma d'entrée ne sait pas exprimer : typage pour la dérive,
# thèmes, seuil et seuils de dérive (surchargeables), distribution de référence.
fiche = ServiceContract(
    facts=ModelFacts(
        version="1.0.0",
        numeric=tuple(numeriques_fiche),
        categorical=tuple(categorielles_fiche),
        themes=themes,
        drift_reference=train_df[model_features].copy(),
    ),
    defaults=OperationalDefaults(
        threshold=float(SEUIL_DEFAUT),
        drift_surveillance=0.10,
        drift_alerte=0.25,
        drift_effectif_min=200,
    ),
)
fiche.validate()

ref_fiche = fiche.facts.drift_reference  # instantané du train scellé dans la fiche

recap = pd.DataFrame(
    {
        "Valeur": [
            fiche.facts.version,
            f"{len(numeriques_fiche)} numériques · {len(categorielles_fiche)} catégorielles",
            f"{len(themes)} variables → {len(set(themes.values()))} thèmes",
            f"seuil {fiche.defaults.threshold:.2f} · dérive "
            f"{fiche.defaults.drift_surveillance:.2f}/{fiche.defaults.drift_alerte:.2f}",
            f"{len(ref_fiche)} lignes × {ref_fiche.shape[1]} variables",  # type: ignore
        ]
    },
    index=pd.Index(
        ["Version", "Typage (dérive)", "Thèmes", "Exploitation", "Réf. dérive"],
        name="Fiche du modèle",
    ),
)
display(recap)

- **La fiche est le descripteur de service** : typage, thèmes, seuil et seuils de dérive, distribution de référence. Le contrat d'entrée (colonnes, bornes, modalités) est porté par `schemas.PredictEtudiantForm` (§10.3).
- **Deux natures** (D38) : faits immuables (typage, thèmes, référence) et paramètres surchargeables en exploitation (seuil, seuils de dérive), sans réentraîner.
- **Distribution de référence scellée** : le train du split §8 (sans identifiant ni attribut protégé), pour mesurer la dérive d'une campagne (§13) - seule donnée persistée dans l'artefact, remplacée à chaque ré-entraînement (D43).


## 10.2 Sérialisation de l'artefact déployable

`save_bundle` écrit trois fichiers et **refuse d'écrire** une fiche incohérente (seuil hors [0, 1], seuils de dérive désordonnés). L'empreinte de chaque fichier trace la version de l'artefact ; celle du jeu de référence, la version des **données** scellées.


In [ ]:
store.save_bundle(
    settings.artifacts_dir, contract=fiche, classifier=modele_abandon, regressor=modele_note
)


def empreinte(chemin):
    """Empreinte courte (sha256) d'un fichier, pour tracer la version de l'artefact."""
    return hashlib.sha256(chemin.read_bytes()).hexdigest()[:12]


artefacts = pd.DataFrame(
    [
        (f.name, f"{f.stat().st_size / 1024:.1f} Ko", empreinte(f))
        for f in sorted(settings.artifacts_dir.glob("*.joblib"))
    ],
    columns=["Fichier", "Taille", "Empreinte"],
).set_index("Fichier")
display(artefacts)

# Versionnage des DONNÉES : la distribution de référence scellée dans la fiche (train du split).
ref = fiche.facts.drift_reference
ref_empreinte = hashlib.sha256(pd.util.hash_pandas_object(ref, index=True).values).hexdigest()[:12]  # type: ignore
display(
    Markdown(
        f"Jeu de référence scellé dans la fiche : **{len(ref)}** lignes × **{ref.shape[1]}** "  # type: ignore
        f"variables (train du split §8, `random_state={SEED}`) - empreinte **{ref_empreinte}**"
    )
)

## 10.3 Le contrat d'entrée / sortie

Le contrat d'entrée est un **schéma Pydantic typé** (`schemas.PredictEtudiantForm`) : types, bornes et modalités y font foi, `extra="forbid"` refuse tout champ hors périmètre (fuites, variables protégées) en le nommant. C'est la source de vérité du contrat ; un contrôle de **non-écart** garantit qu'il couvre exactement les features du modèle.


Les schémas de l'API, regroupés par sens de circulation.

**Entrée**

| Classe | Rôle |
|---|---|
| `PredictEtudiantForm` | Un étudiant à mi-S1 - contrat d'entrée typé (14 champs, bornes, modalités). |
| `PredictCohorteForm` | Un lot de dossiers, validés **ligne à ligne** (refus partiel). |

**Sortie**

| Classe | Rôle |
|---|---|
| `PredictEtudiantReponse` | Résultat d'un dossier : proba, note, indicateur, explicabilité. |
| `ContributionTheme` | Contribution signée d'un thème à la log-cote. |
| `ContributionVariable` | Contribution signée d'une variable (détail sous les thèmes). |
| `DossierRefuse` | Une ligne refusée : index, référence, motifs. |
| `SyntheseCohorte` | Compteurs d'une campagne : reçus, scorés, refusés, part signalée. |
| `PredictCohorteReponse` | Réponse d'une campagne : résultats, refus, synthèse, dérive. |

```mermaid
classDiagram
    class PredictEtudiantForm {
        +str~opt~ reference_dossier
        +int age
        +Filiere filiere
        +BacType bac_type
        +float taux_presence_pct
        +float heures_lms_total
        +int nb_ue_total
        +int nb_devoirs_total
        +int nb_devoirs_rendus
        +int retards_rendus
        +int messages_forum
        +MentionBac~opt~ mention_bac
        +float~opt~ motivation
        +float~opt~ satisfaction
        +float~opt~ sentiment_appartenance
    }
    class PredictCohorteForm {
        +list~dict~ dossiers
    }
    class PredictEtudiantReponse {
        +str~opt~ reference_dossier
        +float proba_abandon
        +float moyenne_finale
        +bool~opt~ signaled
        +float~opt~ seuil_applique
        +str~opt~ provenance_seuil
        +str version_modele
    }
    class ContributionTheme {
        +str theme
        +float contribution
    }
    class ContributionVariable {
        +str variable
        +float contribution
    }
    class DossierRefuse {
        +int index
        +str~opt~ reference_dossier
        +list erreurs
    }
    class SyntheseCohorte {
        +int dossiers_recus
        +int dossiers_scores
        +int dossiers_refuses
        +float~opt~ part_signalee
    }
    class PredictCohorteReponse {
        +float~opt~ seuil_applique
        +str~opt~ provenance_seuil
        +dict~opt~ derive
    }
    PredictCohorteForm ..> PredictEtudiantForm : valide ligne à ligne
    PredictEtudiantReponse --> "*" ContributionTheme
    PredictEtudiantReponse --> "*" ContributionVariable
    PredictCohorteReponse --> "*" PredictEtudiantReponse
    PredictCohorteReponse --> "*" DossierRefuse
    PredictCohorteReponse --> SyntheseCohorte
```


In [ ]:
entrepot = EntrepotModele()
entrepot.load(settings.artifacts_dir)
assert entrepot.ready, entrepot.error
bundle = entrepot.bundle

# Contrôle de NON-ÉCART : le schéma d'entrée couvre exactement les features du modèle.
champs_schema = set(schemas.INPUT_FIELDS)
attendu = set(model_features) - set(derivees)
assert champs_schema == attendu, f"écart schéma/modèle : {champs_schema ^ attendu}"

display(
    Markdown(
        f"**> Entrée** - **{len(schemas.INPUT_FIELDS)}** colonnes typées (`PredictEtudiantForm`) ; "
        f"les dérivées ({', '.join(f'`{d}`' for d in derivees)}) sont calculées par le service ; "
        f"tout autre champ est refusé (`extra=forbid`, HTTP 422). "
        f"Non-écart schéma ↔ modèle : **vérifié** sur {len(champs_schema)} colonnes."
    )
)
display(
    Markdown(
        "**> Sortie** - probabilité d'`abandon` · `moyenne_finale` bornée [0 ; 20] · contributions "
        "par thème et par variable. La garantie de l'art. 22 est portée par la documentation "
        "(model card, §12), non par un champ répété dans chaque réponse (D15)."
    )
)

In [ ]:
# Contrat I/O publié en JSON : le schéma typé Pydantic projeté en JSON Schema (types, bornes,
# modalités), l'interface que l'API implémente. Un dossier valide sert d'exemple,
# réutilisé en §10.4.
dossier_exemple = {
    "reference_dossier": "DEMO-2026-001",
    "age": 19,
    "filiere": "informatique",
    "bac_type": "general",
    "taux_presence_pct": 72.0,
    "heures_lms_total": 14.5,
    "nb_ue_total": 6,
    "nb_devoirs_total": 12,
    "nb_devoirs_rendus": 8,
    "retards_rendus": 3,
    "messages_forum": 2,
    "mention_bac": "assez bien",
    "motivation": 3.0,
    "satisfaction": 3.0,
    "sentiment_appartenance": 2.0,
}
io_contract = {
    "entree": schemas.PredictEtudiantForm.model_json_schema(),
    "sortie": schemas.PredictEtudiantReponse.model_json_schema(),
    "exemple_entree": dossier_exemple,
}
chemin_contrat = settings.artifacts_dir / "io_contract.schema.json"
chemin_contrat.parent.mkdir(parents=True, exist_ok=True)
chemin_contrat.write_text(json.dumps(io_contract, ensure_ascii=False, indent=2), encoding="utf-8")

bornes = {"minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum"}
props = io_contract["entree"]["properties"]
n_bornes = sum(1 for p in props.values() if bornes & set(p))
display(
    Markdown(
        f"Contrat I/O publié en JSON - **{len(props)}** champs d'entrée "
        f"(dont **{n_bornes}** bornés) "
        f"et sortie typée, écrits dans `{chemin_contrat.name}` ; lien en §15.2."
    )
)

## 10.4 Exemple d'usage - `predict()` exécuté

Un dossier valide (entrée **typée**) traverse le chemin réel du service - schéma, calcul des dérivées, score -, sans lancer aucun conteneur (D32). Le test restant scellé jusqu'au §12, la démonstration porte sur un dossier synthétique plausible.


In [ ]:
# Dossier valide défini en §10.3 (`dossier_exemple`) - même source que l'exemple du contrat I/O.
dossier = dossier_exemple
form = schemas.PredictEtudiantForm(**dossier)  # valide le contrat d'entrée (lèverait sinon)
entree = normalization.add_derived(pd.DataFrame([form.model_dump(exclude={"reference_dossier"})]))
score = scoring.score(
    bundle,
    entree,
    [form.reference_dossier],
    threshold=bundle.contract.defaults.threshold,
    expose_indicator=True,
)[0]

display(
    Markdown(
        f"Dossier **{score.reference}** - probabilité d'abandon **{score.probability:.1%}** · "
        f"note estimée **{score.moyenne_finale:.1f}/20** · indicateur : "
        f"**{'signalé' if score.signaled else 'non signalé'}** au seuil "
        f"{bundle.contract.defaults.threshold:.2f}"
    )
)
contrib = (
    pd.DataFrame(
        sorted(score.contributions.by_theme.items(), key=lambda kv: -abs(kv[1])),
        columns=["Thème", "Contribution (log-cote)"],
    )
    .set_index("Thème")
    .round(3)
)
display(contrib)

# Reproductibilité : la prédiction rechargée est identique au modèle §9 en mémoire.
proba_memoire = float(modele_abandon.predict_proba(entree[model_features])[0, 1])
identique = bool(np.isclose(proba_memoire, score.probability))
display(
    Markdown(
        f"Reproductibilité - proba (modèle §9 en mémoire) **{proba_memoire:.6f}** "
        f"= proba (artefact rechargé) **{score.probability:.6f}** : "
        f"**{'identique' if identique else 'DIVERGENCE'}**"
    )
)
assert identique, "l'artefact rechargé diverge du modèle en mémoire"

## _Journal de bord des décisions_

Architecture et observabilité sont portées en §11 et §13 ; ici, ce qui touche l'artefact lui-même.

1. **[D33] Deux artefacts sérialisés séparés** (`classifier.joblib`, `regressor.joblib`, et la fiche). ⇒ les deux cibles ont des cycles de vie indépendants : réentraîner la note n'oblige pas à resérialiser l'abandon.
*Écarté* - un artefact unique multi-cibles, qui couplerait les versions et compliquerait le remplacement de l'un sans l'autre.

2. **[D38] Trois responsabilités distinctes, aucune duplication.** ⇒ le **contrat d'entrée** (colonnes, bornes, modalités) vit dans le schéma Pydantic (code versionné) ; la **fiche** porte les faits de service (typage, thèmes, référence de dérive) ; **seuil et seuils de dérive** portent un défaut dans la fiche, surchargeable par configuration sans réentraîner ni resérialiser.
*Écarté* - une fiche qui redéclarerait le contrat d'entrée en double du schéma : deux sources à maintenir, risque d'écart (un test de non-écart garde le lien schéma ↔ modèle).

3. **[D32] Périmètre : code réel, notebook descriptif.** ⇒ l'artefact et le `predict()` sont exécutés ici ; l'API, Docker et l'observabilité tournent réellement mais se *décrivent* en §11/§13 - le notebook ne lance aucun conteneur.


✅ Artefact déployable sérialisé - deux pipelines + fiche, contrat d'entrée typé (schéma), `predict()` reproductible vérifié


# 11. Architecture cible et contraintes [C7]

**Objectif** : proposer l'architecture cible du dispositif - de la collecte d'une cohorte à la restitution aux équipes - puis cadrer les **contraintes** (technique, RGPD, éco-conception, coût, organisation) et les **acteurs** à mobiliser.

> 🔧 **Comment je m'y prends** - le service décrit ici tourne réellement (`docker compose`, quatre conteneurs), mais le notebook ne lance aucun conteneur (D32) : il montre l'architecture, le contrat de service et l'observabilité, et renvoie aux captures de §15. L'artefact figé en §10 est relu tel quel, jamais réappris.

## 11.1 La chaîne de bout en bout

Le dispositif enchaîne quatre temps, de la donnée brute à la décision d'accompagnement : **ingestion, features, inférence, restitution**. Une campagne annuelle sur toute la promotion (D06) en est le régime, non un flux temps réel.

```mermaid
flowchart TD
    SI["SI scolarité · LMS<br/>relevés à mi-S1"] --> EX["Export d'une cohorte<br/>toute la promotion · 1 campagne par an"]
    EX --> V
    subgraph SVC["Service d'inférence · ne conserve rien"]
      V["1 · Ingestion<br/>validation ligne à ligne<br/>bornes · modalités · anti-fuite"] --> F["2 · Features<br/>dérivées calculées côté serveur"]
      F --> I["3 · Inférence<br/>proba abandon · note /20<br/>explicabilité thème et variable<br/>bilan de dérive (cohorte)"]
    end
    I --> RE["4 · Restitution<br/>une réponse : proba · note · facteurs<br/>+ bilan de dérive"]
    RE --> EQ["Équipe pédagogique<br/>arbitre l'accompagnement"]
    RE -. écart signalé .-> CL["Client<br/>sollicite contrôle ou ré-entraînement"]
```

- **Ingestion** - une campagne, pas du temps réel : toute la promotion scorée en une passe (D06), chaque ligne validée séparément - une ligne aberrante est refusée seule, avec son motif, les autres sont scorées.
- **Features** - les dérivées (`taux_rendu`, `ratio_retards`) sont calculées par le service, du même quotient qu'en §7, jamais acceptées en entrée : préparation et service ne peuvent pas diverger.
- **Inférence** - une seule passe rend les deux cibles : la probabilité d'`abandon` (principale) et la note `moyenne_finale` sur 20 (secondaire, bornée).
- **Restitution** - la sortie porte la probabilité et l'explicabilité ; le service ne tranche pas, l'équipe pédagogique décide (D15), l'avertissement de l'art. 22 relevant de la documentation du modèle (§12).
- **Dérive** - mesurée pendant l'inférence de la cohorte et renvoyée dans la même réponse : une réponse porte les prédictions et le bilan de dérive. Le client, informé de cet écart à la référence, sollicite alors un contrôle ou un ré-entraînement (§11.4).

## 11.2 Le contrat de service

Le service expose sept routes : deux de prédiction, deux de lecture, deux sondes et un point de métriques. Le **contrat d'entrée/sortie détaillé** (schémas, bornes, modalités) est porté par §10.3 ; ici, l'inventaire des points d'accès et leur rôle.

| Route | Accès | Rôle |
|---|---|---|
| `POST /v1/predict-cohorte` | `X-API-Key` | **route principale** : une campagne, un résultat par ligne (proba, note, facteurs), synthèse et refus ligne à ligne, bloc dérive en option |
| `POST /v1/predict-etudiant` | `X-API-Key` | un dossier ponctuel, explicabilité complète |
| `GET /v1/modele` | `X-API-Key` | fiche du modèle : version, variables, seuils de la fiche |
| `GET /v1/seuil` | `X-API-Key` | politique de décision en vigueur, lecture seule |
| `GET /health` | ouvert | le processus répond |
| `GET /ready` | ouvert | le modèle est utilisable (503 motivé sinon) |
| `GET /metrics` | ouvert | métriques d'exploitation (Prometheus) |

### Ce que le service rend, côté métier

- **Une route principale, une secondaire.** Le cas d'usage est une campagne : `/v1/predict-cohorte` score toute la promotion en une fois, dans l'ordre du fichier reçu - jamais réordonné par risque, la priorisation reste à l'équipe. `/v1/predict-etudiant` sert le cas ponctuel d'un dossier isolé.
- **Une explicabilité locale sur deux niveaux.** Le modèle étant linéaire, chaque prédiction se décompose en contributions exactes et signées (D34), restituées par thème (engagement · assiduité · ressenti) pour la vue d'ensemble et par variable pour le détail : l'équipe voit ce qui pèse sur le risque de *cet* étudiant (D05).
- **Deux régimes, au choix de l'établissement.** Par défaut, le service rend la probabilité seule (aide à la décision, sans verdict) ; l'établissement peut activer l'indicateur « signalé » pour une campagne de tri à capacité donnée (D39).
- **Un seuil réglable à l'appel.** Faute de coût métier chiffrable (D03), le seuil ne se calcule pas, il se déclare (D04) : en régime indicateur, l'exploitation l'ajuste par `?seuil=` (ou `?capacite=N` pour un nombre d'accompagnements donné), sans toucher au modèle - il vit hors des poids (D38).
- **Une dérive signalée d'elle-même.** La route principale mesure, en option, l'écart du peuplement à la référence et le renvoie dans la réponse : le client est informé sans surveillance continue, et sait quand nous solliciter (§11.4).
- **Un service qui ne conserve rien.** Aucune écriture, aucune base ; l'accès protégé par clé (`X-API-Key`), les sondes ouvertes pour l'orchestrateur.

### Les paramètres d'exploitation

Le service se règle par l'environnement, sans réentraîner ni resérialiser (D38) : la fiche porte les défauts, la configuration les surcharge.

| Variable | Rôle | Défaut |
|---|---|---|
| `DECROCHAGE_API_KEYS` | clés d'accès autorisées | aucune - routes protégées en 503 |
| `DECROCHAGE_EXPOSER_INDICATEUR` | régime servi : probabilité seule ou avec indicateur (D39) | `false` - probabilité seule |
| `DECROCHAGE_SEUIL_DEFAUT` | seuil de décision (régime indicateur) | fiche (~0,42, §10.1) |
| `DECROCHAGE_DRIFT_SURVEILLANCE` · `_ALERTE` · `_EFFECTIF_MIN` | seuils de dérive | fiche (0,10 · 0,25 · 200) |
| `DECROCHAGE_CORS_ORIGINS` | origines navigateur autorisées | origine du client de démonstration |
| `DECROCHAGE_MONITORING_ACTIF` | exposer `/metrics` | `true` |

## 11.3 Le déploiement conteneurisé

Le dispositif se livre en **quatre conteneurs** orchestrés par `docker-compose` : le service d'inférence, le collecteur Prometheus, le tableau de bord Grafana, et le client de démonstration. Une capture de la pile en marche est jointe en §15.

- **Image du service, socle d'exécution seul.** Construction multi-étapes figée par `uv.lock` ; les bibliothèques d'analyse (SHAP, XGBoost, Optuna, matplotlib) servent à produire le dossier, pas à répondre - l'explicabilité en production est analytique, calculée du modèle linéaire (§11.2, D34), sans SHAP côté service. Leur absence est vérifiée dans l'image (éco-conception, §11.5).
- **Conteneur durci.** Système de fichiers en lecture seule, pas d'élévation de privilèges, compte non-root, `/tmp` en mémoire : le principe « ne conserve rien » tenu jusqu'à l'infrastructure.
- **Démarrage ordonné.** L'artefact est monté en lecture seule ; s'il manque, `/ready` répond 503 sans empêcher le démarrage, et l'orchestrateur attend qu'il soit prêt avant d'exposer le service.

## 11.4 L'observabilité

Deux surfaces se surveillent : la **santé du service** - répond-il, assez vite, sans erreur ? - et la **dérive du peuplement** - la cohorte ressemble-t-elle à celle de l'entraînement ? §13 s'appuie sur cette base pour bâtir le cycle d'amélioration continue.

### Supervision technique et métier

- **Métriques d'exploitation** exposées à Prometheus, lues dans Grafana (captures §15) : trafic et taux de 5xx, latence par route, et trois indicateurs métier - modèle chargé (distingue « le processus répond » de « il peut prédire »), seuil configuré, et refus de périmètre par colonne (une source qui transmet une variable hors contrat se lit colonne par colonne).
- **Règles d'alerte chiffrées**, aux seuils usuels d'un service en ligne : service injoignable (2 min), modèle non chargé (2 min), taux de 5xx > 1 % sur 10 min, latence p95 > 1 s pour un dossier ou > 30 s pour une campagne, seuil de décision inattendu. Chacune porte une durée, un destinataire et une conduite à tenir.
- **Notifications non routées.** Les seuils existent, leur acheminement vers une personne (Alertmanager) n'est pas configuré : c'est un choix d'organisation de l'établissement (D37).

### La dérive du peuplement

- **Mesurée pendant la campagne**, non en continu : le régime réel étant une extraction annuelle, la dérive se lit sur le lot qu'on vient de scorer, comparé à la distribution de référence figée avec l'artefact (D35).
- **Deux mesures.** Le PSI (*Population Stability Index*) chiffre l'écart de distribution d'une variable entre la référence et la cohorte ; le test de Kolmogorov-Smirnov confirme qu'un écart numérique ne tient pas du seul hasard d'échantillonnage.
- **Trois seuils par défaut**, standards du domaine et modifiables par l'environnement : surveiller à PSI ≥ 0,10, alerte à PSI ≥ 0,25, dérive non mesurée sous 200 dossiers (motif renvoyé, prédictions servies). Défauts portés par la fiche, surchargeables sans réentraîner (D38, §11.2).
- **Un signal au client, pas un verdict automatique.** La réponse porte le bilan de dérive pour que l'établissement nous sollicite en cas d'écart matériel : contrôle des données ou ré-entraînement (§13).

## 11.5 Les contraintes

Quatre familles de contraintes cadrent la mise en production, et la réponse que le dispositif y apporte.

| Domaine | Contrainte | Réponse du dispositif |
|---|---|---|
| **Technique** | disponibilité, reproductibilité, pas de matériel spécialisé | batch annuel, pas de temps réel ; versions figées (`uv.lock`, `random_state`) ; modèle linéaire léger, sans GPU |
| **Versioning** | tracer le modèle en production, reproduire une décision | version et empreintes portées par la fiche (§10.1) ; à l'échelle, registre **MLflow** et artefact sous **DVC** - développés en §13 |
| **RGPD** | minimisation, décision humaine, données sensibles | features minimisées (D16) ; **art. 22** - proba par défaut, aucune décision automatisée, l'équipe tranche (D15) ; variables protégées hors modèle (D13) ; service qui ne conserve rien ; base légale à confirmer (D12) |
| **Éco-conception** | empreinte de l'inférence et du service | image socle-seul (libs d'analyse écartées) ; explicabilité **analytique**, pas de SHAP en production (D34) ; instrumentation désactivable |
| **Coût** | budget d'exploitation | briques open-source ; hébergement modeste (une campagne par an) ; aucune donnée stockée à sécuriser |
| **Organisation** | garde-fous effectifs, information des personnes | accompagnement décidé par l'équipe (§4.6) ; garde-fous d'usage à rendre effectifs par l'établissement ; information des personnes (art. 13-14) |

## 11.6 Les acteurs à mobiliser

Les parties prenantes sont posées en §2 ; leur **mise en œuvre** engage, du cadrage à l'exploitation :

- **SI scolarité** extrait la cohorte à mi-S1 et la soumet au service - l'intégration technique annoncée en §2.
- **DSI / exploitant** construit l'image, déploie la pile, supervise (Prometheus, Grafana) et applique la configuration (clés, régime, seuils).
- **DPO** consulté sur l'AIPD et la base légale (D12) avant toute mise en production.
- **Responsable de traitement (l'établissement)** choisit le régime servi (D39) et garantit l'effectivité des garde-fous (§4.6).
- **Équipe pédagogique** lit proba et facteurs, et **décide** l'accompagnement (art. 22).

### Journal de bord [C7]

1. **[D39] Deux régimes servis, au choix de l'établissement.**
⇒ Probabilité seule par défaut (aide à la décision), indicateur « signalé » activable par le responsable de traitement pour un tri à capacité ; le seuil ne gouverne que l'indicateur, jamais les poids. *Pourquoi* - minimisation et art. 22 : par défaut, le service ne tranche rien. *Écarté* - imposer un régime unique : priverait l'établissement, responsable de traitement, de l'arbitrage de proportionnalité qui lui revient.

2. **[D40] Mode capacité `?capacite=N`.**
⇒ La route principale signale les N dossiers les plus à risque de la cohorte (coupe relative), en complément du seuil par valeur et du plancher de rappel (D04). *Pourquoi* - le nombre d'accompagnements est une contrainte réelle du service de réussite, et la coupe se fait sur la cohorte, pas dans les poids.

3. **[D35] Dérive mesurée par cohorte.**
⇒ PSI et KS sur le lot scoré contre la distribution de référence figée, renvoyés dans la réponse, hors Prometheus. *Pourquoi* - le régime est annuel ; une surveillance continue donnerait l'illusion d'un suivi temps réel sans objet. *Écarté* - un flux de dérive en continu : coûteux et trompeur pour une campagne par an.

4. **[D37] Alertmanager écarté.**
⇒ Les règles portent seuils chiffrés, durée, destinataire et conduite ; leur acheminement vers une personne n'est pas livré. *Pourquoi* - le référentiel demande des seuils d'alerte, pas un routage de notifications, qui relève de l'organisation de l'établissement.

5. **[D36] Clé d'API du client en `localStorage`.**
⇒ Le client de démonstration garde la clé dans le navigateur, sans relais serveur. *Pourquoi* - posture démo assumée en usage localhost, hors périmètre livré. *Écarté* - un relais serveur portant le secret : plus sûr, mais hors du cadre d'une maquette de présentation.

✅ Architecture cible posée - service conteneurisé, supervisé, contraint

# 12. Mesure de performance et impacts (métriques techniques + métier) [C8]

**Objectif** : mesurer ce que §8 et §9 ont décidé sur le train seul - le test (20 %) est resté scellé jusqu'ici.

> 🔧 **Comment je m'y prends** - je charge l'artefact déployable (§10) et je prédis **une seule fois** sur le test. Rien ne se règle ici : les hyperparamètres (§9), le seuil 0,42 (D04), la calibration écartée (D31), le jeu de features (§8) et le préprocesseur (refité *dans* le pipeline sur le train seul) sont **figés**. Aucun `fit`, aucun re-seuil, aucune recalibration ne regarde le test - une lecture, définitive.

In [ ]:
# Ouverture du test scellé - une seule passe, aucun ajustement (cf. encart).
X_test = test_df[FEATURES_MIN]
y_test = test_df[CIBLE]
proba_test = bundle.classifier.predict_proba(X_test)[:, 1]  # artefact déployé (§10)
proba_oof = proba_logreg  # OOF train du modèle final (§9), pour lire la généralisation

# Contrôle d'identité §9 (mémoire) <-> §10 (sérialisé) : mêmes probabilités,
# sinon incohérence de fiche.
assert np.allclose(proba_test, modele_abandon.predict_proba(X_test)[:, 1]), (
    "artefact §10 != modèle §9"
)

n_test = len(y_test)
n_decrocheurs = int(y_test.sum())
display(
    Markdown(
        f"Test scellé ouvert - **{n_test}** étudiants · **{n_decrocheurs}** décrocheurs "
        f"(**{n_decrocheurs / n_test:.1%}**) · une seule passe, artefact §10 identique au modèle §9"
    )
)

## 12.1 Pouvoir discriminant sur le test

Trois mesures indépendantes du seuil, lues de l'OOF train au test scellé : le rang global (ROC-AUC), le rang sur la classe minoritaire (PR-AUC) et la fiabilité des probabilités (Brier).

In [ ]:
# Métriques indépendantes du seuil, de l'OOF train au test scellé - lecture de la généralisation.
m_oof = evaluation.classification_metrics(y_train, proba_oof)
m_test = evaluation.classification_metrics(y_test, proba_test)
comparatif = pd.DataFrame(
    {
        "OOF train": [f"{m_oof['roc_auc']:.3f}", f"{m_oof['pr_auc']:.3f}", f"{m_oof['brier']:.4f}"],
        "Test scellé": [
            f"{m_test['roc_auc']:.3f}",
            f"{m_test['pr_auc']:.3f}",
            f"{m_test['brier']:.4f}",
        ],
    },
    index=["ROC-AUC", "PR-AUC", "Brier"],
)
comparatif.index.name = "Mesure"
display(comparatif)

# Trois lectures superposées OOF train / test scellé : rang global (ROC), rang sur la minorité
# (PR, ligne de base = prévalence tracée par plot_pr), fiabilité des probabilités (calibration).
fig, (ax_roc, ax_pr, ax_cal) = plt.subplots(1, 3, figsize=(16, 5))
for jeu, y_jeu, proba_jeu in [
    ("OOF train", y_train, proba_oof),
    ("Test scellé", y_test, proba_test),
]:
    evaluation.plot_roc(y_jeu, {jeu: proba_jeu}, ax=ax_roc)
    evaluation.plot_pr(y_jeu, {jeu: proba_jeu}, ax=ax_pr)
    evaluation.plot_calibration(y_jeu, {jeu: proba_jeu}, ax=ax_cal)
ax_roc.set_title("Courbe ROC")
ax_pr.set_title("Courbe précision-rappel (base = prévalence)")
ax_cal.set_title("Courbe de calibration")
for ax in (ax_roc, ax_pr, ax_cal):
    ax.legend(loc="best")
plt.tight_layout()
plt.show()

### Interprétation

- **Généralisation tenue** - ROC-AUC et PR-AUC quasi inchangées de l'OOF train au test scellé : pas de sur-ajustement, l'estimation du §9 se vérifie sur des données jamais vues.
- **PR-AUC bien au-dessus de la prévalence** - environ **28 %** des étudiants décrochent (part affichée à l'ouverture) ; une PR-AUC très supérieure à cette base montre un pouvoir de rang réel sur la classe minoritaire, non un effet du déséquilibre. La ROC-AUC, elle, se lit contre 0,50.
- **Probabilités fiables** - le Brier tient au niveau de la validation croisée et la courbe de calibration suit la diagonale : la probabilité prédite épouse la fréquence observée, l'écart ne subsistant qu'aux probabilités intermédiaires, où le faible effectif bruite l'estimation - le non-recalibrage (D31) reste justifié sur le test scellé.

La petite taille de la classe positive invite à lire ces valeurs comme des **estimations**, pas à la troisième décimale.


## 12.2 Point de fonctionnement au seuil 0,42

Le seuil 0,42 (D04, plancher de rappel 80 %) a été fixé sur l'OOF train en §9 ; il s'applique ici au test **tel quel**, sans recalage.

In [ ]:
# Matrice de confusion sur le test scellé, au seuil 0,42 (D04) - appliqué tel quel, sans recalage.
fig, ax = plt.subplots(figsize=(3.8, 3.4))
evaluation.plot_confusion(
    y_test, proba_test, SEUIL_DEFAUT, ax=ax, title=f"Seuil {SEUIL_DEFAUT:.2f} (test scellé)"
)
plt.tight_layout()
plt.show()


# Point de fonctionnement au seuil retenu, sur chaque jeu ; F1 dérivé de précision et rappel.
def point_seuil(y, proba):
    """Ligne du seuil retenu (threshold_table), enrichie du F1 (moyenne harmonique P-R)."""
    ligne = threshold.threshold_table(y, proba, thresholds=np.array([SEUIL_DEFAUT])).iloc[0]
    p, r = ligne["precision"], ligne["rappel"]
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return ligne, f1


def colonne_point(ligne, f1):
    """Colonne formatée (chaînes) pour la table du point de fonctionnement."""
    return [
        f"{ligne['rappel']:.1%}",
        f"{ligne['precision']:.1%}",
        f"{f1:.3f}",
        f"{ligne['f2']:.3f}",
        f"{int(ligne['n_alertes'])}",
        f"{int(ligne['tp'])}",
        f"{int(ligne['n_FN'])}",
        f"{ligne['pct_promo']:.1%}",
    ]


pt_oof, f1_oof = point_seuil(y_train, proba_oof)
pt_test, f1_test = point_seuil(y_test, proba_test)
point = pd.DataFrame(
    {"OOF train": colonne_point(pt_oof, f1_oof), "Test scellé": colonne_point(pt_test, f1_test)},
    index=[
        "Rappel",
        "Précision",
        "F1",
        "F2",
        "Signalés",
        "TP (décrocheurs signalés)",
        "FN (décrocheurs manqués)",
        "% promotion",
    ],
)
point.index.name = "Mesure"
display(point)

### Interprétation

- **Rappel juste sous le plancher de 80 %** - le seuil 0,42 calé sur l'OOF (D04) rend un rappel légèrement inférieur sur le test scellé : écart attendu d'un plancher fixé sur l'OOF et lu sur le test, **assumé et non recalé** - le recaler reviendrait à ajuster sur le test.
- **Précision et F-mesures stables** - l'arbitrage rappel/précision se transporte du train au test sans décrochage.
- **Les faux négatifs sont le coût qui compte** - chaque décrocheur manqué pèse plus qu'un signalement à tort (D03, FN >> FP) : c'est cette case que le seuil bas cherche à réduire, au prix d'une part de la promotion signalée.


## 12.3 Cible secondaire : note finale (régression)

La régression de `moyenne_finale` (/20) hiérarchise l'intensité de l'accompagnement (D29) - cible secondaire, lue sobrement. La MAE, en points sur 20, est une erreur directement interprétable.


In [ ]:
# Cible secondaire sur le test scellé - une passe, aucun réglage (le GB est figé §9).
y_note_test = test_df[CIBLE_REGRESSION]
pred_note_test = np.clip(
    bundle.regressor.predict(X_test), 0, 20
)  # note bornée [0, 20], artefact §10

# Contrôle d'identité §9 (mémoire) <-> §10 (sérialisé) sur la régression.
assert np.allclose(pred_note_test, np.clip(modele_note.predict(X_test), 0, 20)), (
    "artefact §10 != modèle §9"
)

# OOF train du GB final, même convention de bornage - référence de généralisation.
pred_note_oof = evaluation.oof_pred(gb_regle, X_train_min, y_note, cv_reg).clip(0, 20)
r_oof = evaluation.regression_metrics(y_note, pred_note_oof)
r_test = evaluation.regression_metrics(y_note_test, pred_note_test)
reg_comparatif = pd.DataFrame(
    {
        "OOF train": [f"{r_oof['mae']:.2f}", f"{r_oof['rmse']:.2f}", f"{r_oof['r2']:.3f}"],
        "Test scellé": [f"{r_test['mae']:.2f}", f"{r_test['rmse']:.2f}", f"{r_test['r2']:.3f}"],
    },
    index=["MAE (pts/20)", "RMSE (pts/20)", "R²"],
)
reg_comparatif.index.name = "Mesure"
display(reg_comparatif)

# Note réelle vs prédite, OOF train et test scellé côte à côte - la diagonale = prédiction parfaite.
fig, (ax_oof, ax_test) = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
evaluation.plot_regression_fit(y_note, pred_note_oof, ax=ax_oof)
evaluation.plot_regression_fit(y_note_test, pred_note_test, ax=ax_test)
ax_oof.set_title("Note réelle vs prédite (OOF train)")
ax_test.set_title("Note réelle vs prédite (test scellé)")
plt.tight_layout()
plt.show()

### Interprétation

- **Erreur de l'ordre de deux points sur vingt** - la MAE se transporte de l'OOF au test sans dérive : la note estimée situe l'étudiant, elle ne le note pas (usage secondaire, D29).
- **Ajustement plus lâche dans le bas de l'échelle** - le nuage se resserre sur la diagonale pour les notes hautes, se disperse pour les notes basses où la prédiction tire vers la moyenne : la priorisation reste fiable en rang, moins en valeur absolue pour les cas fragiles.
- **Généralisation confirmée** - RMSE et R² quasi identiques du train au test.

## 12.4 Explicabilité du modèle

En production, le service explique chaque signalement en **décomposant la log-cote par variable** : chaque variable reçoit sa part `bᵢ·zᵢ` de la somme qui fait le score (contributions analytiques exactes, référence à l'origine, D34). C'est cette décomposition qui est rendue à l'inférence. SHAP n'est calculé **qu'ici, hors ligne** (écarté de l'inférence par éco-conception) : il sert la confirmation globale sur le test scellé et le contrôle de fidélité de l'explication analytique - jamais l'inférence.


In [ ]:
# Explicabilité globale par SHAP - écarté de l'inférence (D34), mobilisé ici pour confirmer les
# moteurs du modèle retenu (les leurres, retirés en §8.6, en sont absents).
lin = modele_abandon[-1]  # régression logistique (dernier maillon)
ct = next(s for _, s in modele_abandon.steps if isinstance(s, ColumnTransformer))
sources = np.array(explain.source_columns(ct))  # variable source de chaque colonne préprocessée
Z_train = modele_abandon[:-1].transform(X_train_min)  # background : sa moyenne = référence SHAP
Z_test = modele_abandon[:-1].transform(X_test)
# Masker sur tout le train (pas de sous-échantillonnage) : la référence est la moyenne complète.
masker = shap.maskers.Independent(Z_train, max_samples=Z_train.shape[0])  # type: ignore
explainer = shap.LinearExplainer(lin, masker)
shap_test = explainer.shap_values(Z_test)  # (n_test, n_colonnes préprocessées)
display(
    Markdown(
        f"SHAP calculé sur **{len(Z_test)}** cas de test · référence = **moyenne** du background "
        f"(train, **{len(Z_train)}** lignes) · le service explique par contributions "
        "analytiques (origine, D34)"
    )
)

### 12.4.1 Importance globale


In [ ]:
# Importance globale : moyenne des |valeurs SHAP|, repliée des colonnes
# préprocessées vers la source.
importance = (
    pd.DataFrame({"variable": sources, "imp": np.abs(shap_test).mean(axis=0)})
    .groupby("variable")["imp"]
    .sum()
    .sort_values()
)
fig, ax = plt.subplots(figsize=(7, 6))
# rose SHAP (#ff0d57), cohérent avec les waterfalls
ax.barh(importance.index, importance.values, color="#ff0d57")  # type: ignore
for y_i, val in enumerate(importance.values):
    ax.text(val, y_i, f" {val:.2f}", va="center", ha="left", fontsize=9, color="#333333")
ax.set_xlim(0, importance.values.max() * 1.15)  # type: ignore # marge pour les étiquettes de valeur
ax.set(
    xlabel="Importance SHAP moyenne (|log-cote|)",
    title="Importance globale des variables (test scellé, SHAP)",
)
plt.tight_layout()
plt.show()

### Interprétation

- **Le modèle décide sur l'engagement et l'assiduité** - heures de connexion au LMS, taux de présence, motivation et rendu des devoirs dominent : des signaux pédagogiques actionnables, cohérents avec l'accompagnement visé (D05).
- **`nb_ue_total` sans effet mesurable** - contribution négligeable ; c'est un constat, pas une décision de retrait : l'inclusion d'une variable se tranche en §8 (ablation sur le train) et par principe de minimisation, jamais sur l'importance mesurée avec la cible sur le test scellé.


### 12.4.2 Explication locale et fidélité de l'inférence

L'explication locale est calculée sur **deux dossiers contrastés** : l'étudiant réellement décrocheur dont la probabilité d'abandon est la plus élevée (`i_haut`), et l'étudiant le moins à risque (`i_bas`). On lit d'abord ce qui pousse chaque dossier, puis on vérifie que l'**explication analytique du service** - la décomposition de la log-cote rendue à l'inférence (D34) - désigne les mêmes moteurs que SHAP.


In [ ]:
# Deux dossiers contrastés : le décrocheur réel le plus probable, et l'étudiant le moins à risque.
dec = np.where(y_test.to_numpy() == 1)[0]
i_haut = int(dec[np.argmax(proba_test[dec])])
i_bas = int(np.argmin(proba_test))

# Objet SHAP au niveau des colonnes préprocessées (valeurs scalées lisibles à gauche du « = »).
base_shap = float(np.ravel(explainer.expected_value)[0])
explication = shap.Explanation(
    values=shap_test,
    base_values=np.full(len(Z_test), base_shap),
    data=np.asarray(Z_test),
    feature_names=list(sources),
)

shap.plots.waterfall(explication[i_haut], max_display=10, show=False)
fig = plt.gcf()
fig.set_size_inches(8, 5)
fig.axes[0].set_title(f"Étudiant à risque (proba d'abandon {proba_test[i_haut]:.2f})", fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
shap.plots.waterfall(explication[i_bas], max_display=10, show=False)
fig = plt.gcf()
fig.set_size_inches(8, 5)
fig.axes[0].set_title(
    f"Étudiant peu à risque (proba d'abandon {proba_test[i_bas]:.4f})", fontsize=11
)
plt.tight_layout()
plt.show()

In [ ]:
# Fidélité : l'explication de production (analytique, origine) rejoint-elle SHAP (moyenne) ?
contrib = explain.contributions(modele_abandon, X_test.iloc[[i_haut]], themes=themes)
analytiques = pd.Series(contrib.by_variable)
shap_local = pd.Series(shap_test[i_haut]).groupby(sources).sum()
fidelite = (
    pd.DataFrame({"Contribution analytique (origine)": analytiques, "SHAP (moyenne)": shap_local})
    .reindex(analytiques.abs().sort_values(ascending=False).index)
    .round(3)
)
display(fidelite.head(8))
display(
    Markdown(
        f"Base analytique (origine) **{contrib.base_logit:.2f}** · base SHAP (moyenne) "
        f"**{base_shap:.2f}** · log-cote du cas **{contrib.total_logit:.2f}** "
        f"(proba **{contrib.probability:.1%}**) - même total, référence différente"
    )
)

### Interprétation

- **Le risque se lit d'un coup d'œil** - chez l'étudiant à risque, faible assiduité, faible engagement LMS et motivation basse empilent des contributions positives qui portent la log-cote loin au-dessus du départ ; chez l'étudiant épargné, un fort engagement LMS et une bonne présence la tirent nettement en dessous.
- **Des leviers actionnables** - les variables qui pèsent sont celles sur lesquelles l'accompagnement peut agir (présence, connexion au LMS, rendu, motivation) : chaque signalement est explicable à l'équipe pédagogique (D05).
- **Fidélité à l'inférence (D34)** - les contributions analytiques du service (table) reprennent les mêmes moteurs, mêmes signes ; l'écart tient à la référence (origine vs moyenne du background), non aux facteurs désignés : SHAP confirme sans être nécessaire en production.

## 12.5 Audit d'équité

Les attributs protégés `sexe` et `boursier` sont exclus du modèle (D13) mais conservés hors modèle par le split : ils servent ici à vérifier que la détection des décrocheurs ne s'effondre pas selon le groupe. L'audit s'étend aux modalités rares repérées en §6 (D17), pas aux seuls attributs protégés.


### 12.5.1 Attributs protégés


In [ ]:
def rappel_par_modalite(y, proba, serie, seuil):
    """Rappel, précision et effectifs par modalité d'un sous-groupe, à un seuil donné."""
    pred = np.asarray(proba) >= seuil
    y = np.asarray(y)
    lignes = []
    for modalite in sorted(serie.dropna().unique(), key=str):
        m = (serie == modalite).to_numpy()
        pos = int(y[m].sum())
        tp = int((pred[m] & (y[m] == 1)).sum())
        sig = int(pred[m].sum())
        lignes.append(
            {
                "Modalité": str(modalite),
                "n": int(m.sum()),
                "décrocheurs": pos,
                "rappel": tp / pos if pos else float("nan"),
                "précision": tp / sig if sig else float("nan"),
            }
        )
    return pd.DataFrame(lignes)


def ligne_ensemble(y, proba, seuil, label="Ensemble (test)"):
    """Ligne de référence : rappel et précision sur tout le test, pour situer les sous-groupes."""
    return rappel_par_modalite(y, proba, pd.Series([label] * len(y), index=y.index), seuil)


def formater_equite(df, index):
    """Formate rappel/précision en % (n.d. si non défini) et indexe le tableau."""
    out = df.copy()
    for col in ("rappel", "précision"):
        out[col] = out[col].map(lambda v: f"{v:.0%}" if pd.notna(v) else "n.d.")
    return out.set_index(index)


ref = ligne_ensemble(y_test, proba_test, SEUIL_DEFAUT)  # rappel global, ligne de comparaison
for attribut in ("sexe", "boursier"):
    groupes = rappel_par_modalite(y_test, proba_test, test_df[attribut], SEUIL_DEFAUT)
    display(Markdown(f"**> Rappel par `{attribut}`**"))
    display(formater_equite(pd.concat([ref, groupes], ignore_index=True), "Modalité"))

### 12.5.2 Modalités rares


In [ ]:
# Modalités sous 5 % (D22) des catégorielles non protégées - une table par variable concernée.
categorielles_audit = ["filiere", "bac_type", "mention_bac", "etablissement_origine"]
SEUIL_RARETE = 0.05
ref_rares = ligne_ensemble(y_test, proba_test, SEUIL_DEFAUT)  # ligne d'ensemble, comparaison
n_rares = 0
for col in categorielles_audit:
    t = rappel_par_modalite(y_test, proba_test, test_df[col], SEUIL_DEFAUT)
    rares_col = t[t["n"] / len(y_test) < SEUIL_RARETE]
    if rares_col.empty:
        continue
    n_rares += len(rares_col)
    display(Markdown(f"**> Modalités rares de `{col}`**"))
    display(formater_equite(pd.concat([ref_rares, rares_col], ignore_index=True), "Modalité"))
display(
    Markdown(
        f"**{n_rares}** modalités sous le seuil de rareté (5 %, D22) sur "
        f"`{'`, `'.join(categorielles_audit)}`, chacune comparée à la ligne "
        "d'ensemble du test scellé"
    )
)

### Interprétation

Le critère retenu est l'**égalité des chances** - des taux de faux négatifs comparables entre groupes, car dans un dispositif d'accompagnement le décrocheur manqué est le coût qui compte (D03). Le rappel de chaque groupe se lit donc contre la ligne d'ensemble.

- **Attributs protégés sans écart matériel** - le rappel tient à ~2 points entre sexes et à ~3 points entre statuts boursiers (plusieurs centaines d'étudiants chacun), autour de l'ensemble : le modèle ne rate pas davantage de décrocheurs selon le sexe ou la précarité. La précision par groupe reste homogène - pas de sur-signalement d'un sous-groupe, donc pas de stigmatisation différenciée.
- **Modalités rares non concluantes** - `sexe` autre / non renseigné (n ≤ 10) et `etablissement_origine` autre (n = 31) reposent sur trop peu d'étudiants : un rappel à 100 % comme à 67 % s'y lit comme du bruit, non comme un biais. Le seul écart sous l'ensemble (établissement « autre ») est à surveiller en exploitation, sur des effectifs cumulés plus grands (§11), pas à conclure ici.

## 12.6 Impacts métier

**Objectif** : traduire le point de fonctionnement (§12.2) en conséquences d'accompagnement à l'échelle d'une promotion, sans coût chiffré - l'énoncé n'en fournit pas (D03, D04). Le volume, lui, ne demande aucune hypothèse : c'est le produit des taux du test par l'effectif de la promotion.


In [ ]:
# Volume à l'échelle d'une promotion : les taux mesurés sur le test scellé (§12.2), portés sur
# l'effectif réel de la cohorte L1 (§1). Aucun coût supposé - un simple produit taux x effectif.
EFFECTIF_PROMO = 5200  # cohorte L1 de l'énoncé (§1)
prevalence = float(y_test.mean())
decrocheurs_promo = prevalence * EFFECTIF_PROMO
signales_promo = float(pt_test["pct_promo"]) * EFFECTIF_PROMO
detectes_promo = float(pt_test["rappel"]) * decrocheurs_promo
manques_promo = decrocheurs_promo - detectes_promo


def _milliers(x):
    """Entier arrondi, séparateur de milliers en espace fine (lisible pour le métier)."""
    return f"{x:,.0f}".replace(",", " ")


volume = pd.DataFrame(
    {
        "Par campagne annuelle": [
            _milliers(EFFECTIF_PROMO),
            _milliers(decrocheurs_promo),
            _milliers(signales_promo),
            _milliers(detectes_promo),
            _milliers(manques_promo),
        ]
    },
    index=pd.Index(
        [
            "Promotion (effectif)",
            "Décrocheurs attendus",
            "Dossiers à instruire (signalés)",
            "Décrocheurs détectés",
            "Décrocheurs manqués",
        ],
        name="Volume",
    ),
)
display(volume)

### Interprétation

- **Charge d'accompagnement** - au seuil retenu, la campagne annuelle (D06) signale environ un tiers de la promotion (tableau ci-dessus) : autant de dossiers à instruire, vrais et faux positifs confondus, dans la limite de la capacité de tutorat.
- **Détection et manqués** - le dispositif détecte la grande majorité des décrocheurs attendus ; les manqués (faux négatifs) sont le risque que le seuil bas cherche d'abord à réduire (FN ≫ FP, D03).
- **Deux leviers sans réentraîner** - l'établissement ajuste le point de fonctionnement par la politique de seuil (plancher de rappel, D04), le régime de sortie (probabilité seule ou indicateur binaire, D39) et une contrainte de capacité (les N plus à risque, D40) : le modèle expose la probabilité, la décision reste pilotable.


## 12.7 Carte du modèle et métadonnées

**Objectif** : sceller le livrable de gouvernance - la *model card* que lisent le client et l'auditeur (usage, données, métriques sur le test scellé, limites, risques, version) - et son jumeau lisible-machine `model_metadata.json`.


In [ ]:
# Métriques sur le test scellé, déjà mesurées en §12.1-12.3, réassemblées pour la carte.
metrics_holdout = {
    "roc_auc": float(m_test["roc_auc"]),
    "pr_auc": float(m_test["pr_auc"]),
    "brier": float(m_test["brier"]),
    "rappel": float(pt_test["rappel"]),
    "precision": float(pt_test["precision"]),
    "f1": float(f1_test),
    "f2": float(pt_test["f2"]),
    "mae_note": float(r_test["mae"]),
    "rmse_note": float(r_test["rmse"]),
    "r2_note": float(r_test["r2"]),
}

# Empreinte du jeu gold : versionne les DONNÉES, comme l'empreinte d'artefact
# versionne le CODE (§10.2).
chemin_gold = settings.gold_dir / "etudiants_gold.csv"
gold_md5 = hashlib.md5(chemin_gold.read_bytes()).hexdigest()[:12]

metadata = model_card.build_metadata(
    version=fiche.facts.version,
    created_at=datetime.now(UTC).isoformat(timespec="seconds"),
    package_version=get_package_version("decrochage-l1"),
    random_seed=SEED,
    target_primary=CIBLE,
    target_secondary=CIBLE_REGRESSION,
    features=model_features,
    n_train_rows=len(train_df),
    abandon_rate_train=float(train_df[CIBLE].mean()),
    dataset=chemin_gold.relative_to(settings.root_dir).as_posix(),
    gold_md5=gold_md5,
    threshold=float(SEUIL_DEFAUT),
    metrics_holdout=metrics_holdout,
)

# Contenu de GOUVERNANCE : déclaré ici, défendu à l'oral (§2 usage, §4 éthique, §13 suivi).
# Livrable AUTONOME (structure Hugging Face) : aucun renvoi interne au notebook dans le texte.
carte = model_card.ModelCard(
    model_name="Détection du décrochage en L1 (mi-S1)",
    version=fiche.facts.version,
    created_at=metadata["created_at"][:10],  # date seule sur la carte
    owners="Établissement",
    description=(
        "Modèle d'aide à la décision qui estime, à mi-parcours du premier semestre, la probabilité "
        "qu'un étudiant de L1 abandonne, afin de déclencher un accompagnement humain."
    ),
    details={
        "Développé par": "Julien ALBURQUERQUE",
        "Type": (
            "deux modèles - classification (régression logistique, probabilités nativement "
            "fiables et non recalibrées, D31) pour la probabilité d'abandon ; régression "
            "(gradient boosting) pour une estimation de la note finale, secondaire, qui "
            "hiérarchise l'intensité de l'accompagnement"
        ),
        "Cibles": f"{CIBLE} (probabilité, principale) · {CIBLE_REGRESSION} (/20, secondaire)",
        "Langue des données": "français",
        "Licence": "usage uniquement dans le cadre de la certification CISIA",
    },
    direct_use=(
        "Prioriser l'accompagnement des étudiants de L1 à mi-S1. La sortie principale est une "
        "probabilité, jamais une classe ni une décision."
    ),
    users="Cellule pédagogique et tuteurs de l'établissement.",
    out_of_scope=(
        "Aucune décision administrative automatisée (orientation, sanction, sélection). Aide à la "
        "décision : la décision d'accompagnement reste humaine (RGPD, art. 22). "
        "Ne jamais exposer la "
        "seule classe binaire à l'utilisateur."
    ),
    limitations=[
        "Cohorte d'un seul établissement et d'une seule année : généralisation à valider ailleurs.",
        (
            "Petits sous-groupes non concluants ; modalité établissement « autre » "
            "à surveiller en exploitation."
        ),
        "Aucune variable protégée ni proxy socio-économique en entrée (minimisation).",
        "Garde-fous contre la stigmatisation et la prophétie auto-réalisatrice à tenir côté usage.",
    ],
    recommendations=[
        (
            "Conserver la décision d'accompagnement humaine ; ne jamais exposer la "
            "seule classe binaire."
        ),
        ("Rappeler l'avertissement de l'article 22 dans la documentation et l'outil"),
        (
            "Suivre la dérive des entrées par campagne et demander un ré-entraînement "
            "quand elle se confirme."
        ),
    ],
    training_data=(
        f"Cohorte L1, jeu « gold » nettoyé et validé : {len(train_df)} étudiants au train, "
        f"{train_df[CIBLE].mean():.1%} d'abandons. Variables connues à mi-S1 "
        "uniquement, sans fuite "
        "temporelle. Attributs protégés (sexe, boursier) exclus du modèle, conservés hors modèle "
        "pour l'audit d'équité."
    ),
    evaluation_protocol=(
        "Test scellé (20 % du jeu, jamais vu à l'entraînement), une seule passe. Seuil et "
        "hyperparamètres figés sur le train ; probabilités non recalibrées (D31)."
    ),
    metrics={
        "PR-AUC (abandon)": f"{m_test['pr_auc']:.3f}",
        "ROC-AUC (abandon)": f"{m_test['roc_auc']:.3f}",
        f"Rappel @seuil {SEUIL_DEFAUT:.2f}": f"{pt_test['rappel']:.0%}",
        f"Précision @seuil {SEUIL_DEFAUT:.2f}": f"{pt_test['precision']:.0%}",
        "Brier (fiabilité des probabilités)": f"{m_test['brier']:.4f}",
        "MAE note finale": f"{r_test['mae']:.2f} pts/20",
    },
    threshold_note=(
        f"{SEUIL_DEFAUT:.2f}, fixé sur un plancher de rappel (le coût d'un décrocheur "
        "manqué dépasse "
        "celui d'une fausse alerte) ; paramètre d'exploitation, jamais figé dans les poids."
    ),
    fairness_note=(
        "Rappel comparable entre sexes et statuts boursiers : pas de décrocheur davantage manqué "
        "selon le sexe ou la précarité."
    ),
    technical_specs=(
        "Pipeline scikit-learn (préprocesseur + estimateur) sérialisé (joblib), servi par une API "
        "FastAPI. L'entrée est typée et validée ; le service ne journalise aucune donnée en "
        "exploitation. Seul un instantané du train (sans identifiant ni attribut protégé) est "
        "scellé dans l'artefact pour mesurer la dérive, remplacé à chaque ré-entraînement."
    ),
    contact="Julien ALBURQUERQUE (responsable du projet).",
)

chemins_carte = model_card.save(settings.artifacts_dir, card=carte, metadata=metadata)

recap_meta = pd.DataFrame(
    {
        "Valeur": [
            metadata["version"],
            f"{metadata['n_train_rows']} lignes · {metadata['abandon_rate_train']:.1%} d'abandons",
            f"{len(metadata['features'])} features",
            metadata["dataset"],
            metadata["gold_md5"],
            f"{metadata['threshold']:.2f}",
        ]
    },
    index=pd.Index(
        ["Version", "Train", "Features", "Jeu gold", "Empreinte gold", "Seuil"],
        name="Métadonnées du modèle",
    ),
)
display(recap_meta)
display(
    Markdown(
        f"Livrables écrits dans `artifacts/` - `{chemins_carte['card'].name}` "
        "(model card lisible par "
        f"le client et l'auditeur) et `{chemins_carte['metadata'].name}` (jumeau lisible-machine). "
        "La carte complète se lit dans ce fichier."
    )
)

## _Journal de bord des décisions_

Décisions **empiriques** - écrites après la mesure sur le test scellé.

1. **[D04] Effet du seuil, mesuré sur le test.** ⇒ Au seuil 0,42 (plancher de rappel 80 % fixé sur l'OOF), le rappel sur le test scellé (78,7 % sur 296 décrocheurs) est au niveau du plancher.
2. **[D31] Calibration confirmée sur le test scellé.** ⇒ Brier et courbe de fiabilité tiennent sur le test (§12.1) : le non-recalibrage reste justifié.
3. **[D34] Explicabilité analytique fidèle.** ⇒ Contributions analytiques et SHAP désignent les mêmes moteurs, mêmes signes, seule la référence diffère (§12.4.2) : SHAP confirme sans être nécessaire à l'inférence.
4. **[D17] Audit d'équité étendu aux modalités rares - tranché.** ⇒ Le rappel se tient sur `sexe` et `boursier`, proche du global ; les modalités rares (quelques dizaines d'étudiants) ne concluent pas, l'établissement « autre » restant à surveiller en exploitation : aucun biais matériel détecté. *Écarté* - restreindre l'audit aux seuls attributs protégés, qui laisserait les petits groupes hors de vue.


✅ Performance et impacts mesurés sur le test scellé - modèle défendable [C8]

# 13. Amélioration continue (ré-entraînement, suivi, versioning) [C9]

**Objectif** : poser le cadre d'**amélioration continue** - suivi en exploitation, déclencheurs et méthodologie de ré-entraînement, versioning - et le rendre défendable.

La dérive se mesure en §11.4 ; §13 ne la rejoue pas, il dit **ce qu'on fait du signal dans le temps**. Tout le chapitre tient à une contrainte : la vérité-terrain `abandon` n'est connue qu'en **fin d'année** (D06) - le suivi et le ré-entraînement s'organisent autour d'elle.

## 13.1 Suivi en deux temps et déclencheurs de ré-entraînement

La vérité-terrain `abandon` n'arrive qu'en fin d'année. Le suivi s'organise donc en deux temps.

- **À chaque campagne (mi-S1)** - sans vérité-terrain, on surveille la **dérive** : celle des entrées (PSI/KS par cohorte, §11.4, D35) et celle des prédictions (distribution des scores, taux de signalés). Une dérive déclenche une **analyse et une revue humaine**, jamais un ré-entraînement.
- **En fin d'année** - la vérité-terrain permet enfin de mesurer le **rappel réel** et de le comparer à la référence (§12). C'est le seul moment où un ré-entraînement se décide.

| Moment | Signal surveillé | Déclencheur | Réponse |
|---|---|---|---|
| **Campagne** (mi-S1) | dérive des entrées et des prédictions | PSI ≥ 0,25 (§11.4) ou taux de signalés anormal | analyse + revue humaine |
| **Fin d'année** | rappel réel | rappel tombé 5 points sous l'objectif | **ré-entraînement** |
| **Fin d'année** | rappel réel | aucun écart | **pas de ré-entraînement** |

Régler le seuil ne demande pas de ré-entraîner (D04, D38) : c'est un réglage d'exploitation, distinct du ré-entraînement du modèle. Le rappel décide seul ; précision, PR-AUC et prévalence servent à **comprendre** un écart, pas à le déclencher.

## 13.2 Ré-entraînement : méthodologie et responsabilités

Le ré-entraînement suit un circuit **client (université) → prestataire** :

1. l'université reçoit le bilan de dérive dans la réponse de campagne (§11.4) ;
2. devant un écart confirmé en fin d'année, elle sollicite le prestataire ;
3. le prestataire analyse, ré-entraîne, **re-mesure sur un test scellé**, puis re-livre l'artefact versionné.

**Si le jeu reçu est incohérent avec l'entraînement**, la réponse dépend de la gravité :

- **incohérence mineure** (format, écriture) - nettoyage automatique, la campagne est scorée ;
- **incohérence structurelle** (population ou schéma vraiment changés) - la prédiction n'est plus fiable ; l'université en est informée, et soit fournit un jeu corrigé, soit attend la fin d'année pour transmettre la campagne **avec ses labels**, qui servira au ré-entraînement.

**Responsabilités.** Le ré-entraînement de maintenance reste **à iso-périmètre** (même schéma, mêmes variables) ; toute évolution (nouvelle source, nouvelle variable, nouvelle cible, changement réglementaire) fait l'objet d'un **avenant**. Acteurs et responsable de traitement sont posés en §4.7 et §11.6.

**Données.** L'université **conserve les cohortes passées** (elle en est le responsable de traitement) et les transmet au prestataire pour un ré-entraînement enrichi. Le prestataire, lui, **ne stocke aucune donnée d'étudiant** - minimisation, art. 5.1.c (D43).


## 13.3 Versioning et reproductibilité hors notebook

**Versioning, sobre par choix.** Version et empreintes vivent dans la fiche (§10.1) ; l'artefact et le jeu gold portent chacun leur empreinte (§10.2, §12.7), et `model_metadata.json` compare une version à la suivante. Une empreinte du gold par campagne versionne l'axe données ; `uv.lock` et les `random_state` fixés garantissent la reproductibilité (D41).

**Reproductibilité hors notebook.** La chaîne préparation → entraînement se rejoue par une **CLI** (`decrochage-l1`, commandes `predict` et `retrain`) qui réutilise le code de `src/` et relit les jugements figés en configuration. Le ré-entraînement rejoue le modèle **tel qu'éprouvé**, sans re-régler les hyperparamètres (D44). Chaque `retrain` écrit un **dossier d'archive horodaté et versionné** (`artifacts/history/<date>_v<version>`, option `--version`) puis rafraîchit le dossier courant que lit le service : les versions passées sont conservées pour un retour arrière.

Le sur-outillage est écarté par sobriété : ni MLflow ni DVC pour une seule campagne par an (D41), ni Prefect pour orchestrer un lancement annuel (D44) - options gardées pour un périmètre pluriannuel.

## 13.4 Revue périodique et perspectives

**Revue des indicateurs.** À chaque campagne, une revue de la dérive et de la volumétrie ; en fin d'année, une revue de la performance sur vérité-terrain. Elle réunit l'équipe pédagogique, la DSI, le DPO (§11.6) et le prestataire, qui décident ensemble d'une simple analyse ou d'un ré-entraînement.

**Perspectives** - trois évolutions au-delà du cadre actuel :

- **Autonomie de l'université sur le ré-entraînement** - intégrer le pipeline préparation → entraînement dans la solution livrée, avec gestion des versions du modèle. Les modèles sont petits et peu coûteux à entraîner : l'université pourrait relancer un ré-entraînement, remplacer une version par une autre à la détection d'une dérive, et revenir à la précédente au besoin. L'autonomie resterait partielle - comprendre la cause d'une dérive garde tout son sens.
- **Recommander un accompagnement, pas seulement un risque** - le modèle prédit l'abandon, non l'effet d'un accompagnement. Avec le recul de plusieurs cohortes sur les accompagnements passés (type proposé, résultat en fin d'année), on pourrait prédire en plus la **probabilité qu'un accompagnement soit efficace** et suggérer un **type** (tutorat, aide méthodologique, aide financière).
- **Suivi des faux négatifs** - réintégrer les décrocheurs manqués comme signal d'amélioration de la campagne suivante.

## _Journal de bord des décisions_

1. **[D41] Versioning sobre.** ⇒ joblib · git · `uv.lock` · empreinte MD5 du gold dans `model_metadata.json`, empreintes SHA-256 des artefacts affichées en §10.2 · une empreinte du gold par campagne. *Écarté* - MLflow (registre de modèles) et DVC (versioning des données), sur-outillage pour une seule campagne par an ; cités comme bascule pluriannuelle.
2. **[D42] Critères de ré-entraînement en deux temps.** ⇒ T0 sans label, analyse et revue ; T1, ré-entraînement si le rappel réel tombe 5 points sous l'objectif métier ; pas de ré-entraînement sans écart. *Écarté* - un ré-entraînement au fil de l'eau ou déclenché par la seule dérive d'entrée, qui recalerait le modèle sans label pour le valider.
3. **[D43] Stockage de l'historique côté client.** ⇒ le client (responsable de traitement, §4.7) conserve les cohortes passées et les retransmet pour un ré-entraînement élargi. *Écarté* - une conservation par le prestataire, contraire à la minimisation (art. 5.1.c) et à son statut de sous-traitant.
4. **[D44] CLI d'industrialisation.** ⇒ `decrochage-l1` (`predict`, `retrain`) rejoue la chaîne hors notebook, sans régler les hyperparamètres. *Écarté* - Prefect pour orchestrer le pipeline, disproportionné pour un lancement annuel.

✅ Cadre d'amélioration continue posé - suivi, déclencheurs, versioning, reproductibilité [C9]

# 14. Conclusion (synthèse et recommandations)


## 14.1 Réponse au besoin

Le besoin de §2 - repérer à mi-S1 les L1 à risque pour agir à temps - est couvert sur ses deux cibles : la classification `abandon` atteint une **AUC de 0,945** (§8) et retrouve **78,7 % des décrocheurs** au seuil retenu, sur le test scellé (§12.2) ; la régression `moyenne_finale` situe la note à **2,24 points près** (MAE, §12.3) pour moduler l'accompagnement. L'explicabilité (SHAP) et l'audit d'équité sont rendus en §12.4-12.5. L'objectif technique est tenu dans le cadre posé ; ce qu'il ne couvre pas est dit en 14.4.

## 14.2 Recommandation d'usage

La solution **assiste** la décision, elle ne la prend pas : l'équipe pédagogique garde la main sur l'accompagnement (art. 22, §4.3).

- **Seuil de signalement** - réglage d'exploitation configurable (§9) sans ré-entraîner le modèle (§13.1).
- **Conditions préalables au déploiement**, à la charge du responsable de traitement - AIPD, base légale confirmée par le DPO, information des étudiants (§4).
- **Effets de bord du marquage** - stigmatisation et prophétie auto-réalisatrice à encadrer et documenter (§4.6).

## 14.3 Intégration au système d'information

- **Recommandation d'action** - la DSI câble les flux d'entrée, de sortie et l'authentification sur le contrat OpenAPI publié (§11), pour un scoring d'une campagne par an ; l'architecture cible et la supervision sont posées en §11, non reprises ici.
- **Évolutions du modèle** - autonomie de ré-entraînement, recommandation d'un type d'accompagnement, suivi des faux négatifs : posées en §13.4.

## 14.4 Limites assumées

- **Généralisation non prouvée** - une seule cohorte, données synthétiques, validation intra-cohorte : un ré-entraînement est probable à la prochaine promotion (§13.1).
- Le détail des limites - complétude des données, calibrage du seuil sans coût métier chiffré, déployabilité conditionnée - est consolidé en §1.

# 15. Annexes (commandes, livrables, captures d'exploitation)

## 15.1 Commandes utiles

Les commandes sont documentées dans le `README.md` (« trois manières de faire tourner le projet ») ; les principales, par usage :

**Lancer le notebook** (livrable certifiant)
- `uv sync --group analysis` - installe le runtime et l'outillage d'étude, puis exécuter les cellules dans l'ordre.

**Rejouer la chaîne via la CLI** (hors notebook, §13.3)
- `uv sync --group cli` puis `uv run decrochage-l1 predict --input <cohorte.csv> --output <scores.csv>` - score une campagne avec l'artefact déployé (§10).
- `uv run decrochage-l1 retrain --students <jeu.csv>` - rejoue bronze → silver → gold → entraînement → package (§13.2, D44).

**Lancer le service d'inférence** (API + supervision + client, §11)
- `docker compose --env-file .env -f deploy/docker-compose.yml up -d --build` - la pile conteneurisée.

**Vérifier** - `uv run pytest` (suite complète, nécessite `--group analysis`).

**Environnement et versions** - Python, plateforme et versions des bibliothèques ci-dessous.


In [ ]:
# Versions des bibliothèques qui produisent les chiffres du dossier (dépendances clés).
def _ver(paquet):
    """Version installée, ou « non installé » (certains paquets vivent hors --group analysis)."""
    try:
        return get_package_version(paquet)
    except Exception:
        return "non installé"


paquets = [
    "scikit-learn",
    "numpy",
    "pandas",
    "scipy",
    "xgboost",
    "shap",
    "codecarbon",
    "optuna",
    "matplotlib",
    "pydantic",
    "fastapi",
    "typer",
]
display(Markdown(f"**Python** {sys.version.split()[0]} · **Plateforme** {platform.platform()}"))
display(
    pd.DataFrame(
        {"Version": [_ver(p) for p in paquets]}, index=pd.Index(paquets, name="Bibliothèque")
    )
)

## 15.2 Documents du projet

Les livrables produits par la chaîne, dans l'ordre du projet - du jeu de référence à la gouvernance du modèle et au journal de projet. Les liens sont **construits par le code** depuis `settings` (aucun chemin tapé à la main) ; les fichiers sont présents après une exécution complète (`artifacts/` et `data/gold/` sont hors dépôt).

In [ ]:
# Liens cliquables vers les livrables, construits par le code depuis settings (aucun chemin tapé) ;
# relatifs au notebook pour rester cliquables dans l'IDE. Un livrable non produit est masqué.
def lien(cible):
    """Chemin relatif au notebook (posix), cliquable dans l'IDE - produit par le code."""
    return "../" + cible.relative_to(settings.root_dir).as_posix()


journal = settings.root_dir / "notebooks" / "JALB-JournalDeBord-Decrochage-l1.ipynb"
documents = [
    (
        "Jeu de référence gold (CSV)",
        chemin_gold,
        "jeu propre et validé, entrée de l'entraînement (§7.3)",
    ),
    (
        "Artefacts sérialisés (dossier)",
        settings.artifacts_dir,
        "pipelines figés et fiche, rechargés par le service (§10.2)",
    ),
    (
        "Contrat d'entrée / sortie (JSON Schema)",
        chemin_contrat,
        "l'interface publique de l'API (§10.3)",
    ),
    (
        "Carte du modèle (model card)",
        chemins_carte["card"],
        "gouvernance et limites, lisible par l'auditeur (§12.7)",
    ),
    (
        "Métadonnées du modèle (JSON)",
        chemins_carte["metadata"],
        "jumeau lisible-machine de la carte (§12.7)",
    ),
    ("Journal de bord (sessions de travail)", journal, "le déroulé du projet, jour par jour"),
]
liens_md = [
    f"- [{libelle}]({lien(cible)}) - {note}" for libelle, cible, note in documents if cible.exists()
]
vide = "_Aucun livrable trouvé - exécuter le notebook de bout en bout._"
display(Markdown("\n".join(liens_md) if liens_md else vide))

## 15.3 Captures d'exploitation

Les surfaces décrites en §11 - la pile conteneurisée, le contrat de service, la supervision et le client de démonstration - saisies en fonctionnement. Le notebook ne lance aucun conteneur (D32) ; ces vues attestent l'exécution réelle.

In [ ]:
# Captures des surfaces d'exploitation (§11), lues depuis settings.resources_dir.
# Images versionnées dans notebooks/ressources.
captures = settings.resources_dir
vues = [
    ("pile-docker.png", "La pile en marche - quatre conteneurs sains (§11.3)"),
    ("api-swagger.png", "Le contrat de service - OpenAPI (§11.2)"),
    ("prometheus-supervision-technique.png", "Supervision technique - RED (§11.4)"),
    ("prometheus-supervision-metier.png", "Supervision métier - jauges et refus (§11.4)"),
    ("maquette-client.png", "Client - écran campagne (§11.2)"),
    ("maquette-client-resultat-1-etudiant.png", "Client - un dossier, contributions (§11.2)"),
    ("maquette-client-resultat-cas-derive.png", "Client - dérive signalée (§11.4)"),
]
for fichier, legende in vues:
    display(Markdown(f"**> {legende}**"))
    display(Image(filename=str(captures / fichier)))

## 15.4 Récapitulatif des métriques (contrôle)

Une ligne par **configuration distincte** (modèle × hyperparamètres × jeu de features), **recalculée de façon uniforme** (fit-and-discard, aucun modèle conservé) : panel « moyenne par pli » **et** « OOF regroupé » pour chaque config, sur les mêmes plis (`cv`, `seed`) que §8-§9. Le **test scellé** n'apparaît que pour les deux modèles finaux (l'ouvrir ailleurs serait une faute). Exporté en **Excel** - un onglet par cible, colonnes groupées (Train folds / Train OOF / Test), **statut coloré** (Validé / Écarté / Comparé). Un bloc de **contrôle** vérifie que ce recalcul indépendant **coïncide** avec les sections - et met l'écart 0,9453 (folds) vs 0,9451 (OOF) sur une même ligne.


In [ ]:
# Récapitulatif de contrôle - une ligne par CONFIGURATION distincte (modèle × hyperparamètres ×
# jeu). Chaque ligne est RECALCULÉE uniformément (fit-and-discard, aucun modèle conservé) : panel
# « moyenne par pli » ET « OOF regroupé », identique d'une config à l'autre. Le test scellé n'est
# renseigné que pour les deux modèles finaux (l'ouvrir ailleurs serait une faute).
lignes_metriques, controles = [], []


def _fold(recette, x, scoring):
    """Moyenne par pli du critère `scoring` (mêmes plis que §8-§9)."""
    return evaluation.cv_scores(recette, x, y_train, cv, scoring=scoring).mean()


def _panel_clf(recette, features):
    """Panel classification uniforme : ROC/PR/Brier en moyenne par pli ET en OOF regroupé."""
    x = X_train[features]
    proba = evaluation.oof_proba(recette, x, y_train, cv)
    m_oof = evaluation.classification_metrics(y_train, proba)
    return {
        "roc_folds": _fold(recette, x, "roc_auc"),
        "pr_folds": _fold(recette, x, "average_precision"),
        "brier_folds": -_fold(recette, x, "neg_brier_score"),
        "roc_oof": m_oof["roc_auc"],
        "pr_oof": m_oof["pr_auc"],
        "brier_oof": m_oof["brier"],
    }


def _panel_reg(recette, features):
    """Panel régression uniforme : MAE/RMSE/R² en OOF regroupé (le notebook n'évalue qu'en OOF)."""
    pred = evaluation.oof_pred(recette, X_train[features], y_moyenne, cv_reg).clip(0, 20)
    m = evaluation.regression_metrics(y_moyenne, pred)
    return {"mae_oof": m["mae"], "rmse_oof": m["rmse"], "r2_oof": m["r2"]}


def _row(cible, modele, hyper, label, features, panel, sections, note="", **extra):
    """Empile une ligne (une config) ; le jeu est étiqueté avec son nombre de variables."""
    lignes_metriques.append(
        {
            "cible": cible,
            "modèle": modele,
            "hyperparam.": hyper,
            "jeu": f"{label} ({len(features)})",
            **panel,
            **extra,
            "note": note,
            "sections": sections,
        }
    )


# --- Classification (abandon) : jeux d'ablation reconstruits depuis FEATURES et blocs ---
_sl = [f for f in FEATURES if f not in blocs["leurres"]]
_sp = [f for f in FEATURES if f not in blocs["proxies"]]
_slms = [f for f in FEATURES if f not in blocs["LMS redondantes"]]
_logreg_c01 = construire_pipeline_retenu(FEATURES_MIN).set_params(
    model__C=0.1, model__class_weight=None
)
_logreg_bal = construire_pipeline_retenu(FEATURES_MIN).set_params(model__class_weight="balanced")
_logreg_cal = CalibratedClassifierCV(logreg_final, method="isotonic", cv=cv)

_row(
    "abandon",
    "Régression logistique",
    "défaut",
    "complet",
    FEATURES,
    _panel_clf(construire_pipeline_retenu(FEATURES), FEATURES),
    "§8.4·§8.5·§8.6",
)
_row(
    "abandon",
    "Random Forest",
    "défaut",
    "complet",
    FEATURES,
    _panel_clf(modeles["random_forest"], FEATURES),
    "§8.5",
)
_row(
    "abandon",
    "XGBoost",
    "défaut",
    "complet",
    FEATURES,
    _panel_clf(modeles["xgboost"], FEATURES),
    "§8.5",
)
_row(
    "abandon",
    "Régression logistique",
    "défaut",
    "sans leurres",
    _sl,
    _panel_clf(construire_pipeline_retenu(_sl), _sl),
    "§8.6",
)
_row(
    "abandon",
    "Régression logistique",
    "défaut",
    "sans proxies",
    _sp,
    _panel_clf(construire_pipeline_retenu(_sp), _sp),
    "§8.6",
)
_row(
    "abandon",
    "Régression logistique",
    "défaut",
    "sans LMS redondantes",
    _slms,
    _panel_clf(construire_pipeline_retenu(_slms), _slms),
    "§8.6",
)
_row(
    "abandon",
    "Régression logistique",
    "défaut",
    "minimisé",
    FEATURES_MIN,
    _panel_clf(construire_pipeline_retenu(FEATURES_MIN), FEATURES_MIN),
    "§8.6·§9.1·§9.2",
)
_row(
    "abandon",
    "Régression logistique",
    "balanced",
    "minimisé",
    FEATURES_MIN,
    _panel_clf(_logreg_bal, FEATURES_MIN),
    "§9.1",
    note="écartée",
)
_row(
    "abandon",
    "Régression logistique",
    "réglée (C=0,1)",
    "minimisé",
    FEATURES_MIN,
    _panel_clf(_logreg_c01, FEATURES_MIN),
    "§9.2·§9.3·§12",
    note="✓ retenu",
    roc_test=float(m_test["roc_auc"]),
    pr_test=float(m_test["pr_auc"]),
    brier_test=float(m_test["brier"]),
    rappel_test=float(pt_test["rappel"]),
    precision_test=float(pt_test["precision"]),
)
_row(
    "abandon",
    "Régression logistique",
    "réglée + recalibrée (isotonic)",
    "minimisé",
    FEATURES_MIN,
    _panel_clf(_logreg_cal, FEATURES_MIN),
    "§9.3",
    note="écartée",
)
_row(
    "abandon",
    "Random Forest",
    "réglée (Optuna)",
    "minimisé",
    FEATURES_MIN,
    _panel_clf(build_rf(rf["best_params"]), FEATURES_MIN),
    "§9.2",
    note="challenger écarté",
)

# --- Régression (moyenne_finale) ---
_row(
    "moyenne_finale",
    "Ridge",
    "défaut",
    "complet",
    FEATURES,
    _panel_reg(regresseurs["ridge"], FEATURES),
    "§8.8·§8.9",
)
_row(
    "moyenne_finale",
    "Random Forest",
    "défaut",
    "complet",
    FEATURES,
    _panel_reg(regresseurs["random_forest"], FEATURES),
    "§8.9",
)
_row(
    "moyenne_finale",
    "Gradient Boosting",
    "défaut",
    "complet",
    FEATURES,
    _panel_reg(regresseurs["gradient_boosting"], FEATURES),
    "§8.9",
)
_row(
    "moyenne_finale",
    "Gradient Boosting",
    "défaut",
    "minimisé",
    FEATURES_MIN,
    _panel_reg(construire_regresseur_retenu(FEATURES_MIN), FEATURES_MIN),
    "§8.10·§9.2·§12.3",
    note="✓ retenu",
    mae_test=float(r_test["mae"]),
    rmse_test=float(r_test["rmse"]),
    r2_test=float(r_test["r2"]),
)
_row(
    "moyenne_finale",
    "Gradient Boosting",
    "Optuna frugal",
    "minimisé",
    FEATURES_MIN,
    _panel_reg(build_gb(best_frugal), FEATURES_MIN),
    "§9.2",
    note="écarté",
)
_row(
    "moyenne_finale",
    "Gradient Boosting",
    "Optuna lourd",
    "minimisé",
    FEATURES_MIN,
    _panel_reg(build_gb(best_lourd), FEATURES_MIN),
    "§9.2",
    note="écarté",
)

colonnes = [
    "cible",
    "modèle",
    "hyperparam.",
    "jeu",
    "roc_folds",
    "roc_oof",
    "pr_folds",
    "pr_oof",
    "brier_folds",
    "brier_oof",
    "mae_oof",
    "rmse_oof",
    "r2_oof",
    "roc_test",
    "pr_test",
    "brier_test",
    "rappel_test",
    "precision_test",
    "mae_test",
    "rmse_test",
    "r2_test",
    "note",
    "sections",
]
metrics_modeles = pd.DataFrame(lignes_metriques).reindex(columns=colonnes).round(4)


# Contrôle d'indépendance : le recalcul doit coïncider avec les chiffres affichés dans les sections.
def _cell(modele, hyper, jeu_prefixe, colonne):
    """Valeur recalculée d'une cellule (config = modèle × hyperparam × préfixe de jeu)."""
    masque = (
        (metrics_modeles["modèle"] == modele)
        & (metrics_modeles["hyperparam."] == hyper)
        & (metrics_modeles["jeu"].str.startswith(jeu_prefixe))
    )
    return float(metrics_modeles.loc[masque, colonne].iloc[0])


def _check(libelle, recalcule, attendu):
    """Signale un écart entre recalcul et valeur affichée en section (tolérance 1e-3)."""
    if abs(recalcule - attendu) > 1e-3:
        controles.append(f"{libelle} : recalculé {recalcule:.4f} ≠ section {attendu:.4f}")


_check(
    "logreg complet ROC folds",
    _cell("Régression logistique", "défaut", "complet", "roc_folds"),
    familles_scores["logreg"]["ROC-AUC (moy. folds)"],
)
_check(
    "logreg minimisé ROC OOF",
    _cell("Régression logistique", "défaut", "minimisé", "roc_oof"),
    float(table_ablation.loc["minimisé", "roc_auc"]),  # type: ignore
)  # type: ignore
_check(
    "logreg réglée ROC folds",
    _cell("Régression logistique", "réglée (C=0,1)", "minimisé", "roc_folds"),
    float(auc_regle),
)
_check(
    "GB minimisé MAE OOF",
    _cell("Gradient Boosting", "défaut", "minimisé", "mae_oof"),
    float(recap_gb.loc["base (défauts §8)", "MAE (pts/20, OOF)"]),  # type: ignore
)  # type: ignore

# Statut lisible (3 états) dérivé de la note, pour la mise en forme Excel.
metrics_modeles["statut"] = metrics_modeles["note"].map(
    lambda note: (
        "Validé"
        if note == "✓ retenu"
        else ("Écarté" if isinstance(note, str) and "écart" in note else "Comparé")
    )
)

# Export Excel : un onglet par cible, colonnes groupées (Train folds / Train OOF / Test), statut
# coloré. Chaque entrée de layout = (groupe | None, en-tête, colonne source, format numérique).
_GROUP_BG = {
    None: "#404040",
    "Train — moy. folds": "#8EA9DB",
    "Train — OOF regroupé": "#B4C6E7",
    "Test scellé": "#F4B183",
}
_seuil_txt = f"{SEUIL_DEFAUT:.2f}".replace(".", ",")
_LAYOUT_CLF = [
    (None, "Modèle", "modèle", "txt"),
    (None, "Hyperparamètres", "hyperparam.", "txt"),
    (None, "Jeu (features)", "jeu", "txt"),
    ("Train — moy. folds", "ROC-AUC", "roc_folds", "auc"),
    ("Train — moy. folds", "PR-AUC", "pr_folds", "auc"),
    ("Train — moy. folds", "Brier", "brier_folds", "brier"),
    ("Train — OOF regroupé", "ROC-AUC", "roc_oof", "auc"),
    ("Train — OOF regroupé", "PR-AUC", "pr_oof", "auc"),
    ("Train — OOF regroupé", "Brier", "brier_oof", "brier"),
    ("Test scellé", "ROC-AUC", "roc_test", "auc"),
    ("Test scellé", "PR-AUC", "pr_test", "auc"),
    ("Test scellé", "Brier", "brier_test", "brier"),
    ("Test scellé", f"Précision @{_seuil_txt}", "precision_test", "pct"),
    ("Test scellé", f"Rappel @{_seuil_txt}", "rappel_test", "pct"),
    (None, "Statut", "statut", "txt"),
    (None, "Sections", "sections", "txt"),
]
_LAYOUT_REG = [
    (None, "Modèle", "modèle", "txt"),
    (None, "Hyperparamètres", "hyperparam.", "txt"),
    (None, "Jeu (features)", "jeu", "txt"),
    ("Train — OOF regroupé", "MAE", "mae_oof", "reg2"),
    ("Train — OOF regroupé", "RMSE", "rmse_oof", "reg2"),
    ("Train — OOF regroupé", "R²", "r2_oof", "r2"),
    ("Test scellé", "MAE", "mae_test", "reg2"),
    ("Test scellé", "RMSE", "rmse_test", "reg2"),
    ("Test scellé", "R²", "r2_test", "r2"),
    (None, "Statut", "statut", "txt"),
    (None, "Sections", "sections", "txt"),
]


def _ecrire_feuille(wb, nom, sous, layout):
    """Écrit une feuille : en-têtes groupés fusionnés, formats numériques, statut coloré."""
    ws = wb.add_worksheet(nom)
    ws.freeze_panes(2, 3)
    grp = {
        g: wb.add_format(
            {
                "bold": True,
                "bg_color": bg,
                "align": "center",
                "valign": "vcenter",
                "border": 1,
                "font_color": "white" if g is None else "black",
            }
        )
        for g, bg in _GROUP_BG.items()
    }
    sub = wb.add_format(
        {
            "bold": True,
            "bg_color": "#F2F2F2",
            "align": "center",
            "valign": "vcenter",
            "border": 1,
            "text_wrap": True,
        }
    )
    num = {
        "txt": wb.add_format({"border": 1, "valign": "vcenter"}),
        "auc": wb.add_format({"num_format": "0.000", "border": 1, "align": "center"}),
        "brier": wb.add_format({"num_format": "0.0000", "border": 1, "align": "center"}),
        "pct": wb.add_format({"num_format": "0.0%", "border": 1, "align": "center"}),
        "reg2": wb.add_format({"num_format": "0.00", "border": 1, "align": "center"}),
        "r2": wb.add_format({"num_format": "0.000", "border": 1, "align": "center"}),
    }
    coul = {
        "Validé": wb.add_format(
            {
                "bg_color": "#C6EFCE",
                "font_color": "#006100",
                "bold": True,
                "border": 1,
                "align": "center",
            }
        ),
        "Écarté": wb.add_format(
            {
                "bg_color": "#FFC7CE",
                "font_color": "#9C0006",
                "bold": True,
                "border": 1,
                "align": "center",
            }
        ),
        "Comparé": wb.add_format(
            {"bg_color": "#F2F2F2", "font_color": "#808080", "border": 1, "align": "center"}
        ),
    }
    n = len(layout)
    c = 0
    while c < n:
        g = layout[c][0]
        if g is None:
            ws.merge_range(0, c, 1, c, layout[c][1], grp[None])
            c += 1
        else:
            j = c
            while j < n and layout[j][0] == g:
                j += 1
            ws.merge_range(0, c, 0, j - 1, g, grp[g])
            for k in range(c, j):
                ws.write(1, k, layout[k][1], sub)
            c = j
    for r, (_, row) in enumerate(sous.iterrows(), start=2):
        for cc, (_, _, col, kind) in enumerate(layout):
            val = row.get(col)
            if col == "statut":
                ws.write(r, cc, val, coul.get(val, num["txt"]))
            elif kind == "txt":
                ws.write(r, cc, "" if pd.isna(val) else val, num["txt"])
            elif pd.isna(val):
                ws.write_blank(r, cc, None, num[kind])
            else:
                ws.write_number(r, cc, float(val), num[kind])
    ws.set_column(0, 0, 24)
    ws.set_column(1, 2, 24)
    ws.set_column(3, n - 3, 11)
    ws.set_column(n - 2, n - 2, 10)
    ws.set_column(n - 1, n - 1, 18)


chemin_metrics = settings.report_dir / "metrics_modeles.xlsx"
chemin_metrics.parent.mkdir(parents=True, exist_ok=True)
classeur = xlsxwriter.Workbook(str(chemin_metrics))
_abandon = metrics_modeles[metrics_modeles["cible"] == "abandon"].reset_index(drop=True)
_note = metrics_modeles[metrics_modeles["cible"] == "moyenne_finale"].reset_index(drop=True)
_ecrire_feuille(classeur, "abandon", _abandon, _LAYOUT_CLF)
_ecrire_feuille(classeur, "moyenne_finale", _note, _LAYOUT_REG)
classeur.close()

display(metrics_modeles)
verdict = (
    "**Contrôle d'indépendance : OK** - recalcul = sections."
    if not controles
    else "⚠️ **Écarts détectés** : " + " · ".join(controles)
)
display(
    Markdown(
        f"**{len(metrics_modeles)}** configurations → Excel `{chemin_metrics.name}` "
        f"(deux onglets, statut coloré). {verdict}"
    )
)